# Backgammon RGB-D board-state project

working notebook for the project modules. The `%%writefile` cells export the current versions into the project folders, and the validation section at the end is for quick checks after edits.

Current organisation:

```text
Project root
  backgammon_types.py
  manual_board_corners.json

Acquisition/
  streams_module.py
  keyframe_gate.py
  rgbd_recorder.py
  live_stream_viewer.py
  depth_stack_live_viewer.py

Geometric + Visual Preprocessing/
  board_registration.py
  perspective_rectification.py
  image_normalisation.py
  point_tray_segmentation.py
  roi_preview.py

Interpretation/
  depth_stack_classifier.py
  piece_detection.py
  dice_cube_reader.py
  temporal_state_estimator.py
  rule_validation.py
  board_state_pipeline.py
```

The latest ROI values and grid-mask segmentation are built into `point_tray_segmentation.py`.


## 0. Path setup

Run this before importing the local modules. It adds the project folders to `sys.path`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

MODULE_DIRS = [
    PROJECT_ROOT,
    PROJECT_ROOT / "Acquisition",
    PROJECT_ROOT / "Geometric + Visual Preprocessing",
    PROJECT_ROOT / "Interpretation",
    PROJECT_ROOT / "Evaluation",
]

for path in MODULE_DIRS:
    path_str = str(path.resolve())
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)
        print("Added:", path_str)

print("Project root:", PROJECT_ROOT.resolve())


# 1. Shared root module


## Shared dataclasses and pipeline result types

Shared data classes used to pass results between the pipeline stages.


In [ ]:
%%writefile backgammon_types.py
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import numpy as np


ColourName = str
RegionName = str


@dataclass
class BoardLock:
    valid: bool
    corners_xy: np.ndarray
    confidence: float
    method: str
    frame_size_hw: Tuple[int, int]
    debug: Dict[str, Any] = field(default_factory=dict)


@dataclass
class RectifiedBoard:
    rgb_bgr: np.ndarray
    depth_mm: np.ndarray
    valid_mask: np.ndarray
    height_map_mm: np.ndarray
    homography: np.ndarray
    inverse_homography: np.ndarray
    board_mask: np.ndarray
    plane_coeffs: Tuple[float, float, float]


@dataclass
class NormalisedBoard:
    rgb_bgr: np.ndarray
    rgb_gray: np.ndarray
    depth_mm: np.ndarray
    depth_gray: np.ndarray
    valid_mask: np.ndarray
    height_map_mm: np.ndarray
    height_uint8: np.ndarray


@dataclass
class RegionMasks:
    masks: Dict[RegionName, np.ndarray]
    overlay_bgr: np.ndarray
    point_names: List[RegionName]
    auxiliary_names: List[RegionName]


@dataclass
class PieceInstance:
    region_name: RegionName
    colour_name: ColourName
    centroid_xy: Tuple[float, float]
    area_px: float
    radius_px: float
    height_mm: float
    stack_count: int
    confidence: float
    contour: Optional[np.ndarray] = None


@dataclass
class PieceDetectionResult:
    pieces: List[PieceInstance]
    region_counts: Dict[RegionName, Dict[ColourName, int]]
    overlay_bgr: np.ndarray
    confidence: float
    # Diagnostic masks used while tuning the detector. The main pipeline can
    # ignore these once the counts and overlay have been produced.
    stable_depth_support_mask: Optional[np.ndarray] = None
    slot_candidate_mask: Optional[np.ndarray] = None
    checker_detection_area_mask: Optional[np.ndarray] = None
    stack_class_bgr: Optional[np.ndarray] = None
    debug: Dict[str, Any] = field(default_factory=dict)


@dataclass
class DiceObservation:
    bbox_xyxy: Tuple[int, int, int, int]
    value: Optional[int]
    confidence: float


@dataclass
class CubeObservation:
    bbox_xyxy: Tuple[int, int, int, int]
    value: Optional[int]
    confidence: float
    text: Optional[str] = None


@dataclass
class DiceCubeObservation:
    dice: List[DiceObservation]
    cube: Optional[CubeObservation]
    overlay_bgr: np.ndarray
    confidence: float


@dataclass
class BoardState:
    region_counts: Dict[RegionName, Dict[ColourName, int]]
    dice: Tuple[Optional[int], Optional[int]] = (None, None)
    cube_value: Optional[int] = None
    confidence: float = 0.0
    timestamp: Optional[float] = None
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class ValidationReport:
    valid: bool
    errors: List[str]
    warnings: List[str]
    confidence: float


@dataclass
class TemporalEstimate:
    candidate_state: BoardState
    fused_state: BoardState
    changed_vs_last_commit: bool
    history_size: int


@dataclass
class StateEvent:
    event_type: str
    description: str
    timestamp: Optional[float] = None
    confidence: float = 0.0
    payload: Dict[str, Any] = field(default_factory=dict)


@dataclass
class PipelineResult:
    board_lock: BoardLock
    rectified: Optional[RectifiedBoard]
    normalised: Optional[NormalisedBoard]
    regions: Optional[RegionMasks]
    pieces: Optional[PieceDetectionResult]
    dice_cube: Optional[DiceCubeObservation]
    temporal: Optional[TemporalEstimate]
    validation: Optional[ValidationReport]
    committed: bool
    state_changed: bool
    debug: Dict[str, Any] = field(default_factory=dict)
    events: List[StateEvent] = field(default_factory=list)


def make_empty_region_counts(region_names: List[RegionName]) -> Dict[RegionName, Dict[ColourName, int]]:
    return {name: {"light": 0, "dark": 0} for name in region_names}


__all__ = [
    "BoardLock",
    "RectifiedBoard",
    "NormalisedBoard",
    "RegionMasks",
    "PieceInstance",
    "PieceDetectionResult",
    "DiceObservation",
    "CubeObservation",
    "DiceCubeObservation",
    "BoardState",
    "ValidationReport",
    "TemporalEstimate",
    "StateEvent",
    "PipelineResult",
    "make_empty_region_counts",
]


# 2. Acquisition

Camera streams, keyframe gating, recording, and live diagnostics.


## Streams module

Creates the OAK-D SR RGB/depth streams and exposes simple frame getters.


In [ ]:
%%writefile Acquisition/streams_module.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterator, Literal, Optional, Tuple
import time

import cv2
import depthai as dai
import numpy as np

Size = Tuple[int, int]
Roi = Tuple[int, int, int, int]
SocketName = Literal["left", "right"]


@dataclass(frozen=True)
class DepthCropGeometry:
    """Geometry for the cropped stereo/depth processing area."""

    full_w: int
    full_h: int
    roi_x1: int
    roi_y1: int
    roi_x2: int
    roi_y2: int
    proc_x1: int
    proc_y1: int
    proc_x2: int
    proc_y2: int
    proc_w: int
    proc_h: int
    inner_x1: int
    inner_y1: int
    inner_x2: int
    inner_y2: int


def snap_width_crop_to_multiple_of_16(x1: int, x2: int, full_w: int) -> Tuple[int, int]:
    """Stereo input widths must be multiples of 16 on this DepthAI path."""
    width = int(x2 - x1)
    snapped_width = max(16, (width // 16) * 16)

    center = (x1 + x2) / 2.0
    new_x1 = int(round(center - snapped_width / 2.0))
    new_x2 = new_x1 + snapped_width

    if new_x1 < 0:
        new_x1 = 0
        new_x2 = snapped_width
    if new_x2 > full_w:
        new_x2 = full_w
        new_x1 = full_w - snapped_width

    return new_x1, new_x2


def compute_depth_crop_geometry(
    *,
    full_size: Size = (640, 400),
    roi: Optional[Roi] = (120, 0, 550, 400),
    pad: Tuple[int, int] = (40, 0),
) -> DepthCropGeometry:
    """
    Compute the crop used before StereoDepth.

    The defaults mirror the working camtest.py setup:
    a holder/board ROI is padded before depth is computed, and the crop width is
    snapped to a multiple of 16 so StereoDepth accepts it reliably.
    """
    full_w, full_h = full_size
    if roi is None:
        roi_x1, roi_y1, roi_x2, roi_y2 = 0, 0, full_w, full_h
    else:
        roi_x1, roi_y1, roi_x2, roi_y2 = roi

    pad_x, pad_y = pad
    proc_x1 = max(0, roi_x1 - pad_x)
    proc_y1 = max(0, roi_y1 - pad_y)
    proc_x2 = min(full_w, roi_x2 + pad_x)
    proc_y2 = min(full_h, roi_y2 + pad_y)

    proc_x1, proc_x2 = snap_width_crop_to_multiple_of_16(proc_x1, proc_x2, full_w)

    proc_w = proc_x2 - proc_x1
    proc_h = proc_y2 - proc_y1

    return DepthCropGeometry(
        full_w=full_w,
        full_h=full_h,
        roi_x1=roi_x1,
        roi_y1=roi_y1,
        roi_x2=roi_x2,
        roi_y2=roi_y2,
        proc_x1=proc_x1,
        proc_y1=proc_y1,
        proc_x2=proc_x2,
        proc_y2=proc_y2,
        proc_w=proc_w,
        proc_h=proc_h,
        inner_x1=roi_x1 - proc_x1,
        inner_y1=roi_y1 - proc_y1,
        inner_x2=roi_x2 - proc_x1,
        inner_y2=roi_y2 - proc_y1,
    )


def get_latest_packet(queue):
    """Return the newest packet currently in a non-blocking queue."""
    pkt = queue.tryGet()
    if pkt is None:
        return None
    while True:
        newer = queue.tryGet()
        if newer is None:
            break
        pkt = newer
    return pkt


def depth_to_grayscale(
    depth_frame: np.ndarray,
    near_percentile: float = 3.0,
    far_percentile: float = 95.0,
    invert: bool = False,
    dmin: Optional[float] = None,
    dmax: Optional[float] = None,
) -> np.ndarray:
    """Convert a raw depth frame in mm into an 8-bit grayscale image."""
    valid = depth_frame[depth_frame > 0]
    if valid.size == 0:
        return np.zeros(depth_frame.shape, dtype=np.uint8)

    if dmin is None:
        dmin = float(np.percentile(valid, near_percentile))
    if dmax is None:
        dmax = float(np.percentile(valid, far_percentile))
    if dmax <= dmin:
        dmax = dmin + 1.0

    clipped = np.clip(depth_frame.astype(np.float32), dmin, dmax)
    norm = ((clipped - dmin) * 255.0 / (dmax - dmin)).astype(np.uint8)

    if invert:
        norm = 255 - norm

    norm[depth_frame == 0] = 0
    return norm


def median_filter_from_mode(mode: int):
    if mode <= 0:
        return None
    if mode == 1:
        return dai.MedianFilter.KERNEL_3x3
    if mode == 2:
        return dai.MedianFilter.KERNEL_5x5
    return dai.MedianFilter.KERNEL_7x7


class OakSRStreams:
    """
    Reusable OAK-D SR stream provider for other scripts.

    This version intentionally follows the same DepthAI v3 pipeline pattern as
    the user's working camtest.py:
      - Camera nodes on CAM_B and CAM_C
      - RGB888p preview streams for left/right viewing
      - GRAY8 left/right streams into ImageManip crops
      - cropped GRAY8 stereo inputs into StereoDepth
      - queue maxSize=1, blocking=False, latest-packet reads

    Depth is computed on the cropped processing region, then returned as a
    full-size canvas by default so it spatially matches the left/right previews.
    Use get_depth_crop_frame() when you specifically need the raw cropped depth.
    """

    def __init__(
        self,
        *,
        enable_rgb: bool = False,
        rgb_socket: SocketName = "left",
        enable_left: bool = False,
        enable_right: bool = False,
        enable_depth: bool = False,
        enable_depth_gray: bool = False,
        fps: float = 30.0,
        view_size: Size = (640, 400),
        stereo_size: Optional[Size] = None,
        depth_roi: Optional[Roi] = (120, 0, 550, 400),
        depth_pad: Tuple[int, int] = (40, 0),
        align_mode: Literal["left", "center", "centre"] = "centre",
        lr_check: bool = True,
        lr_threshold: int = 6,
        subpixel: bool = True,
        subpixel_bits: int = 5,
        extended_disparity: bool = False,
        confidence: int = 100,
        median_mode: int = 2,
        temporal_filter: bool = True,
        spatial_filter: bool = True,
        speckle_filter: bool = True,
        device_depth_min_mm: int = 360,
        device_depth_max_mm: int = 450,
    ) -> None:
        if enable_rgb:
            if rgb_socket == "left":
                enable_left = True
            else:
                enable_right = True

        self.enable_left = enable_left
        self.enable_right = enable_right
        self.enable_depth = enable_depth or enable_depth_gray
        self.enable_depth_gray = enable_depth_gray
        self.enable_rgb = enable_rgb
        self.rgb_socket = rgb_socket
        self.view_size = view_size
        self.fps = fps

        if not (self.enable_left or self.enable_right or self.enable_depth):
            raise ValueError("Enable at least one stream: rgb, left, right, depth, or depth_gray.")

        if extended_disparity and subpixel:
            # The working control script disables extended disparity when subpixel is enabled.
            extended_disparity = False

        self.depth_geometry = compute_depth_crop_geometry(
            full_size=view_size,
            roi=depth_roi,
            pad=depth_pad,
        )

        self.pipeline = dai.Pipeline()
        try:
            self.pipeline.setXLinkChunkSize(0)
        except Exception:
            pass

        self._queues: Dict[str, object] = {}
        self._cache: Dict[str, np.ndarray] = {}
        self._running = False

        need_left_cam = self.enable_left or self.enable_depth
        need_right_cam = self.enable_right or self.enable_depth

        self._left_cam = (
            self.pipeline.create(dai.node.Camera).build(dai.CameraBoardSocket.CAM_B)
            if need_left_cam
            else None
        )
        self._right_cam = (
            self.pipeline.create(dai.node.Camera).build(dai.CameraBoardSocket.CAM_C)
            if need_right_cam
            else None
        )

        if self.enable_left:
            left_out = self._left_cam.requestOutput(
                view_size,
                type=dai.ImgFrame.Type.RGB888p,
                fps=fps,
            )
            self._queues["left"] = left_out.createOutputQueue(maxSize=1, blocking=False)

        if self.enable_right:
            right_out = self._right_cam.requestOutput(
                view_size,
                type=dai.ImgFrame.Type.RGB888p,
                fps=fps,
            )
            self._queues["right"] = right_out.createOutputQueue(maxSize=1, blocking=False)

        if self.enable_depth:
            stereo = self.pipeline.create(dai.node.StereoDepth)
            self._stereo = stereo

            full_w, full_h = view_size if stereo_size is None else stereo_size
            if (full_w, full_h) != view_size:
                # Keep this conservative: the crop geometry and full-frame canvas assume view_size.
                raise ValueError("stereo_size must currently match view_size for aligned full-frame depth output.")

            left_proc_src = self._left_cam.requestOutput(
                view_size,
                type=dai.ImgFrame.Type.GRAY8,
                fps=fps,
            )
            right_proc_src = self._right_cam.requestOutput(
                view_size,
                type=dai.ImgFrame.Type.GRAY8,
                fps=fps,
            )

            left_crop = self.pipeline.create(dai.node.ImageManip)
            right_crop = self.pipeline.create(dai.node.ImageManip)

            try:
                left_crop.initialConfig.setFrameType(dai.ImgFrame.Type.GRAY8)
                right_crop.initialConfig.setFrameType(dai.ImgFrame.Type.GRAY8)
            except Exception:
                pass

            g = self.depth_geometry
            left_crop.initialConfig.addCrop(g.proc_x1, g.proc_y1, g.proc_w, g.proc_h)
            right_crop.initialConfig.addCrop(g.proc_x1, g.proc_y1, g.proc_w, g.proc_h)

            try:
                left_crop.setMaxOutputFrameSize(g.proc_w * g.proc_h)
                right_crop.setMaxOutputFrameSize(g.proc_w * g.proc_h)
            except Exception:
                pass

            left_proc_src.link(left_crop.inputImage)
            right_proc_src.link(right_crop.inputImage)
            left_crop.out.link(stereo.left)
            right_crop.out.link(stereo.right)

            stereo.setExtendedDisparity(bool(extended_disparity))
            stereo.setLeftRightCheck(bool(lr_check))
            stereo.setSubpixel(bool(subpixel))

            if subpixel:
                try:
                    stereo.setSubpixelFractionalBits(int(np.clip(subpixel_bits, 3, 5)))
                except Exception:
                    pass

            if align_mode.lower() in ("center", "centre"):
                try:
                    stereo.setDepthAlign(dai.StereoDepthConfig.AlgorithmControl.DepthAlign.CENTER)
                except Exception:
                    try:
                        stereo.initialConfig.setDepthAlign(dai.StereoDepthConfig.AlgorithmControl.DepthAlign.CENTER)
                    except Exception:
                        pass

            try:
                stereo.initialConfig.costMatching.enableSwConfidenceThresholding = True
                stereo.initialConfig.costMatching.confidenceThreshold = int(confidence)
            except Exception:
                try:
                    stereo.initialConfig.setConfidenceThreshold(int(confidence))
                except Exception:
                    pass

            try:
                stereo.initialConfig.setLeftRightCheckThreshold(int(lr_threshold))
            except Exception:
                try:
                    stereo.initialConfig.algorithmControl.leftRightCheckThreshold = int(lr_threshold)
                except Exception:
                    pass

            med = median_filter_from_mode(int(median_mode))
            try:
                stereo.initialConfig.setMedianFilter(med if med is not None else dai.MedianFilter.MEDIAN_OFF)
            except Exception:
                pass

            pp = stereo.initialConfig.postProcessing
            try:
                pp.speckleFilter.enable = bool(speckle_filter)
                pp.speckleFilter.speckleRange = 32
            except Exception:
                pass
            try:
                pp.thresholdFilter.minRange = int(device_depth_min_mm)
                pp.thresholdFilter.maxRange = int(device_depth_max_mm)
            except Exception:
                pass
            try:
                pp.decimationFilter.decimationFactor = 1
            except Exception:
                pass
            try:
                pp.temporalFilter.enable = bool(temporal_filter)
            except Exception:
                pass
            try:
                pp.spatialFilter.enable = bool(spatial_filter)
            except Exception:
                pass
            try:
                pp.holeFilling.enable = False
            except Exception:
                pass

            self._queues["depth_crop"] = stereo.depth.createOutputQueue(maxSize=1, blocking=False)

    def start(self) -> "OakSRStreams":
        if not self._running:
            self.pipeline.start()
            self._running = True
            time.sleep(0.05)
        return self

    def stop(self) -> None:
        if self._running:
            try:
                self.pipeline.stop()
                try:
                    self.pipeline.wait()
                except Exception:
                    pass
            finally:
                self._running = False

    def __enter__(self) -> "OakSRStreams":
        return self.start()

    def __exit__(self, exc_type, exc, tb) -> None:
        self.stop()

    def is_running(self) -> bool:
        try:
            return bool(self._running and self.pipeline.isRunning())
        except Exception:
            return bool(self._running)

    def _read_packet(self, name: str, block: bool, use_cached: bool) -> Optional[np.ndarray]:
        if not self._running:
            raise RuntimeError("Call start() before reading frames.")
        if name not in self._queues:
            raise RuntimeError(f"Stream '{name}' was not enabled.")

        queue = self._queues[name]
        packet = queue.get() if block else get_latest_packet(queue)

        if packet is not None:
            if name in ("left", "right"):
                self._cache[name] = packet.getCvFrame()
            else:
                self._cache[name] = packet.getFrame()

        if packet is not None:
            return self._cache[name]
        if use_cached:
            return self._cache.get(name)
        return None

    def depth_crop_to_full_frame(self, depth_crop: np.ndarray) -> np.ndarray:
        """Place a cropped depth frame back into a full preview-sized canvas."""
        g = self.depth_geometry
        full = np.zeros((g.full_h, g.full_w), dtype=depth_crop.dtype)
        h = min(depth_crop.shape[0], g.proc_h)
        w = min(depth_crop.shape[1], g.proc_w)
        full[g.proc_y1 : g.proc_y1 + h, g.proc_x1 : g.proc_x1 + w] = depth_crop[:h, :w]
        return full

    def get_rgb_frame(self, block: bool = False, use_cached: bool = True) -> Optional[np.ndarray]:
        stream_name = "left" if self.rgb_socket == "left" else "right"
        return self._read_packet(stream_name, block=block, use_cached=use_cached)

    def get_left_frame(self, block: bool = False, use_cached: bool = True) -> Optional[np.ndarray]:
        return self._read_packet("left", block=block, use_cached=use_cached)

    def get_right_frame(self, block: bool = False, use_cached: bool = True) -> Optional[np.ndarray]:
        return self._read_packet("right", block=block, use_cached=use_cached)

    def get_depth_crop_frame(self, block: bool = False, use_cached: bool = True) -> Optional[np.ndarray]:
        return self._read_packet("depth_crop", block=block, use_cached=use_cached)

    def get_depth_frame(
        self,
        block: bool = False,
        use_cached: bool = True,
        as_full_frame: bool = True,
    ) -> Optional[np.ndarray]:
        depth_crop = self.get_depth_crop_frame(block=block, use_cached=use_cached)
        if depth_crop is None:
            return None
        if as_full_frame:
            return self.depth_crop_to_full_frame(depth_crop)
        return depth_crop

    def get_depth_grayscale(
        self,
        block: bool = False,
        use_cached: bool = True,
        near_percentile: float = 3.0,
        far_percentile: float = 95.0,
        invert: bool = False,
        dmin: Optional[float] = None,
        dmax: Optional[float] = None,
        as_full_frame: bool = True,
    ) -> Optional[np.ndarray]:
        depth = self.get_depth_frame(block=block, use_cached=use_cached, as_full_frame=as_full_frame)
        if depth is None:
            return None
        return depth_to_grayscale(
            depth,
            near_percentile=near_percentile,
            far_percentile=far_percentile,
            invert=invert,
            dmin=dmin,
            dmax=dmax,
        )

    def iter_rgb(self) -> Iterator[np.ndarray]:
        while self._running:
            frame = self.get_rgb_frame(block=True, use_cached=False)
            if frame is not None:
                yield frame

    def iter_left(self) -> Iterator[np.ndarray]:
        while self._running:
            frame = self.get_left_frame(block=True, use_cached=False)
            if frame is not None:
                yield frame

    def iter_right(self) -> Iterator[np.ndarray]:
        while self._running:
            frame = self.get_right_frame(block=True, use_cached=False)
            if frame is not None:
                yield frame

    def iter_depth(self, *, as_full_frame: bool = True) -> Iterator[np.ndarray]:
        while self._running:
            frame = self.get_depth_frame(block=True, use_cached=False, as_full_frame=as_full_frame)
            if frame is not None:
                yield frame

    def iter_depth_grayscale(
        self,
        near_percentile: float = 3.0,
        far_percentile: float = 95.0,
        invert: bool = False,
    ) -> Iterator[np.ndarray]:
        while self._running:
            frame = self.get_depth_grayscale(
                block=True,
                use_cached=False,
                near_percentile=near_percentile,
                far_percentile=far_percentile,
                invert=invert,
            )
            if frame is not None:
                yield frame


def create_rgb_stream(rgb_socket: SocketName = "left", **kwargs) -> OakSRStreams:
    return OakSRStreams(enable_rgb=True, rgb_socket=rgb_socket, **kwargs)


def create_left_stream(**kwargs) -> OakSRStreams:
    return OakSRStreams(enable_left=True, **kwargs)


def create_right_stream(**kwargs) -> OakSRStreams:
    return OakSRStreams(enable_right=True, **kwargs)


def create_depth_stream(**kwargs) -> OakSRStreams:
    return OakSRStreams(enable_depth=True, **kwargs)


def create_depth_gray_stream(**kwargs) -> OakSRStreams:
    return OakSRStreams(enable_depth_gray=True, **kwargs)


def create_all_streams(rgb_socket: SocketName = "left", **kwargs) -> OakSRStreams:
    return OakSRStreams(
        enable_rgb=True,
        rgb_socket=rgb_socket,
        enable_left=True,
        enable_right=True,
        enable_depth=True,
        enable_depth_gray=True,
        **kwargs,
    )


__all__ = [
    "OakSRStreams",
    "DepthCropGeometry",
    "compute_depth_crop_geometry",
    "snap_width_crop_to_multiple_of_16",
    "get_latest_packet",
    "depth_to_grayscale",
    "create_rgb_stream",
    "create_left_stream",
    "create_right_stream",
    "create_depth_stream",
    "create_depth_gray_stream",
    "create_all_streams",
]


## Keyframe selection

Filters unstable or occluded frames and packages the accepted frame data.


In [ ]:
%%writefile Acquisition/keyframe_gate.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterator, Optional, Tuple
import time

import cv2
import numpy as np

from streams_module import OakSRStreams, depth_to_grayscale


RoiFrac = Tuple[float, float, float, float]


@dataclass
class KeyframePacket:
    timestamp: float
    frame_index: int

    left_raw: np.ndarray
    left_norm: np.ndarray
    left_gray: np.ndarray

    depth_raw: np.ndarray
    depth_gray: np.ndarray
    depth_valid_mask: np.ndarray
    near_mask: np.ndarray

    motion_score: float
    rgb_motion_score: float
    depth_motion_score: float
    novelty_score: float

    near_ratio: float
    invalid_ratio: float

    stable: bool
    occluded: bool
    keyframe: bool
    stable_count: int


class OakSRKeyframeGate:
    """
    Wraps streams_module.OakSRStreams and outputs processed frame packets
    plus keyframe decisions.

    Intended use:
      - Iterate all processed packets with iter_packets()
      - Or iterate only accepted keyframes with iter_keyframes()

    Main idea:
      - RGB stability gate via frame differencing
      - Depth hand/occlusion gate via near-field ratio + invalid depth ratio
      - Keyframe only after N stable frames and enough change from the last keyframe
    """

    def __init__(
        self,
        *,
        fps: float = 30.0,
        view_size: Tuple[int, int] = (640, 400),
        roi_frac: RoiFrac = (0.08, 0.08, 0.92, 0.92),
        motion_threshold: float = 2.5,
        min_stable_frames: int = 5,
        min_keyframe_gap_frames: int = 10,
        min_keyframe_delta: float = 1.5,
        hand_near_mm: int = 450,
        min_valid_mm: int = 80,
        max_invalid_ratio: float = 0.80,
        max_near_ratio: float = 0.03,
        ema_alpha: float = 0.2,
        wb_strength: float = 1.0,
        clahe_clip_limit: float = 2.0,
        clahe_tile_grid: Tuple[int, int] = (8, 8),
        depth_blur_ksize: int = 5,
        enable_right: bool = False,
        depth_roi: Optional[Tuple[int, int, int, int]] = (120, 0, 550, 400),
        depth_pad: Tuple[int, int] = (40, 0),
        streams: Optional[OakSRStreams] = None,
    ) -> None:
        # Accept an externally-created stream object so callers such
        # as live_stream_viewer.py do not accidentally construct two DepthAI
        # pipelines before starting one. The earlier version did that, which
        # could leave the OAK-D SR in a failed boot state.
        self.streams = streams if streams is not None else OakSRStreams(
            enable_left=True,
            enable_right=enable_right,
            enable_depth=True,
            fps=fps,
            view_size=view_size,
            stereo_size=view_size,
            depth_roi=depth_roi,
            depth_pad=depth_pad,
            lr_check=True,
            subpixel=True,
            extended_disparity=False,
        )

        self.roi_frac = roi_frac
        self.motion_threshold = motion_threshold
        self.min_stable_frames = min_stable_frames
        self.min_keyframe_gap_frames = min_keyframe_gap_frames
        self.min_keyframe_delta = min_keyframe_delta

        self.hand_near_mm = hand_near_mm
        self.min_valid_mm = min_valid_mm
        self.max_invalid_ratio = max_invalid_ratio
        self.max_near_ratio = max_near_ratio

        self.ema_alpha = float(np.clip(ema_alpha, 0.01, 0.99))
        self.wb_strength = float(np.clip(wb_strength, 0.0, 1.0))
        self.clahe_clip_limit = clahe_clip_limit
        self.clahe_tile_grid = clahe_tile_grid
        self.depth_blur_ksize = max(3, depth_blur_ksize | 1)  # force odd

        self._frame_index = 0
        self._stable_count = 0
        self._last_keyframe_index = -10_000

        self._prev_left_gray_roi: Optional[np.ndarray] = None
        self._prev_depth_gray_roi: Optional[np.ndarray] = None
        self._last_keyframe_gray_roi: Optional[np.ndarray] = None

        self._running = False

    def start(self) -> "OakSRKeyframeGate":
        self.streams.start()
        self._running = True
        return self

    def stop(self) -> None:
        self.streams.stop()
        self._running = False

    def __enter__(self) -> "OakSRKeyframeGate":
        return self.start()

    def __exit__(self, exc_type, exc, tb) -> None:
        self.stop()

    @staticmethod
    def _gray_world_white_balance(img_bgr: np.ndarray, strength: float = 1.0) -> np.ndarray:
        img = img_bgr.astype(np.float32)
        means = img.reshape(-1, 3).mean(axis=0)
        gray_mean = float(means.mean()) + 1e-6
        scale = gray_mean / (means + 1e-6)
        balanced = img * scale.reshape(1, 1, 3)
        mixed = img * (1.0 - strength) + balanced * strength
        return np.clip(mixed, 0, 255).astype(np.uint8)

    def _normalize_rgb(self, frame_bgr: np.ndarray) -> np.ndarray:
        wb = self._gray_world_white_balance(frame_bgr, strength=self.wb_strength)

        lab = cv2.cvtColor(wb, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=self.clahe_clip_limit,
            tileGridSize=self.clahe_tile_grid,
        )
        l_eq = clahe.apply(l)

        merged = cv2.merge((l_eq, a, b))
        out = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

        out = cv2.GaussianBlur(out, (3, 3), 0)
        return out

    def _normalize_depth_gray(self, depth_raw: np.ndarray) -> np.ndarray:
        gray = depth_to_grayscale(
            depth_raw,
            near_percentile=3.0,
            far_percentile=95.0,
            invert=False,
        )
        gray = cv2.medianBlur(gray, self.depth_blur_ksize)
        return gray

    @staticmethod
    def _resize_to_match(src: np.ndarray, target_shape_hw: Tuple[int, int], is_mask: bool = False) -> np.ndarray:
        h, w = target_shape_hw
        interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
        return cv2.resize(src, (w, h), interpolation=interp)

    def _roi_bounds(self, h: int, w: int) -> Tuple[int, int, int, int]:
        x1f, y1f, x2f, y2f = self.roi_frac
        x1 = int(np.clip(x1f, 0.0, 1.0) * w)
        y1 = int(np.clip(y1f, 0.0, 1.0) * h)
        x2 = int(np.clip(x2f, 0.0, 1.0) * w)
        y2 = int(np.clip(y2f, 0.0, 1.0) * h)

        if x2 <= x1:
            x2 = min(w, x1 + 1)
        if y2 <= y1:
            y2 = min(h, y1 + 1)

        return x1, y1, x2, y2

    def _extract_roi(self, img: np.ndarray) -> np.ndarray:
        h, w = img.shape[:2]
        x1, y1, x2, y2 = self._roi_bounds(h, w)
        return img[y1:y2, x1:x2]

    @staticmethod
    def _mean_absdiff(a: np.ndarray, b: np.ndarray) -> float:
        diff = cv2.absdiff(a, b)
        return float(diff.mean())

    def _compute_motion_scores(
        self,
        left_gray_roi: np.ndarray,
        depth_gray_roi: np.ndarray,
    ) -> Tuple[float, float, float]:
        if self._prev_left_gray_roi is None or self._prev_depth_gray_roi is None:
            rgb_motion = 999.0
            depth_motion = 999.0
        else:
            rgb_motion = self._mean_absdiff(left_gray_roi, self._prev_left_gray_roi)
            depth_motion = self._mean_absdiff(depth_gray_roi, self._prev_depth_gray_roi)

        combined = 0.75 * rgb_motion + 0.25 * depth_motion
        return combined, rgb_motion, depth_motion

    def _compute_novelty_score(self, left_gray_roi: np.ndarray) -> float:
        if self._last_keyframe_gray_roi is None:
            return 999.0
        return self._mean_absdiff(left_gray_roi, self._last_keyframe_gray_roi)

    def _compute_occlusion_metrics(
        self,
        depth_raw: np.ndarray,
        target_shape_hw: Tuple[int, int],
    ) -> Tuple[np.ndarray, np.ndarray, float, float, bool]:
        valid_mask = (depth_raw >= self.min_valid_mm).astype(np.uint8)

        # Use a dynamic near-field threshold relative to the current board depth.
        # The earlier fixed threshold (e.g. 450 mm) treated most of the board as
        # an occluder when the board itself was around 390-430 mm from camera.
        # Here, only things substantially closer than the median board/ROI depth
        # are classed as near occlusions. This should catch hands while not
        # rejecting the board or normal checker stacks.
        roi_depth = self._extract_roi(depth_raw)
        roi_valid = roi_depth[roi_depth >= self.min_valid_mm]
        if roi_valid.size > 0:
            board_median_mm = float(np.median(roi_valid))
            dynamic_near_mm = min(float(self.hand_near_mm), board_median_mm - 40.0)
        else:
            dynamic_near_mm = float(self.hand_near_mm)

        near_mask = (
            (depth_raw >= self.min_valid_mm)
            & (depth_raw < dynamic_near_mm)
        ).astype(np.uint8)

        kernel = np.ones((5, 5), np.uint8)
        near_mask = cv2.morphologyEx(near_mask * 255, cv2.MORPH_OPEN, kernel)
        near_mask = cv2.morphologyEx(near_mask, cv2.MORPH_CLOSE, kernel)
        near_mask = (near_mask > 0).astype(np.uint8)

        valid_mask = self._resize_to_match(valid_mask, target_shape_hw, is_mask=True)
        near_mask = self._resize_to_match(near_mask, target_shape_hw, is_mask=True)

        valid_roi = self._extract_roi(valid_mask)
        near_roi = self._extract_roi(near_mask)

        invalid_ratio = 1.0 - float(valid_roi.mean())
        near_ratio = float(near_roi.mean())

        occluded = (near_ratio > self.max_near_ratio) or (invalid_ratio > self.max_invalid_ratio)
        return valid_mask, near_mask, near_ratio, invalid_ratio, occluded

    def get_packet(self, block: bool = True) -> Optional[KeyframePacket]:
        if not self._running:
            raise RuntimeError("Call start() before reading packets.")

        left_raw = self.streams.get_left_frame(block=block, use_cached=True)
        depth_raw = self.streams.get_depth_frame(block=block, use_cached=True)

        if left_raw is None or depth_raw is None:
            return None

        left_norm = self._normalize_rgb(left_raw)
        left_gray = cv2.cvtColor(left_norm, cv2.COLOR_BGR2GRAY)

        depth_gray = self._normalize_depth_gray(depth_raw)

        if depth_gray.shape[:2] != left_gray.shape[:2]:
            depth_gray = self._resize_to_match(depth_gray, left_gray.shape[:2], is_mask=False)

        left_gray_roi = self._extract_roi(left_gray)
        depth_gray_roi = self._extract_roi(depth_gray)

        motion_score, rgb_motion_score, depth_motion_score = self._compute_motion_scores(
            left_gray_roi,
            depth_gray_roi,
        )

        valid_mask, near_mask, near_ratio, invalid_ratio, occluded = self._compute_occlusion_metrics(
            depth_raw,
            left_gray.shape[:2],
        )

        stable = (motion_score <= self.motion_threshold) and not occluded

        if stable:
            self._stable_count += 1
        else:
            self._stable_count = 0

        novelty_score = self._compute_novelty_score(left_gray_roi)

        enough_gap = (self._frame_index - self._last_keyframe_index) >= self.min_keyframe_gap_frames
        keyframe = (
            self._stable_count >= self.min_stable_frames
            and enough_gap
            and not occluded
            and invalid_ratio <= self.max_invalid_ratio
            and novelty_score >= self.min_keyframe_delta
        )

        if keyframe:
            self._last_keyframe_gray_roi = left_gray_roi.copy()
            self._last_keyframe_index = self._frame_index

        self._prev_left_gray_roi = left_gray_roi.copy()
        self._prev_depth_gray_roi = depth_gray_roi.copy()

        packet = KeyframePacket(
            timestamp=time.time(),
            frame_index=self._frame_index,
            left_raw=left_raw,
            left_norm=left_norm,
            left_gray=left_gray,
            depth_raw=depth_raw,
            depth_gray=depth_gray,
            depth_valid_mask=valid_mask,
            near_mask=near_mask,
            motion_score=motion_score,
            rgb_motion_score=rgb_motion_score,
            depth_motion_score=depth_motion_score,
            novelty_score=novelty_score,
            near_ratio=near_ratio,
            invalid_ratio=invalid_ratio,
            stable=stable,
            occluded=occluded,
            keyframe=keyframe,
            stable_count=self._stable_count,
        )

        self._frame_index += 1
        return packet

    def iter_packets(self, block: bool = True) -> Iterator[KeyframePacket]:
        while self._running:
            packet = self.get_packet(block=block)
            if packet is not None:
                yield packet

    def iter_keyframes(self, block: bool = True) -> Iterator[KeyframePacket]:
        while self._running:
            packet = self.get_packet(block=block)
            if packet is not None and packet.keyframe:
                yield packet


if __name__ == "__main__":
    gate = OakSRKeyframeGate(
        motion_threshold=2.5,
        min_stable_frames=5,
        min_keyframe_gap_frames=10,
        min_keyframe_delta=1.5,
        hand_near_mm=450,
        max_near_ratio=0.03,
        max_invalid_ratio=0.80,
    ).start()

    try:
        for packet in gate.iter_packets():
            rgb_vis = packet.left_norm.copy()

            h, w = rgb_vis.shape[:2]
            x1 = int(0.08 * w)
            y1 = int(0.08 * h)
            x2 = int(0.92 * w)
            y2 = int(0.92 * h)
            cv2.rectangle(rgb_vis, (x1, y1), (x2, y2), (0, 255, 255), 1)

            status = f"stable={packet.stable} occluded={packet.occluded} keyframe={packet.keyframe}"
            stats = (
                f"motion={packet.motion_score:.2f} "
                f"near={packet.near_ratio:.3f} "
                f"invalid={packet.invalid_ratio:.3f} "
                f"stable_count={packet.stable_count}"
            )

            cv2.putText(rgb_vis, status, (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            cv2.putText(rgb_vis, stats, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

            near_vis = (packet.near_mask * 255).astype(np.uint8)
            valid_vis = (packet.depth_valid_mask * 255).astype(np.uint8)

            cv2.imshow("Left Normalised", rgb_vis)
            cv2.imshow("Depth BW", packet.depth_gray)
            cv2.imshow("Near/Occlusion Mask", near_vis)
            cv2.imshow("Depth Valid Mask", valid_vis)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        gate.stop()
        cv2.destroyAllWindows()


## RGB-D recorder for dataset creation

Records RGB-D frames, depth arrays, optional right-camera frames and metadata for later testing.


In [ ]:
%%writefile Acquisition/rgbd_recorder.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import json
import time

import cv2
import numpy as np

from streams_module import OakSRStreams, depth_to_grayscale
from keyframe_gate import OakSRKeyframeGate


@dataclass
class RecorderConfig:
    root_dir: str = "recordings"
    session_name: Optional[str] = None
    fps: float = 30.0
    view_size: tuple[int, int] = (640, 400)
    save_right: bool = False
    save_depth_npy: bool = True
    save_depth_png: bool = True
    use_keyframe_gate: bool = False
    keyframes_only: bool = False


class RGBDRecorder:
    """
    Records RGB-D data for model training.

    Output layout:
      recordings/<session_name>/
        rgb/
        depth_raw_npy/
        depth_gray_png/
        right/                  # optional
        metadata/
          frames.jsonl
          session.json
    """

    def __init__(self, config: Optional[RecorderConfig] = None) -> None:
        self.config = config or RecorderConfig()
        session_name = self.config.session_name or time.strftime("session_%Y%m%d_%H%M%S")
        self.session_dir = Path(self.config.root_dir) / session_name

        (self.session_dir / "rgb").mkdir(parents=True, exist_ok=True)
        (self.session_dir / "metadata").mkdir(parents=True, exist_ok=True)

        if self.config.save_depth_npy:
            (self.session_dir / "depth_raw_npy").mkdir(parents=True, exist_ok=True)
        if self.config.save_depth_png:
            (self.session_dir / "depth_gray_png").mkdir(parents=True, exist_ok=True)
        if self.config.save_right:
            (self.session_dir / "right").mkdir(parents=True, exist_ok=True)

        self.frame_index = 0
        self.metadata_path = self.session_dir / "metadata" / "frames.jsonl"

        if self.config.use_keyframe_gate:
            self.gate = OakSRKeyframeGate(fps=self.config.fps, view_size=self.config.view_size)
            self.streams = None
        else:
            self.gate = None
            self.streams = OakSRStreams(
                enable_left=True,
                enable_right=self.config.save_right,
                enable_depth=True,
                fps=self.config.fps,
                view_size=self.config.view_size,
                stereo_size=self.config.view_size,
            )

    def start(self) -> "RGBDRecorder":
        if self.gate is not None:
            self.gate.start()
        if self.streams is not None:
            self.streams.start()

        session_info = {
            "root_dir": str(self.session_dir),
            "fps": self.config.fps,
            "view_size": list(self.config.view_size),
            "save_right": self.config.save_right,
            "save_depth_npy": self.config.save_depth_npy,
            "save_depth_png": self.config.save_depth_png,
            "use_keyframe_gate": self.config.use_keyframe_gate,
            "keyframes_only": self.config.keyframes_only,
            "created_at": time.time(),
        }
        with open(self.session_dir / "metadata" / "session.json", "w", encoding="utf-8") as f:
            json.dump(session_info, f, indent=2)

        return self

    def stop(self) -> None:
        if self.gate is not None:
            self.gate.stop()
        if self.streams is not None:
            self.streams.stop()

    def __enter__(self) -> "RGBDRecorder":
        return self.start()

    def __exit__(self, exc_type, exc, tb) -> None:
        self.stop()

    def _write_metadata(self, record: dict) -> None:
        with open(self.metadata_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

    def record_next(self) -> Optional[dict]:
        timestamp = time.time()
        idx = self.frame_index

        if self.gate is not None:
            packet = self.gate.get_packet(block=True)
            if packet is None:
                return None
            if self.config.keyframes_only and not packet.keyframe:
                return None

            left = packet.left_raw
            depth_raw = packet.depth_raw
            depth_gray = packet.depth_gray
            right = None
            keyframe = packet.keyframe
        else:
            left = self.streams.get_left_frame(block=True, use_cached=False)
            depth_raw = self.streams.get_depth_frame(block=True, use_cached=False)
            right = self.streams.get_right_frame(block=False, use_cached=True) if self.config.save_right else None

            if left is None or depth_raw is None:
                return None

            depth_gray = depth_to_grayscale(depth_raw)
            keyframe = None

        stem = f"{idx:06d}"
        cv2.imwrite(str(self.session_dir / "rgb" / f"{stem}.png"), left)

        if self.config.save_depth_npy:
            np.save(self.session_dir / "depth_raw_npy" / f"{stem}.npy", depth_raw)
        if self.config.save_depth_png:
            cv2.imwrite(str(self.session_dir / "depth_gray_png" / f"{stem}.png"), depth_gray)
        if self.config.save_right and right is not None:
            cv2.imwrite(str(self.session_dir / "right" / f"{stem}.png"), right)

        record = {
            "frame_index": idx,
            "timestamp": timestamp,
            "rgb_path": f"rgb/{stem}.png",
            "depth_npy_path": f"depth_raw_npy/{stem}.npy" if self.config.save_depth_npy else None,
            "depth_png_path": f"depth_gray_png/{stem}.png" if self.config.save_depth_png else None,
            "right_path": f"right/{stem}.png" if self.config.save_right and right is not None else None,
            "keyframe": keyframe,
        }
        self._write_metadata(record)
        self.frame_index += 1
        return record

    def record_frames(self, max_frames: int) -> list[dict]:
        records = []
        while len(records) < max_frames:
            record = self.record_next()
            if record is not None:
                records.append(record)
        return records


## Live stream viewer / camera diagnostics

Views raw streams, keyframe-gate outputs, preprocessing previews, and troubleshooting recordings.


In [ ]:
%%writefile Acquisition/live_stream_viewer.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Literal, Optional, Tuple
import argparse
import json
import re
import sys
import time
from datetime import datetime

import cv2
import numpy as np

_MODULE_DIR = Path(__file__).resolve().parent
_PROJECT_ROOT = _MODULE_DIR.parent if _MODULE_DIR.name in {"Acquisition", "Geometric + Visual Preprocessing", "Interpretation"} else _MODULE_DIR
for _rel in ("", "Acquisition", "Geometric + Visual Preprocessing", "Interpretation"):
    _path = _PROJECT_ROOT / _rel if _rel else _PROJECT_ROOT
    if _path.exists():
        _path_str = str(_path)
        if _path_str not in sys.path:
            sys.path.insert(0, _path_str)

from streams_module import OakSRStreams, depth_to_grayscale
from keyframe_gate import OakSRKeyframeGate, KeyframePacket

try:
    from board_registration import BoardRegistrar
    from perspective_rectification import PerspectiveRectifier
    from image_normalisation import BoardNormaliser
    from point_tray_segmentation import PointTraySegmenter
except Exception:  # pragma: no cover - lets acquisition-only viewing still work
    BoardRegistrar = None
    PerspectiveRectifier = None
    BoardNormaliser = None
    PointTraySegmenter = None


ViewMode = Literal["streams", "gate", "preprocess", "all"]
Size = Tuple[int, int]


@dataclass
class LiveViewConfig:
    """Configuration for the live diagnostic stream viewer."""

    mode: ViewMode = "all"
    fps: float = 30.0
    view_size: Size = (640, 400)
    grid_cell_size: Size = (420, 260)
    grid_columns: int = 3
    window_name: str = "OAK-D SR live stream viewer"

    motion_threshold: float = 2.5
    hand_near_mm: int = 450
    max_near_ratio: float = 0.03
    max_invalid_ratio: float = 0.80

    enable_preprocessing: bool = True
    min_piece_height_mm: float = 3.0
    max_piece_height_mm: float = 40.0
    min_piece_area_px: int = 80
    height_max_mm: float = 35.0
    auto_baseline: bool = True
    auto_baseline_frames: int = 30
    baseline_min_valid_ratio: float = 0.20

    show_help: bool = True

    record: bool = False
    record_dir: Path = Path("troubleshooting_recordings")
    record_every: int = 1
    record_dashboard: bool = True
    record_views: bool = True
    record_raw_npz: bool = False
    record_codec: str = "mp4v"


def parse_size(text: str) -> Size:
    """Parse a size string such as '640x400' into (width, height)."""
    if "x" not in text.lower():
        raise argparse.ArgumentTypeError("Size must be written as WIDTHxHEIGHT, e.g. 640x400")
    w_str, h_str = text.lower().split("x", 1)
    return int(w_str), int(h_str)


def ensure_bgr(image: np.ndarray) -> np.ndarray:
    """Convert grayscale/mask/float images into uint8 BGR for display."""
    if image is None:
        return np.zeros((100, 100, 3), dtype=np.uint8)

    arr = image
    if arr.dtype == bool:
        arr = arr.astype(np.uint8) * 255
    elif np.issubdtype(arr.dtype, np.floating):
        finite = np.isfinite(arr)
        if not np.any(finite):
            arr = np.zeros(arr.shape, dtype=np.uint8)
        else:
            lo = float(np.percentile(arr[finite], 2))
            hi = float(np.percentile(arr[finite], 98))
            if hi <= lo:
                hi = lo + 1.0
            arr = np.clip((arr - lo) * 255.0 / (hi - lo), 0, 255).astype(np.uint8)
    elif arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)

    if arr.ndim == 2:
        return cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)
    if arr.ndim == 3 and arr.shape[2] == 1:
        return cv2.cvtColor(arr[:, :, 0], cv2.COLOR_GRAY2BGR)
    if arr.ndim == 3 and arr.shape[2] == 3:
        return arr.copy()

    raise ValueError(f"Unsupported image shape for display: {arr.shape}")


def resize_letterbox(image_bgr: np.ndarray, size_wh: Size) -> np.ndarray:
    """Resize image to fit inside size while preserving aspect ratio."""
    target_w, target_h = size_wh
    h, w = image_bgr.shape[:2]
    if h == 0 or w == 0:
        return np.zeros((target_h, target_w, 3), dtype=np.uint8)

    scale = min(target_w / w, target_h / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    resized = cv2.resize(image_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    x0 = (target_w - new_w) // 2
    y0 = (target_h - new_h) // 2
    canvas[y0 : y0 + new_h, x0 : x0 + new_w] = resized
    return canvas


def add_label(image_bgr: np.ndarray, label: str, *, ok: Optional[bool] = None) -> np.ndarray:
    """Add a small label strip to a display image."""
    out = image_bgr.copy()
    h, w = out.shape[:2]
    strip_h = 28
    cv2.rectangle(out, (0, 0), (w, strip_h), (0, 0, 0), -1)

    colour = (230, 230, 230)
    if ok is True:
        colour = (0, 220, 0)
    elif ok is False:
        colour = (0, 0, 255)

    cv2.putText(out, label, (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.52, colour, 1, cv2.LINE_AA)
    return out



def sanitise_filename(text: str) -> str:
    """Make a stream/view label safe for use as a filename."""
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", text.strip().lower())
    safe = re.sub(r"_+", "_", safe).strip("_")
    return safe or "view"


def render_view_cell(label: str, image: np.ndarray, ok: Optional[bool], cell_size: Size) -> np.ndarray:
    """Render one named view exactly like a dashboard grid cell."""
    bgr = ensure_bgr(image)
    cell = resize_letterbox(bgr, cell_size)
    return add_label(cell, label, ok=ok)


def make_grid(items: Iterable[Tuple[str, np.ndarray, Optional[bool]]], *, cell_size: Size, columns: int) -> np.ndarray:
    """Build a labelled image grid from named views."""
    cells: List[np.ndarray] = []
    for label, image, ok in items:
        cells.append(render_view_cell(label, image, ok, cell_size))

    if not cells:
        w, h = cell_size
        return np.zeros((h, w, 3), dtype=np.uint8)

    columns = max(1, int(columns))
    rows = int(np.ceil(len(cells) / columns))
    w, h = cell_size
    blank = np.zeros((h, w, 3), dtype=np.uint8)

    padded = cells + [blank] * (rows * columns - len(cells))
    row_imgs = []
    for r in range(rows):
        row_imgs.append(np.hstack(padded[r * columns : (r + 1) * columns]))
    return np.vstack(row_imgs)


def draw_gate_status(frame_bgr: np.ndarray, packet: KeyframePacket) -> np.ndarray:
    """Overlay keyframe-gate scores onto an RGB frame."""
    out = frame_bgr.copy()
    colour = (0, 255, 0) if packet.keyframe else ((0, 200, 255) if packet.stable else (0, 0, 255))

    lines = [
        f"frame={packet.frame_index} stable={packet.stable} keyframe={packet.keyframe}",
        f"occluded={packet.occluded} stable_count={packet.stable_count}",
        f"motion={packet.motion_score:.2f} rgb={packet.rgb_motion_score:.2f} depth={packet.depth_motion_score:.2f}",
        f"near={packet.near_ratio:.3f} invalid={packet.invalid_ratio:.3f} novelty={packet.novelty_score:.2f}",
    ]

    y = 24
    for line in lines:
        cv2.putText(out, line, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.52, colour, 2, cv2.LINE_AA)
        y += 24
    return out


def draw_board_lock(frame_bgr: np.ndarray, lock) -> np.ndarray:
    """Draw board-registration corners on a copy of the input frame."""
    out = frame_bgr.copy()
    if lock is None or not getattr(lock, "valid", False):
        cv2.putText(out, "board lock: invalid", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2, cv2.LINE_AA)
        return out

    corners = lock.corners_xy.astype(np.int32).reshape(-1, 1, 2)
    cv2.polylines(out, [corners], isClosed=True, color=(0, 255, 0), thickness=2)
    for idx, pt in enumerate(corners.reshape(-1, 2)):
        cv2.circle(out, tuple(pt), 5, (0, 255, 255), -1)
        cv2.putText(out, str(idx), tuple(pt + np.array([6, -6])), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

    cv2.putText(
        out,
        f"board lock: {lock.method} conf={lock.confidence:.2f}",
        (10, 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2,
        cv2.LINE_AA,
    )
    return out


def build_piece_height_mask(
    height_map_mm: np.ndarray,
    valid_mask: np.ndarray,
    *,
    min_height_mm: float,
    max_height_mm: float,
    min_area_px: int,
) -> np.ndarray:
    """Create a simple black/white pre-CNN piece candidate mask from height above board."""
    mask = (
        (height_map_mm >= min_height_mm)
        & (height_map_mm <= max_height_mm)
        & (valid_mask > 0)
    ).astype(np.uint8) * 255

    open_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    close_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, open_k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, close_k)

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    cleaned = np.zeros_like(mask)
    for label in range(1, n_labels):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area >= min_area_px:
            cleaned[labels == label] = 255
    return cleaned


class DepthBaselineModel:
    """
    Empty-board baseline model for board/chip height diagnostics.

    This is deliberately independent of board registration. It works directly
    on the cropped/full-frame depth stream, so it is available even when the
    geometric preprocessing/rectification stage has not locked the board yet.
    """

    def __init__(
        self,
        *,
        frames_required: int = 30,
        min_valid_ratio: float = 0.20,
        height_max_mm: float = 35.0,
        min_piece_height_mm: float = 3.0,
        max_piece_height_mm: float = 40.0,
        min_piece_area_px: int = 80,
    ) -> None:
        self.frames_required = max(1, int(frames_required))
        self.min_valid_ratio = float(min_valid_ratio)
        self.height_max_mm = float(height_max_mm)
        self.min_piece_height_mm = float(min_piece_height_mm)
        self.max_piece_height_mm = float(max_piece_height_mm)
        self.min_piece_area_px = int(min_piece_area_px)

        self.baseline_depth_mm: Optional[np.ndarray] = None
        self._samples: List[np.ndarray] = []
        self._capture_requested = False
        self.status = "baseline: collecting empty-board frames"

    def clear(self) -> None:
        self.baseline_depth_mm = None
        self._samples.clear()
        self._capture_requested = True
        self.status = "baseline: cleared, collecting"

    def request_capture(self) -> None:
        self.baseline_depth_mm = None
        self._samples.clear()
        self._capture_requested = True
        self.status = "baseline: capture requested"

    @property
    def ready(self) -> bool:
        return self.baseline_depth_mm is not None

    def update(self, depth_mm: np.ndarray, *, allow_auto_capture: bool = True) -> None:
        if depth_mm is None:
            return
        if self.ready and not self._capture_requested:
            return
        if not allow_auto_capture and not self._capture_requested:
            return

        valid_ratio = float(np.count_nonzero(depth_mm > 0)) / float(depth_mm.size)
        if valid_ratio < self.min_valid_ratio:
            self.status = f"baseline: waiting for valid depth ({valid_ratio:.2f})"
            return

        self._samples.append(depth_mm.copy())
        self.status = f"baseline: collecting {len(self._samples)}/{self.frames_required}"

        if len(self._samples) < self.frames_required:
            return

        stack = np.stack([np.where(f > 0, f.astype(np.float32), np.nan) for f in self._samples], axis=0)
        baseline = np.nanmedian(stack, axis=0)
        baseline[np.isnan(baseline)] = 0
        self.baseline_depth_mm = baseline.astype(np.float32)
        self._samples.clear()
        self._capture_requested = False

        valid_ratio = float(np.count_nonzero(self.baseline_depth_mm > 0)) / float(self.baseline_depth_mm.size)
        self.status = f"baseline: ready valid={valid_ratio:.2f}"

    def compute(self, depth_mm: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Returns: height_mm, height_uint8, piece_mask, overlap_valid_mask.
        """
        if self.baseline_depth_mm is None:
            blank = np.zeros(depth_mm.shape[:2], dtype=np.uint8)
            return np.zeros(depth_mm.shape[:2], dtype=np.float32), blank, blank, blank

        depth = depth_mm.astype(np.float32)
        baseline = self.baseline_depth_mm
        if baseline.shape != depth.shape:
            baseline = cv2.resize(baseline, (depth.shape[1], depth.shape[0]), interpolation=cv2.INTER_NEAREST)

        current_valid = depth > 0
        baseline_valid = baseline > 0
        overlap_valid = current_valid & baseline_valid

        height = baseline - depth
        height[height < 0] = 0
        height[~overlap_valid] = 0

        height_vis = np.clip(height, 0, max(1.0, self.height_max_mm))
        height_uint8 = (height_vis * 255.0 / max(1.0, self.height_max_mm)).astype(np.uint8)
        height_uint8[~overlap_valid] = 0

        piece_mask = build_piece_height_mask(
            height,
            overlap_valid.astype(np.uint8),
            min_height_mm=self.min_piece_height_mm,
            max_height_mm=self.max_piece_height_mm,
            min_area_px=self.min_piece_area_px,
        )

        return height.astype(np.float32), height_uint8, piece_mask, overlap_valid.astype(np.uint8) * 255



def blend_overlay(base_bgr: np.ndarray, overlay_bgr: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """Blend a sparse overlay onto a base image."""
    if overlay_bgr.shape[:2] != base_bgr.shape[:2]:
        overlay_bgr = cv2.resize(overlay_bgr, (base_bgr.shape[1], base_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
    mask = np.any(overlay_bgr > 0, axis=2)
    out = base_bgr.copy()
    out[mask] = cv2.addWeighted(base_bgr, 1.0 - alpha, overlay_bgr, alpha, 0)[mask]
    return out


class PreprocessingPreview:
    """Runs the pre-CNN geometric/visual pipeline for live diagnostics."""

    def __init__(self, config: LiveViewConfig) -> None:
        if any(x is None for x in (BoardRegistrar, PerspectiveRectifier, BoardNormaliser, PointTraySegmenter)):
            raise RuntimeError("Preprocessing modules are not importable. Check project paths.")

        self.config = config
        self.registrar = BoardRegistrar()
        self.rectifier = PerspectiveRectifier()
        self.normaliser = BoardNormaliser()
        self.segmenter = PointTraySegmenter()

    @staticmethod
    def _resize_depth_to_rgb(depth_mm: np.ndarray, rgb_bgr: np.ndarray) -> np.ndarray:
        if depth_mm.shape[:2] == rgb_bgr.shape[:2]:
            return depth_mm
        return cv2.resize(depth_mm.astype(np.float32), (rgb_bgr.shape[1], rgb_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)

    def build_views(self, packet: KeyframePacket) -> List[Tuple[str, np.ndarray, Optional[bool]]]:
        views: List[Tuple[str, np.ndarray, Optional[bool]]] = []

        lock = self.registrar.update(packet.left_norm)
        views.append(("board registration", draw_board_lock(packet.left_norm, lock), bool(getattr(lock, "valid", False))))

        if not getattr(lock, "valid", False):
            return views

        try:
            depth_for_rgb = self._resize_depth_to_rgb(packet.depth_raw, packet.left_norm)
            rectified = self.rectifier.rectify(packet.left_norm, depth_for_rgb, lock)
            normalised = self.normaliser.normalize(rectified)
            regions = self.segmenter.segment(normalised.rgb_bgr.shape[:2])

            region_overlay = blend_overlay(normalised.rgb_bgr, regions.overlay_bgr, alpha=0.45)
            piece_mask = build_piece_height_mask(
                normalised.height_map_mm,
                normalised.valid_mask,
                min_height_mm=self.config.min_piece_height_mm,
                max_height_mm=self.config.max_piece_height_mm,
                min_area_px=self.config.min_piece_area_px,
            )

            views.extend(
                [
                    ("rectified RGB", rectified.rgb_bgr, True),
                    ("rectified raw depth BW", depth_to_grayscale(rectified.depth_mm), True),
                    ("normalised RGB", normalised.rgb_bgr, True),
                    ("normalised depth BW", normalised.depth_gray, True),
                    ("height above board", normalised.height_uint8, True),
                    ("point/tray masks", region_overlay, True),
                    ("pre-CNN piece height mask", piece_mask, True),
                ]
            )
        except Exception as exc:
            error_img = packet.left_norm.copy()
            cv2.putText(error_img, f"preprocess error: {exc}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2, cv2.LINE_AA)
            views.append(("preprocess error", error_img, False))

        return views



class DiagnosticRecorder:
    """
    Records the live diagnostic views for troubleshooting.

    When active it can save:
      - dashboard.mp4: the combined viewer exactly as displayed
      - views/*.mp4: one labelled video per stream/view
      - raw_npz/*.npz: optional exact numeric arrays from the keyframe packet
      - metadata.jsonl: one JSON record per saved frame

    Use the `r` key in the live viewer to toggle recording.
    """

    def __init__(self, config: LiveViewConfig) -> None:
        self.config = config
        self.active = False
        self.session_dir: Optional[Path] = None
        self.views_dir: Optional[Path] = None
        self.raw_dir: Optional[Path] = None
        self._writers: Dict[str, cv2.VideoWriter] = {}
        self._metadata_file = None
        self._saved_frame_count = 0
        self._seen_view_names: set[str] = set()

    @property
    def saved_frame_count(self) -> int:
        return self._saved_frame_count

    def start(self) -> Path:
        if self.active and self.session_dir is not None:
            return self.session_dir

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.session_dir = Path(self.config.record_dir) / f"live_view_{timestamp}"
        self.views_dir = self.session_dir / "views"
        self.raw_dir = self.session_dir / "raw_npz"
        self.views_dir.mkdir(parents=True, exist_ok=True)
        if self.config.record_raw_npz:
            self.raw_dir.mkdir(parents=True, exist_ok=True)

        session_info = {
            "started_at": timestamp,
            "mode": self.config.mode,
            "fps": self.config.fps,
            "view_size": list(self.config.view_size),
            "grid_cell_size": list(self.config.grid_cell_size),
            "grid_columns": self.config.grid_columns,
            "record_every": self.config.record_every,
            "record_dashboard": self.config.record_dashboard,
            "record_views": self.config.record_views,
            "record_raw_npz": self.config.record_raw_npz,
            "record_codec": self.config.record_codec,
            "notes": "Videos are 8-bit BGR diagnostic views. raw_npz contains exact arrays when enabled.",
        }
        (self.session_dir / "session_info.json").write_text(json.dumps(session_info, indent=2), encoding="utf-8")
        self._metadata_file = (self.session_dir / "metadata.jsonl").open("a", encoding="utf-8")
        self.active = True
        self._saved_frame_count = 0
        print(f"Recording started: {self.session_dir}")
        return self.session_dir

    def stop(self) -> None:
        if not self.active:
            return
        for writer in self._writers.values():
            writer.release()
        self._writers.clear()
        if self._metadata_file is not None:
            self._metadata_file.close()
            self._metadata_file = None
        print(f"Recording stopped: {self.session_dir} ({self._saved_frame_count} saved frames)")
        self.active = False

    def toggle(self) -> None:
        if self.active:
            self.stop()
        else:
            self.start()

    def _video_fps(self) -> float:
        every = max(1, int(self.config.record_every))
        return max(1.0, float(self.config.fps) / every)

    def _writer_for(self, name: str, frame_bgr: np.ndarray, *, is_dashboard: bool = False) -> cv2.VideoWriter:
        if self.session_dir is None:
            self.start()
        assert self.session_dir is not None
        assert self.views_dir is not None

        key = "dashboard" if is_dashboard else f"view_{sanitise_filename(name)}"
        if key in self._writers:
            return self._writers[key]

        h, w = frame_bgr.shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*self.config.record_codec[:4])
        filename = "dashboard.mp4" if is_dashboard else f"{sanitise_filename(name)}.mp4"
        output_path = (self.session_dir if is_dashboard else self.views_dir) / filename
        writer = cv2.VideoWriter(str(output_path), fourcc, self._video_fps(), (w, h))
        if not writer.isOpened():
            raise RuntimeError(f"Could not open video writer for {output_path}")
        self._writers[key] = writer
        return writer

    def _write_video(self, name: str, frame_bgr: np.ndarray, *, is_dashboard: bool = False) -> None:
        writer = self._writer_for(name, frame_bgr, is_dashboard=is_dashboard)
        writer.write(ensure_bgr(frame_bgr))

    def _write_raw_npz(self, packet: KeyframePacket, right_frame: Optional[np.ndarray]) -> None:
        if not self.config.record_raw_npz:
            return
        if self.raw_dir is None:
            return
        filename = self.raw_dir / f"frame_{packet.frame_index:06d}.npz"
        arrays = {
            "left_raw": packet.left_raw,
            "left_norm": packet.left_norm,
            "left_gray": packet.left_gray,
            "depth_raw_mm": packet.depth_raw,
            "depth_gray": packet.depth_gray,
            "depth_valid_mask": packet.depth_valid_mask,
            "near_mask": packet.near_mask,
        }
        if right_frame is not None:
            arrays["right_raw"] = right_frame
        np.savez_compressed(filename, **arrays)

    def record(
        self,
        *,
        packet: KeyframePacket,
        right_frame: Optional[np.ndarray],
        views: List[Tuple[str, np.ndarray, Optional[bool]]],
        dashboard: np.ndarray,
    ) -> None:
        if not self.active:
            return
        every = max(1, int(self.config.record_every))
        if packet.frame_index % every != 0:
            return

        if self.config.record_dashboard:
            self._write_video("dashboard", dashboard, is_dashboard=True)

        if self.config.record_views:
            for label, image, ok in views:
                cell = render_view_cell(label, image, ok, self.config.grid_cell_size)
                self._write_video(label, cell, is_dashboard=False)
                self._seen_view_names.add(label)

        self._write_raw_npz(packet, right_frame)

        if self._metadata_file is not None:
            row = {
                "timestamp": packet.timestamp,
                "frame_index": packet.frame_index,
                "keyframe": packet.keyframe,
                "stable": packet.stable,
                "occluded": packet.occluded,
                "stable_count": packet.stable_count,
                "motion_score": packet.motion_score,
                "rgb_motion_score": packet.rgb_motion_score,
                "depth_motion_score": packet.depth_motion_score,
                "novelty_score": packet.novelty_score,
                "near_ratio": packet.near_ratio,
                "invalid_ratio": packet.invalid_ratio,
                "view_labels": [label for label, _, _ in views],
            }
            self._metadata_file.write(json.dumps(row) + "\n")
            self._metadata_file.flush()

        self._saved_frame_count += 1

    def save_snapshot(
        self,
        *,
        packet: KeyframePacket,
        right_frame: Optional[np.ndarray],
        views: List[Tuple[str, np.ndarray, Optional[bool]]],
        dashboard: np.ndarray,
    ) -> Path:
        if self.session_dir is None:
            self.start()
        assert self.session_dir is not None
        snapshot_dir = self.session_dir / "snapshots" / f"frame_{packet.frame_index:06d}"
        snapshot_dir.mkdir(parents=True, exist_ok=True)

        cv2.imwrite(str(snapshot_dir / "dashboard.png"), dashboard)
        for label, image, ok in views:
            cell = render_view_cell(label, image, ok, self.config.grid_cell_size)
            cv2.imwrite(str(snapshot_dir / f"{sanitise_filename(label)}.png"), cell)

        np.savez_compressed(
            snapshot_dir / "raw_packet.npz",
            left_raw=packet.left_raw,
            left_norm=packet.left_norm,
            left_gray=packet.left_gray,
            depth_raw_mm=packet.depth_raw,
            depth_gray=packet.depth_gray,
            depth_valid_mask=packet.depth_valid_mask,
            near_mask=packet.near_mask,
            right_raw=right_frame if right_frame is not None else np.array([], dtype=np.uint8),
        )
        print(f"Snapshot saved: {snapshot_dir}")
        return snapshot_dir


class LiveStreamViewer:
    """
    Live diagnostic viewer for OAK-D SR streams and pre-CNN processing stages.

    Recommended command-line use:
        python Acquisition/live_stream_viewer.py --mode all

    Recommended Jupyter use:
        %run Acquisition/live_stream_viewer.py --mode all
    """

    def __init__(self, config: Optional[LiveViewConfig] = None) -> None:
        self.config = config or LiveViewConfig()
        self._last_right_frame: Optional[np.ndarray] = None
        self._preprocessing_enabled = self.config.enable_preprocessing and self.config.mode in {"preprocess", "all"}
        self._preprocessor: Optional[PreprocessingPreview] = None

        if self._preprocessing_enabled:
            try:
                self._preprocessor = PreprocessingPreview(self.config)
            except Exception as exc:
                print(f"Preprocessing preview disabled: {exc}")
                self._preprocessing_enabled = False

        self._baseline = DepthBaselineModel(
            frames_required=self.config.auto_baseline_frames,
            min_valid_ratio=self.config.baseline_min_valid_ratio,
            height_max_mm=self.config.height_max_mm,
            min_piece_height_mm=self.config.min_piece_height_mm,
            max_piece_height_mm=self.config.max_piece_height_mm,
            min_piece_area_px=self.config.min_piece_area_px,
        )

    def _make_gate(self) -> OakSRKeyframeGate:
        # Create exactly one DepthAI stream/pipeline. The earlier implementation
        # built a gate, then replaced gate.streams with a second OakSRStreams
        # instance. That meant two dai.Pipeline objects were constructed for one
        # viewer launch, unlike the user's working camtest.py.
        return OakSRKeyframeGate(
            fps=self.config.fps,
            view_size=self.config.view_size,
            motion_threshold=self.config.motion_threshold,
            hand_near_mm=self.config.hand_near_mm,
            max_near_ratio=self.config.max_near_ratio,
            max_invalid_ratio=self.config.max_invalid_ratio,
            enable_right=True,
        )

    def _build_views(self, packet: KeyframePacket, right_frame: Optional[np.ndarray]) -> List[Tuple[str, np.ndarray, Optional[bool]]]:
        mode = self.config.mode
        views: List[Tuple[str, np.ndarray, Optional[bool]]] = []

        self._baseline.update(packet.depth_raw, allow_auto_capture=self.config.auto_baseline)
        height_mm, height_uint8, baseline_piece_mask, overlap_mask = self._baseline.compute(packet.depth_raw)

        if mode in {"streams", "gate", "preprocess", "all"}:
            views.append(("left RGB raw", packet.left_raw, None))
            if right_frame is not None:
                views.append(("right RGB raw", right_frame, None))
            views.append(("raw depth BW", depth_to_grayscale(packet.depth_raw), None))

        if mode in {"gate", "preprocess", "all"}:
            views.extend(
                [
                    ("keyframe gate status", draw_gate_status(packet.left_norm, packet), packet.keyframe),
                    ("left RGB normalised", packet.left_norm, packet.stable),
                    ("gate depth BW", packet.depth_gray, None),
                    ("valid depth mask", packet.depth_valid_mask * 255, packet.invalid_ratio <= self.config.max_invalid_ratio),
                    ("near/hand occlusion mask", packet.near_mask * 255, not packet.occluded),
                    ("baseline valid-overlap mask", overlap_mask, self._baseline.ready),
                    ("baseline height above board", height_uint8, self._baseline.ready),
                    ("baseline pre-CNN piece mask", baseline_piece_mask, self._baseline.ready),
                ]
            )

        if mode in {"preprocess", "all"} and self._preprocessing_enabled and self._preprocessor is not None:
            views.extend(self._preprocessor.build_views(packet))

        return views

    def run(self) -> None:
        gate = self._make_gate().start()
        recorder = DiagnosticRecorder(self.config)
        if self.config.record:
            recorder.start()

        cv2.namedWindow(self.config.window_name, cv2.WINDOW_NORMAL)

        try:
            while True:
                packet = gate.get_packet(block=True)
                if packet is None:
                    continue

                right_frame = gate.streams.get_right_frame(block=False, use_cached=True)
                if right_frame is not None:
                    self._last_right_frame = right_frame
                else:
                    right_frame = self._last_right_frame

                views = self._build_views(packet, right_frame)
                grid = make_grid(views, cell_size=self.config.grid_cell_size, columns=self.config.grid_columns)

                if recorder.active:
                    rec_text = f"REC frames={recorder.saved_frame_count} dir={recorder.session_dir.name if recorder.session_dir else ''}"
                    cv2.putText(grid, rec_text, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 0, 255), 2, cv2.LINE_AA)

                if self.config.show_help:
                    help_text = "q/ESC quit | r record | s snapshot | b recapture baseline | c clear baseline | p preprocessing | mode=" + self.config.mode
                    cv2.putText(grid, help_text, (10, grid.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

                recorder.record(packet=packet, right_frame=right_frame, views=views, dashboard=grid)

                cv2.imshow(self.config.window_name, grid)
                key = cv2.waitKey(1) & 0xFF

                if key in (ord("q"), 27):
                    break
                if key == ord("p"):
                    self._preprocessing_enabled = not self._preprocessing_enabled
                    print(f"Preprocessing preview: {self._preprocessing_enabled}")
                if key == ord("r"):
                    recorder.toggle()
                if key == ord("s"):
                    recorder.save_snapshot(packet=packet, right_frame=right_frame, views=views, dashboard=grid)
                if key == ord("b"):
                    self._baseline.request_capture()
                    print("Baseline recapture requested. Clear the board and hold it still.")
                if key == ord("c"):
                    self._baseline.clear()
                    print("Baseline cleared.")

        finally:
            recorder.stop()
            gate.stop()
            cv2.destroyAllWindows()


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Live viewer for OAK-D SR raw streams, keyframe gate, and pre-CNN pipeline outputs.")
    parser.add_argument("--mode", choices=["streams", "gate", "preprocess", "all"], default="all", help="Which group of views to show.")
    parser.add_argument("--fps", type=float, default=30.0, help="Camera FPS.")
    parser.add_argument("--view-size", type=parse_size, default=(640, 400), help="Camera output size, e.g. 640x400.")
    parser.add_argument("--cell-size", type=parse_size, default=(420, 260), help="Each grid cell size, e.g. 420x260.")
    parser.add_argument("--columns", type=int, default=3, help="Number of columns in the display grid.")
    parser.add_argument("--no-preprocess", action="store_true", help="Disable preprocessing views even in preprocess/all mode.")
    parser.add_argument("--motion-threshold", type=float, default=2.5, help="Keyframe-gate motion threshold.")
    parser.add_argument("--hand-near-mm", type=int, default=450, help="Depth threshold for near-field hand/occlusion detection.")
    parser.add_argument("--max-near-ratio", type=float, default=0.03, help="Max ROI fraction allowed to be near/occluding.")
    parser.add_argument("--max-invalid-ratio", type=float, default=0.80, help="Max invalid-depth fraction allowed in gate ROI.")
    parser.add_argument("--min-piece-height-mm", type=float, default=3.0, help="Lower threshold for the pre-CNN piece height mask.")
    parser.add_argument("--max-piece-height-mm", type=float, default=40.0, help="Upper threshold for the pre-CNN piece height mask.")
    parser.add_argument("--min-piece-area-px", type=int, default=80, help="Minimum connected-component area for the pre-CNN piece mask.")
    parser.add_argument("--height-max-mm", type=float, default=35.0, help="Display scaling max for baseline height-above-board view.")
    parser.add_argument("--no-auto-baseline", action="store_true", help="Do not automatically capture an empty-board depth baseline from startup frames.")
    parser.add_argument("--auto-baseline-frames", type=int, default=30, help="Number of valid empty-board frames used for baseline capture.")
    parser.add_argument("--baseline-min-valid-ratio", type=float, default=0.20, help="Minimum valid depth fraction required for a frame to be used in baseline capture.")
    parser.add_argument("--record", action="store_true", help="Start recording diagnostic videos immediately.")
    parser.add_argument("--record-dir", type=Path, default=Path("troubleshooting_recordings"), help="Directory for diagnostic recording sessions.")
    parser.add_argument("--record-every", type=int, default=1, help="Save every Nth frame while recording.")
    parser.add_argument("--record-raw-npz", action="store_true", help="Also save exact raw arrays per recorded frame as compressed NPZ files.")
    parser.add_argument("--no-record-dashboard", action="store_true", help="Do not save the combined dashboard video.")
    parser.add_argument("--no-record-views", action="store_true", help="Do not save one video per individual view/stream.")
    parser.add_argument("--record-codec", type=str, default="mp4v", help="OpenCV video codec fourcc, e.g. mp4v or XVID.")
    return parser


def main(argv: Optional[List[str]] = None) -> None:
    args = build_arg_parser().parse_args(argv)
    config = LiveViewConfig(
        mode=args.mode,
        fps=args.fps,
        view_size=args.view_size,
        grid_cell_size=args.cell_size,
        grid_columns=args.columns,
        motion_threshold=args.motion_threshold,
        hand_near_mm=args.hand_near_mm,
        max_near_ratio=args.max_near_ratio,
        max_invalid_ratio=args.max_invalid_ratio,
        enable_preprocessing=not args.no_preprocess,
        min_piece_height_mm=args.min_piece_height_mm,
        max_piece_height_mm=args.max_piece_height_mm,
        min_piece_area_px=args.min_piece_area_px,
        height_max_mm=args.height_max_mm,
        auto_baseline=not args.no_auto_baseline,
        auto_baseline_frames=args.auto_baseline_frames,
        baseline_min_valid_ratio=args.baseline_min_valid_ratio,
        record=args.record,
        record_dir=args.record_dir,
        record_every=max(1, args.record_every),
        record_dashboard=not args.no_record_dashboard,
        record_views=not args.no_record_views,
        record_raw_npz=args.record_raw_npz,
        record_codec=args.record_codec,
    )
    LiveStreamViewer(config).run()


if __name__ == "__main__":
    main()


## Depth stack live viewer

Diagnostic viewer for depth support, RGB-restored chip masks, stack classification and mask tuning.


In [ ]:
%%writefile Acquisition/depth_stack_live_viewer.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple
import argparse
import importlib.util
import json
import sys
import time

import cv2
import numpy as np

_MODULE_DIR = Path(__file__).resolve().parent
_PROJECT_ROOT = _MODULE_DIR.parent if _MODULE_DIR.name in {"Acquisition", "Interpretation", "Geometric + Visual Preprocessing"} else _MODULE_DIR
for _rel in ("", "Acquisition", "Interpretation", "Geometric + Visual Preprocessing"):
    _p = _PROJECT_ROOT / _rel if _rel else _PROJECT_ROOT
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from streams_module import OakSRStreams

# Load the classifier from the project Interpretation folder explicitly. This avoids
# Jupyter accidentally reusing an older cached depth_stack_classifier module.
_classifier_path = _PROJECT_ROOT / "Interpretation" / "depth_stack_classifier.py"
if not _classifier_path.exists():
    _classifier_path = _MODULE_DIR / "depth_stack_classifier.py"
if not _classifier_path.exists():
    _classifier_path = Path("depth_stack_classifier.py").resolve()

_spec = importlib.util.spec_from_file_location("depth_stack_classifier_runtime", _classifier_path)
if _spec is None or _spec.loader is None:
    raise ImportError(f"Could not load depth_stack_classifier from {_classifier_path}")
_dsc = importlib.util.module_from_spec(_spec)
# Python 3.12 dataclasses expect the module being executed to already be
# registered in sys.modules. Without this, @dataclass can fail with:
# AttributeError: 'NoneType' object has no attribute '__dict__'.
sys.modules[_spec.name] = _dsc
_spec.loader.exec_module(_dsc)
print(f"Loaded depth_stack_classifier from: {_classifier_path}")

EmptyBoardBaseline = _dsc.EmptyBoardBaseline
TemporalMaskFilter = _dsc.TemporalMaskFilter
StackCountSmoother = _dsc.StackCountSmoother
classify_rgb_candidates_by_depth = _dsc.classify_rgb_candidates_by_depth
depth_to_gray_fixed = _dsc.depth_to_gray_fixed
height_to_gray = _dsc.height_to_gray

Size = Tuple[int, int]
RoiFrac = Optional[Tuple[float, float, float, float]]


@dataclass
class Config:
    fps: float = 15.0
    view_size: Size = (640, 400)
    cell_size: Size = (420, 260)
    columns: int = 3

    baseline_frames: int = 60
    chip_thickness_mm: float = 10.0
    min_piece_height_mm: float = 3.0
    max_piece_height_mm: float = 45.0
    height_stat: str = "top35"

    depth_min_mm: float = 360.0
    depth_max_mm: float = 435.0
    height_max_visual_mm: float = 30.0

    min_chip_radius_px: float = 7.0
    max_chip_radius_px: float = 30.0
    expected_chip_radius_px: Optional[float] = None

    split_touching: bool = True
    use_hough: bool = False
    draw_rejected_candidates: bool = False
    min_candidate_support_ratio: float = 0.025
    min_support_pixels: int = 6
    min_depth_pixels: int = 8
    min_depth_ratio: float = 0.025
    roi_frac: RoiFrac = (0.20, 0.00, 0.82, 1.00)
    # Exclude the central horizontal dice strip from stack/chip detection.
    # Format x1,y1,x2,y2 as fractions of the viewer image. Use "none" to disable.
    middle_exclusion_frac: RoiFrac = (0.00, 0.40, 1.00, 0.60)

    temporal_enabled: bool = True
    temporal_window: int = 3
    temporal_require: int = 2
    temporal_open_px: int = 3
    temporal_close_px: int = 5
    temporal_min_area_px: int = 30
    noise_margin_mm: float = 1.5

    # Stack-count hysteresis. Chips only go up to 2-high in this project.
    # Promote to 2 at this height, but only demote back to 1 after repeated
    # lower-height evidence. This prevents green/yellow flicker.
    stack_promote_height_mm: float = 16.5
    stack_demote_height_mm: float = 14.5
    stack_temporal_window: int = 5
    stack_promote_votes: int = 2
    stack_demote_votes: int = 4
    stack_hold_misses: int = 2

    record: bool = False
    record_every: int = 3
    record_raw_npz: bool = False
    out_dir: Path = Path("troubleshooting_recordings/depth_stack_viewer")


def parse_size(text: str) -> Size:
    try:
        w, h = text.lower().split("x", 1)
        return int(w), int(h)
    except Exception as exc:
        raise argparse.ArgumentTypeError("size must look like 640x400") from exc


def parse_roi_frac(text: str) -> RoiFrac:
    if text.lower().strip() in {"none", "off", "false", "0"}:
        return None
    try:
        vals = tuple(float(v.strip()) for v in text.split(","))
    except Exception as exc:
        raise argparse.ArgumentTypeError("ROI must be x1,y1,x2,y2 or 'none'") from exc
    if len(vals) != 4:
        raise argparse.ArgumentTypeError("ROI must contain four comma-separated floats: x1,y1,x2,y2")
    x1, y1, x2, y2 = vals
    if not (0.0 <= x1 < x2 <= 1.0 and 0.0 <= y1 < y2 <= 1.0):
        raise argparse.ArgumentTypeError("ROI fractions must satisfy 0<=x1<x2<=1 and 0<=y1<y2<=1")
    return vals  # type: ignore[return-value]


def ensure_bgr(img: np.ndarray) -> np.ndarray:
    if img.ndim == 2:
        return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.ndim == 3:
        return img.copy()
    raise ValueError(str(img.shape))


def letterbox(img: np.ndarray, size: Size) -> np.ndarray:
    img = ensure_bgr(img)
    tw, th = size
    h, w = img.shape[:2]
    scale = min(tw / w, th / h)
    nw, nh = max(1, int(w * scale)), max(1, int(h * scale))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((th, tw, 3), dtype=np.uint8)
    x, y = (tw - nw) // 2, (th - nh) // 2
    canvas[y:y + nh, x:x + nw] = resized
    return canvas


def label(img: np.ndarray, text: str) -> np.ndarray:
    out = img.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 28), (0, 0, 0), -1)
    cv2.putText(out, text, (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (235, 235, 235), 1, cv2.LINE_AA)
    return out


def grid(items: List[Tuple[str, np.ndarray]], cell_size: Size, columns: int) -> np.ndarray:
    cells = [label(letterbox(img, cell_size), name) for name, img in items]
    if not cells:
        return np.zeros((cell_size[1], cell_size[0], 3), dtype=np.uint8)
    blank = np.zeros_like(cells[0])
    rows = int(np.ceil(len(cells) / max(columns, 1)))
    cells += [blank] * (rows * columns - len(cells))
    return np.vstack([np.hstack(cells[r * columns:(r + 1) * columns]) for r in range(rows)])


class VideoRecorder:
    def __init__(self, out_dir: Path, fps: float) -> None:
        self.out_dir = out_dir / time.strftime("depth_stack_%Y%m%d_%H%M%S")
        self.views_dir = self.out_dir / "views"
        self.raw_dir = self.out_dir / "raw_npz"
        self.views_dir.mkdir(parents=True, exist_ok=True)
        self.raw_dir.mkdir(parents=True, exist_ok=True)
        self.fps = fps
        self.writers = {}
        self.metadata = open(self.out_dir / "metadata.jsonl", "w", encoding="utf-8")

    def _writer(self, name: str, shape) -> cv2.VideoWriter:
        if name in self.writers:
            return self.writers[name]
        h, w = shape[:2]
        path = self.views_dir / f"{name}.mp4"
        wr = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), self.fps, (w, h))
        self.writers[name] = wr
        return wr

    def write(self, views: List[Tuple[str, np.ndarray]], meta: dict, raw: Optional[dict] = None) -> None:
        for name, img in views:
            safe = name.lower().replace(" ", "_").replace("/", "-").replace("+", "plus")
            bgr = ensure_bgr(img)
            self._writer(safe, bgr.shape).write(bgr)
        self.metadata.write(json.dumps(meta) + "\n")
        self.metadata.flush()
        if raw is not None:
            np.savez_compressed(self.raw_dir / f"frame_{meta['frame']:06d}.npz", **raw)

    def close(self) -> None:
        for wr in self.writers.values():
            wr.release()
        self.metadata.close()


def build_arg_parser() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(
        description="Live stack-height diagnostic viewer: stable depth support + RGB-restored chip circles + stack classification."
    )
    ap.add_argument("--fps", type=float, default=15.0)
    ap.add_argument("--view-size", type=parse_size, default=(640, 400))
    ap.add_argument("--cell-size", type=parse_size, default=(420, 260))
    ap.add_argument("--columns", type=int, default=3)

    ap.add_argument("--baseline-frames", type=int, default=60)
    ap.add_argument("--chip-thickness-mm", type=float, default=10.0)
    ap.add_argument("--min-piece-height-mm", type=float, default=3.0)
    ap.add_argument("--max-piece-height-mm", type=float, default=45.0)
    ap.add_argument("--height-stat", choices=["median", "p75", "p90", "max", "top35", "top25"], default="top35")

    ap.add_argument("--depth-min-mm", type=float, default=360.0)
    ap.add_argument("--depth-max-mm", type=float, default=435.0)
    ap.add_argument("--height-max-visual-mm", type=float, default=30.0)

    ap.add_argument("--min-chip-radius-px", type=float, default=7.0)
    ap.add_argument("--max-chip-radius-px", type=float, default=30.0)
    ap.add_argument("--expected-chip-radius-px", type=float, default=None)

    ap.add_argument("--no-split-touching", dest="split_touching", action="store_false", default=True)
    ap.add_argument("--hough", dest="use_hough", action="store_true", default=False)
    ap.add_argument("--draw-rejected-candidates", action="store_true", default=False)
    ap.add_argument("--min-candidate-support-ratio", type=float, default=0.025)
    ap.add_argument("--min-support-pixels", type=int, default=6)
    ap.add_argument("--min-depth-pixels", type=int, default=8)
    ap.add_argument("--min-depth-ratio", type=float, default=0.025)
    ap.add_argument("--roi-frac", type=parse_roi_frac, default=(0.10, 0.00, 1.00, 1.00))
    ap.add_argument("--middle-exclusion-frac", type=parse_roi_frac, default=(0.00, 0.38, 1.00, 0.60), help="Rectangular region to exclude from chip/stack detection, e.g. dice strip. Use 'none' to disable.")

    ap.add_argument("--disable-temporal", dest="temporal_enabled", action="store_false", default=True)
    ap.add_argument("--temporal-window", type=int, default=3)
    ap.add_argument("--temporal-require", type=int, default=2)
    ap.add_argument("--temporal-open-px", type=int, default=3)
    ap.add_argument("--temporal-close-px", type=int, default=5)
    ap.add_argument("--temporal-min-area-px", type=int, default=30)
    ap.add_argument("--noise-margin-mm", type=float, default=1.5)
    ap.add_argument("--stack-promote-height-mm", type=float, default=16.5, help="Height at/above which a candidate can promote to a 2-high stack.")
    ap.add_argument("--stack-demote-height-mm", type=float, default=14.5, help="Height below which repeated evidence can demote a 2-high stack back to 1-high.")
    ap.add_argument("--stack-temporal-window", type=int, default=5)
    ap.add_argument("--stack-promote-votes", type=int, default=2)
    ap.add_argument("--stack-demote-votes", type=int, default=4)
    ap.add_argument("--stack-hold-misses", type=int, default=2)

    ap.add_argument("--record", action="store_true")
    ap.add_argument("--record-every", type=int, default=3)
    ap.add_argument("--record-raw-npz", action="store_true")
    ap.add_argument("--out-dir", type=Path, default=Path("troubleshooting_recordings/depth_stack_viewer"))
    return ap


def main(argv: Optional[List[str]] = None) -> None:
    args = build_arg_parser().parse_args(argv)
    cfg = Config(**vars(args))

    baseline = EmptyBoardBaseline(samples_required=cfg.baseline_frames)
    temporal_filter = None
    if cfg.temporal_enabled:
        temporal_filter = TemporalMaskFilter(
            window=cfg.temporal_window,
            required=cfg.temporal_require,
            open_px=cfg.temporal_open_px,
            close_px=cfg.temporal_close_px,
            min_area_px=cfg.temporal_min_area_px,
        )

    stack_smoother = StackCountSmoother(
        window=cfg.stack_temporal_window,
        promote_votes=cfg.stack_promote_votes,
        demote_votes=cfg.stack_demote_votes,
        hold_misses=cfg.stack_hold_misses,
        promote_height_mm=cfg.stack_promote_height_mm,
        demote_height_mm=cfg.stack_demote_height_mm,
        match_distance_px=max(24.0, (cfg.expected_chip_radius_px or cfg.max_chip_radius_px) * 1.5),
    )

    cam = OakSRStreams(
        enable_left=True,
        enable_right=True,
        enable_depth=True,
        fps=cfg.fps,
        view_size=cfg.view_size,
        stereo_size=cfg.view_size,
    ).start()
    recorder = VideoRecorder(cfg.out_dir, max(1.0, cfg.fps / max(cfg.record_every, 1))) if cfg.record else None
    cv2.namedWindow("Depth stack diagnostic viewer", cv2.WINDOW_NORMAL)

    frame_i = 0
    try:
        while True:
            left = cam.get_left_frame(block=True, use_cached=True)
            right = cam.get_right_frame(block=False, use_cached=True)
            depth = cam.get_depth_frame(block=True, use_cached=True)
            if left is None or depth is None:
                continue

            raw_depth_bw = depth_to_gray_fixed(depth, cfg.depth_min_mm, cfg.depth_max_mm)
            views: List[Tuple[str, np.ndarray]] = [("left RGB", left), ("raw depth BW", raw_depth_bw)]
            if right is not None:
                views.insert(1, ("right RGB", right))

            if not baseline.ready:
                baseline.add_sample(depth)
                if temporal_filter is not None:
                    temporal_filter.reset()
                stack_smoother.reset()
                status = np.zeros((*left.shape[:2], 3), dtype=np.uint8)
                cv2.putText(status, f"Capturing empty-board baseline: {len(baseline.samples)}/{cfg.baseline_frames}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
                cv2.putText(status, "Keep board empty and still. Press b to restart baseline.", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (255, 255, 255), 2)
                views.append(("baseline status", status))
            else:
                result = classify_rgb_candidates_by_depth(
                    left,
                    depth,
                    baseline,
                    chip_thickness_mm=cfg.chip_thickness_mm,
                    min_piece_height_mm=cfg.min_piece_height_mm,
                    max_piece_height_mm=cfg.max_piece_height_mm,
                    height_stat=cfg.height_stat,
                    min_depth_pixels=cfg.min_depth_pixels,
                    min_depth_ratio=cfg.min_depth_ratio,
                    height_max_visual_mm=cfg.height_max_visual_mm,
                    min_radius_px=cfg.min_chip_radius_px,
                    max_radius_px=cfg.max_chip_radius_px,
                    expected_radius_px=cfg.expected_chip_radius_px,
                    split_touching=cfg.split_touching,
                    use_hough=cfg.use_hough,
                    min_candidate_support_ratio=cfg.min_candidate_support_ratio,
                    min_support_pixels=cfg.min_support_pixels,
                    draw_rejected_candidates=cfg.draw_rejected_candidates,
                    roi_frac=cfg.roi_frac,
                    middle_exclusion_frac=cfg.middle_exclusion_frac,
                    temporal_filter=temporal_filter,
                    stack_smoother=stack_smoother,
                    stack_promote_height_mm=cfg.stack_promote_height_mm,
                    stack_demote_height_mm=cfg.stack_demote_height_mm,
                    noise_margin_mm=cfg.noise_margin_mm,
                )
                height_bw = height_to_gray(result.height_mm, cfg.height_max_visual_mm)
                valid = result.valid_overlap_mask * 255
                stable_support = result.stable_depth_support_mask
                restored = result.rgb_restored_circle_mask
                stack_overlay = result.overlay_bgr.copy()
                text = (
                    f"chips={int(result.diagnostics['candidate_count'])} "
                    f"classified={int(result.diagnostics['classified_count'])} "
                    f"raw={result.diagnostics['raw_support_ratio']:.4f} "
                    f"stable={result.diagnostics['stable_support_ratio']:.4f} "
                    f"T={int(result.diagnostics['temporal_history'])}/{cfg.temporal_window if cfg.temporal_enabled else 0} "
                    f"2@{cfg.stack_promote_height_mm:.1f}/1@{cfg.stack_demote_height_mm:.1f}"
                )
                cv2.putText(stack_overlay, text, (10, stack_overlay.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 2)
                views.extend([
                    ("height above board BW", height_bw),
                    ("valid overlap mask", valid),
                    ("stable depth support mask", stable_support),
                    ("RGB candidate mask", result.rgb_candidate_mask),
                    ("RGB-restored circle mask", restored),
                    ("stack class mask", result.stack_class_bgr),
                    ("RGB+depth stack overlay", stack_overlay),
                ])

                if recorder and (frame_i % max(cfg.record_every, 1) == 0):
                    raw = None
                    if cfg.record_raw_npz:
                        raw = {
                            "left_rgb": left,
                            "right_rgb": right if right is not None else np.zeros_like(left),
                            "depth_raw_mm": depth,
                            "height_mm": result.height_mm,
                            "valid_overlap_mask": result.valid_overlap_mask,
                            "stable_depth_support_mask": result.stable_depth_support_mask,
                            "rgb_candidate_mask": result.rgb_candidate_mask,
                            "rgb_restored_circle_mask": result.rgb_restored_circle_mask,
                            "stack_class_bgr": result.stack_class_bgr,
                        }
                    recorder.write(views, {"frame": frame_i, "time": time.time(), **result.diagnostics}, raw=raw)

            dash = grid(views, cfg.cell_size, cfg.columns)
            cv2.putText(dash, "q/ESC quit | b recapture empty-board baseline", (10, dash.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
            cv2.imshow("Depth stack diagnostic viewer", dash)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord("q"), 27):
                break
            if key == ord("b"):
                baseline.clear()
                if temporal_filter is not None:
                    temporal_filter.reset()
                stack_smoother.reset()
                print("Baseline cleared. Keep board empty and still.")
            frame_i += 1
    finally:
        cam.stop()
        if recorder:
            recorder.close()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


# 3. Geometric + Visual Preprocessing

Board lock, rectification, RGB/depth normalisation, grid/ROI masks and ROI validation.


## Board registration

Finds or reuses the board corners using contour detection, previous lock recovery and manual-corner fallback.


In [ ]:
%%writefile "Geometric + Visual Preprocessing/board_registration.py"
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple
import json

import cv2
import numpy as np

from backgammon_types import BoardLock


@dataclass
class BoardRegistrationConfig:
    method: str = "auto"
    use_aruco: bool = True
    smoothing: float = 0.75
    min_area_ratio: float = 0.12
    min_confidence: float = 0.15
    canny_low: int = 50
    canny_high: int = 150
    debug: bool = False

    # If automatic registration fails, the fixed-camera prototype can use a
    # saved corner calibration from roi_preview.py. The JSON format is:
    # {"corners_xy": [[tl_x, tl_y], [tr_x, tr_y], [br_x, br_y], [bl_x, bl_y]]}
    use_manual_file_fallback: bool = True
    manual_corners_path: str = "manual_board_corners.json"


def order_corners(corners_xy: np.ndarray) -> np.ndarray:
    corners = np.asarray(corners_xy, dtype=np.float32).reshape(4, 2)
    s = corners.sum(axis=1)
    d = np.diff(corners, axis=1).ravel()

    ordered = np.zeros((4, 2), dtype=np.float32)
    ordered[0] = corners[np.argmin(s)]  # top-left
    ordered[2] = corners[np.argmax(s)]  # bottom-right
    ordered[1] = corners[np.argmin(d)]  # top-right
    ordered[3] = corners[np.argmax(d)]  # bottom-left
    return ordered


def polygon_area(corners_xy: np.ndarray) -> float:
    corners = np.asarray(corners_xy, dtype=np.float32).reshape(-1, 2)
    x = corners[:, 0]
    y = corners[:, 1]
    return float(0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def contour_to_quad(contour: np.ndarray) -> Optional[np.ndarray]:
    peri = cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, 0.02 * peri, True)
    if len(approx) != 4 or not cv2.isContourConvex(approx):
        return None
    return approx.reshape(4, 2).astype(np.float32)


def detect_board_corners_contour(frame_bgr: np.ndarray, cfg: BoardRegistrationConfig) -> Tuple[Optional[np.ndarray], float, dict]:
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(blur, cfg.canny_low, cfg.canny_high)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=2)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h, w = gray.shape[:2]
    frame_area = float(h * w)
    best_quad = None
    best_area = 0.0

    for contour in contours:
        quad = contour_to_quad(contour)
        if quad is None:
            continue

        area = polygon_area(quad)
        if area < cfg.min_area_ratio * frame_area:
            continue

        if area > best_area:
            best_area = area
            best_quad = quad

    confidence = 0.0 if best_quad is None else min(1.0, best_area / (0.55 * frame_area + 1e-6))
    debug = {"edge_nonzero": int(np.count_nonzero(edges)), "best_area": best_area}
    return best_quad, confidence, debug


def detect_board_corners_aruco(frame_bgr: np.ndarray) -> Tuple[Optional[np.ndarray], float, dict]:
    if not hasattr(cv2, "aruco"):
        return None, 0.0, {"aruco_available": False}

    dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    detector = cv2.aruco.ArucoDetector(dictionary, cv2.aruco.DetectorParameters())
    corners, ids, _ = detector.detectMarkers(frame_bgr)

    if ids is None or len(corners) < 2:
        return None, 0.0, {"aruco_markers": 0}

    points = np.concatenate([c.reshape(-1, 2) for c in corners], axis=0).astype(np.float32)
    hull = cv2.convexHull(points).reshape(-1, 2)
    rect = cv2.minAreaRect(hull)
    quad = cv2.boxPoints(rect).astype(np.float32)

    confidence = min(1.0, 0.2 + 0.15 * len(corners))
    return quad, confidence, {"aruco_markers": int(len(corners))}


class BoardRegistrar:
    """
    Initial board-registration / recovery stage.

    Classical board-localisation stage used to get a stable board lock before
    collecting aligned RGB-D data. The same interface could later be backed
    by a dedicated corner-regression model if needed.
    """

    def __init__(self, config: Optional[BoardRegistrationConfig] = None) -> None:
        self.config = config or BoardRegistrationConfig()
        self._previous_corners: Optional[np.ndarray] = None
        self._manual_corners: Optional[np.ndarray] = None
        self._manual_file_corners: Optional[np.ndarray] = self._load_manual_corners_file()

    def _load_manual_corners_file(self) -> Optional[np.ndarray]:
        if not self.config.use_manual_file_fallback:
            return None

        path = Path(self.config.manual_corners_path)
        if not path.is_absolute():
            # Run scripts from the project root where the notebook lives.
            path = Path.cwd() / path

        if not path.exists():
            return None

        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            corners = np.asarray(payload.get("corners_xy"), dtype=np.float32)
            if corners.shape != (4, 2):
                return None
            return order_corners(corners)
        except Exception as exc:  # keep registration robust during live runs
            if self.config.debug:
                print(f"Could not load manual corners from {path}: {exc}")
            return None

    def set_manual_corners(self, corners_xy: np.ndarray) -> None:
        self._manual_corners = order_corners(corners_xy)

    def clear_manual_corners(self) -> None:
        self._manual_corners = None

    def _smooth(self, corners_xy: np.ndarray) -> np.ndarray:
        if self._previous_corners is None:
            return corners_xy
        alpha = float(np.clip(self.config.smoothing, 0.0, 1.0))
        return alpha * self._previous_corners + (1.0 - alpha) * corners_xy

    def _build_lock(
        self,
        frame_bgr: np.ndarray,
        corners_xy: Optional[np.ndarray],
        confidence: float,
        method: str,
        debug: dict,
    ) -> BoardLock:
        h, w = frame_bgr.shape[:2]
        if corners_xy is None:
            return BoardLock(
                valid=False,
                corners_xy=np.zeros((4, 2), dtype=np.float32),
                confidence=0.0,
                method=method,
                frame_size_hw=(h, w),
                debug=debug,
            )

        ordered = order_corners(corners_xy)
        ordered = self._smooth(ordered).astype(np.float32)
        self._previous_corners = ordered.copy()

        valid = confidence >= self.config.min_confidence
        return BoardLock(
            valid=valid,
            corners_xy=ordered,
            confidence=float(confidence),
            method=method,
            frame_size_hw=(h, w),
            debug=debug,
        )

    def update(self, frame_bgr: np.ndarray) -> BoardLock:
        if self._manual_corners is not None:
            return self._build_lock(
                frame_bgr,
                self._manual_corners,
                confidence=1.0,
                method="manual",
                debug={"source": "manual"},
            )

        if self.config.use_aruco:
            corners_xy, confidence, debug = detect_board_corners_aruco(frame_bgr)
            if corners_xy is not None:
                return self._build_lock(frame_bgr, corners_xy, confidence, "aruco", debug)

        corners_xy, confidence, debug = detect_board_corners_contour(frame_bgr, self.config)
        if corners_xy is not None:
            return self._build_lock(frame_bgr, corners_xy, confidence, "contour", debug)

        if self._previous_corners is not None:
            return self._build_lock(
                frame_bgr,
                self._previous_corners,
                confidence=0.05,
                method="previous",
                debug={"source": "previous"},
            )

        return self._build_lock(frame_bgr, None, 0.0, "none", {"source": "none"})


## Perspective rectification

Warps RGB/depth into the canonical board view and computes height above the board plane.


In [ ]:
%%writefile "Geometric + Visual Preprocessing/perspective_rectification.py"
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Tuple

import cv2
import numpy as np

from backgammon_types import BoardLock, RectifiedBoard


@dataclass
class RectificationConfig:
    target_size_wh: Tuple[int, int] = (1200, 900)
    plane_fit_upper_quantile: float = 0.70
    fit_stride: int = 8
    min_plane_points: int = 500


def destination_corners(target_size_wh: Tuple[int, int]) -> np.ndarray:
    width, height = target_size_wh
    return np.array(
        [
            [0, 0],
            [width - 1, 0],
            [width - 1, height - 1],
            [0, height - 1],
        ],
        dtype=np.float32,
    )


def fit_depth_plane(depth_mm: np.ndarray, valid_mask: np.ndarray, cfg: RectificationConfig) -> Tuple[Tuple[float, float, float], np.ndarray]:
    ys, xs = np.indices(depth_mm.shape)
    valid = (valid_mask > 0) & np.isfinite(depth_mm) & (depth_mm > 0)

    if np.count_nonzero(valid) < cfg.min_plane_points:
        plane = (0.0, 0.0, float(np.nanmedian(depth_mm[valid])) if np.count_nonzero(valid) else 0.0)
        z_plane = np.full(depth_mm.shape, plane[2], dtype=np.float32)
        return plane, z_plane

    depth_valid = depth_mm[valid]
    threshold = float(np.quantile(depth_valid, cfg.plane_fit_upper_quantile))
    board_like = valid & (depth_mm >= threshold)

    xs_s = xs[board_like][:: cfg.fit_stride].astype(np.float32)
    ys_s = ys[board_like][:: cfg.fit_stride].astype(np.float32)
    zs_s = depth_mm[board_like][:: cfg.fit_stride].astype(np.float32)

    if len(xs_s) < 3:
        plane = (0.0, 0.0, float(np.nanmedian(depth_valid)))
        z_plane = np.full(depth_mm.shape, plane[2], dtype=np.float32)
        return plane, z_plane

    A = np.column_stack([xs_s, ys_s, np.ones_like(xs_s)])
    coeffs, _, _, _ = np.linalg.lstsq(A, zs_s, rcond=None)

    a, b, c = [float(v) for v in coeffs]
    z_plane = a * xs.astype(np.float32) + b * ys.astype(np.float32) + c
    return (a, b, c), z_plane


class PerspectiveRectifier:
    """
    Homography-based rectification plus a simple depth-plane fit.

    The depth-plane model makes it easy to convert raw depth into an
    approximate height-above-board map for downstream segmentation.
    """

    def __init__(self, config: Optional[RectificationConfig] = None) -> None:
        self.config = config or RectificationConfig()

    def rectify(self, rgb_bgr: np.ndarray, depth_mm: np.ndarray, board_lock: BoardLock) -> RectifiedBoard:
        if not board_lock.valid:
            raise ValueError("Board lock is not valid.")

        dst = destination_corners(self.config.target_size_wh)
        H = cv2.getPerspectiveTransform(board_lock.corners_xy.astype(np.float32), dst)
        Hinv = cv2.getPerspectiveTransform(dst, board_lock.corners_xy.astype(np.float32))

        width, height = self.config.target_size_wh
        rgb_rectified = cv2.warpPerspective(
            rgb_bgr,
            H,
            (width, height),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REPLICATE,
        )

        depth_rectified = cv2.warpPerspective(
            depth_mm.astype(np.float32),
            H,
            (width, height),
            flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0,
        )

        valid_mask = (depth_rectified > 0).astype(np.uint8)
        board_mask = np.ones((height, width), dtype=np.uint8)

        plane_coeffs, plane_depth = fit_depth_plane(depth_rectified, valid_mask, self.config)
        height_map_mm = np.maximum(plane_depth - depth_rectified, 0.0).astype(np.float32)
        height_map_mm[valid_mask == 0] = 0.0

        return RectifiedBoard(
            rgb_bgr=rgb_rectified,
            depth_mm=depth_rectified.astype(np.float32),
            valid_mask=valid_mask,
            height_map_mm=height_map_mm,
            homography=H.astype(np.float32),
            inverse_homography=Hinv.astype(np.float32),
            board_mask=board_mask,
            plane_coeffs=plane_coeffs,
        )


## Lighting / colour / depth normalisation

Normalises RGB, depth and height-map views.


In [ ]:
%%writefile "Geometric + Visual Preprocessing/image_normalisation.py"
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Tuple

import cv2
import numpy as np

from backgammon_types import NormalisedBoard, RectifiedBoard
from streams_module import depth_to_grayscale


@dataclass
class NormalisationConfig:
    wb_strength: float = 1.0
    clahe_clip_limit: float = 2.0
    clahe_tile_grid: Tuple[int, int] = (8, 8)
    rgb_blur_ksize: int = 3
    depth_median_ksize: int = 5
    depth_bilateral_d: int = 5
    depth_bilateral_sigma_color: float = 25.0
    depth_bilateral_sigma_space: float = 25.0
    max_height_mm: float = 40.0


def gray_world_white_balance(img_bgr: np.ndarray, strength: float = 1.0) -> np.ndarray:
    img = img_bgr.astype(np.float32)
    means = img.reshape(-1, 3).mean(axis=0)
    gray_mean = float(means.mean()) + 1e-6
    scale = gray_mean / (means + 1e-6)
    balanced = img * scale.reshape(1, 1, 3)
    mixed = img * (1.0 - strength) + balanced * strength
    return np.clip(mixed, 0, 255).astype(np.uint8)


def normalize_rgb(rgb_bgr: np.ndarray, cfg: NormalisationConfig) -> Tuple[np.ndarray, np.ndarray]:
    wb = gray_world_white_balance(rgb_bgr, strength=cfg.wb_strength)

    lab = cv2.cvtColor(wb, cv2.COLOR_BGR2LAB)
    l_chan, a_chan, b_chan = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=cfg.clahe_clip_limit,
        tileGridSize=cfg.clahe_tile_grid,
    )
    l_eq = clahe.apply(l_chan)

    merged = cv2.merge((l_eq, a_chan, b_chan))
    rgb_norm = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    rgb_norm = cv2.GaussianBlur(rgb_norm, (cfg.rgb_blur_ksize | 1, cfg.rgb_blur_ksize | 1), 0)

    gray = cv2.cvtColor(rgb_norm, cv2.COLOR_BGR2GRAY)
    return rgb_norm, gray


def normalize_depth(depth_mm: np.ndarray, valid_mask: np.ndarray, cfg: NormalisationConfig) -> Tuple[np.ndarray, np.ndarray]:
    depth = depth_mm.astype(np.float32).copy()
    depth[valid_mask == 0] = 0.0

    if cfg.depth_median_ksize >= 3:
        depth = cv2.medianBlur(depth, cfg.depth_median_ksize | 1)

    depth = cv2.bilateralFilter(
        depth,
        d=cfg.depth_bilateral_d,
        sigmaColor=cfg.depth_bilateral_sigma_color,
        sigmaSpace=cfg.depth_bilateral_sigma_space,
    )
    depth[valid_mask == 0] = 0.0

    depth_gray = depth_to_grayscale(depth, near_percentile=3.0, far_percentile=95.0, invert=False)
    return depth.astype(np.float32), depth_gray


def normalize_height_map(height_map_mm: np.ndarray, valid_mask: np.ndarray, max_height_mm: float = 40.0) -> np.ndarray:
    height = np.clip(height_map_mm.astype(np.float32), 0.0, max_height_mm)
    norm = (height * (255.0 / max_height_mm)).astype(np.uint8)
    norm[valid_mask == 0] = 0
    return norm


class BoardNormaliser:
    def __init__(self, config: Optional[NormalisationConfig] = None) -> None:
        self.config = config or NormalisationConfig()

    def normalize(self, rectified: RectifiedBoard) -> NormalisedBoard:
        rgb_norm, gray = normalize_rgb(rectified.rgb_bgr, self.config)
        depth_norm, depth_gray = normalize_depth(rectified.depth_mm, rectified.valid_mask, self.config)
        height_uint8 = normalize_height_map(
            rectified.height_map_mm,
            rectified.valid_mask,
            max_height_mm=self.config.max_height_mm,
        )

        return NormalisedBoard(
            rgb_bgr=rgb_norm,
            rgb_gray=gray,
            depth_mm=depth_norm,
            depth_gray=depth_gray,
            valid_mask=rectified.valid_mask.copy(),
            height_map_mm=rectified.height_map_mm.copy(),
            height_uint8=height_uint8,
        )


## Point / tray segmentation and manual ROIs

Creates grid-style point masks, dice/cube areas, the middle-strip exclusion, and `checker_detection_area`.

Current ROI values:

```python
checker_detection_area_frac = (0.10, 0.00, 1.00, 1.00)
middle_strip_exclusion_frac = (0.00, 0.38, 1.00, 0.60)
dice_area_frac = (0.10, 0.38, 1.00, 0.60)
cube_area_frac = (0.01, 0.45, 0.08, 0.52)
depth_analysis_area_frac = (0.10, 0.00, 1.00, 1.00)
```

The point masks use grid/rectangular columns by default, not triangle contours.


In [ ]:
%%writefile "Geometric + Visual Preprocessing/point_tray_segmentation.py"
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np

from backgammon_types import RegionMasks

RectFrac = Tuple[float, float, float, float]


@dataclass
class SegmentationConfig:
    outer_margin_frac_x: float = 0.045
    outer_margin_frac_y: float = 0.05
    bar_frac: float = 0.07
    point_tip_frac: float = 0.16
    point_base_frac: float = 0.15
    bearoff_frac: float = 0.09

    # Manually tunable rectified-board regions. Fractions are (x1, y1, x2, y2)
    # in the rectified image coordinate frame, not raw camera coordinates.
    # These defaults are based on the most stable live-viewer crop reported so far.
    checker_detection_area_frac: Optional[RectFrac] = (0.10, 0.00, 1.00, 1.00)
    middle_strip_exclusion_frac: Optional[RectFrac] = (0.00, 0.38, 1.00, 0.60)
    dice_area_frac: Optional[RectFrac] = (0.10, 0.38, 1.00, 0.60)
    cube_area_frac: Optional[RectFrac] = (0.01, 0.45, 0.08, 0.52)
    depth_analysis_area_frac: Optional[RectFrac] = (0.10, 0.00, 1.00, 1.00)

    # Borne-off checker trays are deliberately excluded from normal point-stack
    # detection because they behave differently from playable points. Add a
    # separate tray counter later if needed.
    include_bar_in_checker_area: bool = True
    include_bearoff_in_checker_area: bool = False

    # Visualisation opacity for ROI overlays.
    roi_overlay_alpha: float = 0.28

    # The original version used triangular point masks matching the printed
    # board graphics. For stable checker detection it is often better to use
    # grid/rectangular point columns instead, because checker occupancy is
    # constrained by board regions rather than by the printed triangle art.
    #
    # False = grid/rectangular point regions.
    # True  = old triangular point regions.
    use_triangular_point_masks: bool = False

    # Draw the final checker_detection_area contour into RegionMasks.overlay_bgr.
    # This is off by default to keep ROI preview uncluttered.
    draw_checker_detection_contour: bool = False


def polygon_mask(shape_hw: Tuple[int, int], polygon_xy: np.ndarray) -> np.ndarray:
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(mask, polygon_xy.astype(np.int32), 1)
    return mask


def rect_to_pixels(shape_hw: Tuple[int, int], frac: Optional[RectFrac]) -> Optional[Tuple[int, int, int, int]]:
    if frac is None:
        return None
    h, w = shape_hw
    x1f, y1f, x2f, y2f = frac
    x1 = int(round(np.clip(x1f, 0.0, 1.0) * w))
    y1 = int(round(np.clip(y1f, 0.0, 1.0) * h))
    x2 = int(round(np.clip(x2f, 0.0, 1.0) * w))
    y2 = int(round(np.clip(y2f, 0.0, 1.0) * h))
    x1, x2 = sorted((max(0, min(w, x1)), max(0, min(w, x2))))
    y1, y2 = sorted((max(0, min(h, y1)), max(0, min(h, y2))))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def rect_mask(shape_hw: Tuple[int, int], frac: Optional[RectFrac]) -> np.ndarray:
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    rect = rect_to_pixels(shape_hw, frac)
    if rect is None:
        return mask
    x1, y1, x2, y2 = rect
    mask[y1:y2, x1:x2] = 1
    return mask


def union_masks(shape_hw: Tuple[int, int], masks: List[np.ndarray]) -> np.ndarray:
    out = np.zeros(shape_hw, dtype=np.uint8)
    for mask in masks:
        out = cv2.bitwise_or(out, mask.astype(np.uint8))
    return out


def subtract_masks(base: np.ndarray, exclusions: List[np.ndarray]) -> np.ndarray:
    out = base.copy().astype(np.uint8)
    for exclusion in exclusions:
        out[exclusion.astype(bool)] = 0
    return out


class PointTraySegmenter:
    """
    Produces canonical masks for the 24 points, central bar, bear-off trays,
    dice/cube areas, middle-strip exclusion, and checker-only detection area.

    All manually tuned ROIs are defined after perspective rectification, so the
    numbers are stable fractions of the canonical board image rather than raw
    camera pixel coordinates.
    """

    def __init__(self, config: Optional[SegmentationConfig] = None) -> None:
        self.config = config or SegmentationConfig()

    @staticmethod
    def _label_rect(overlay: np.ndarray, rect: Optional[Tuple[int, int, int, int]], name: str, colour: Tuple[int, int, int]) -> None:
        if rect is None:
            return
        x1, y1, x2, y2 = rect
        cv2.rectangle(overlay, (x1, y1), (x2, y2), colour, 2)
        cv2.putText(
            overlay,
            name,
            (x1 + 6, max(18, y1 + 18)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            colour,
            1,
            cv2.LINE_AA,
        )

    def segment(self, shape_hw: Tuple[int, int]) -> RegionMasks:
        h, w = shape_hw
        cfg = self.config

        mx = int(cfg.outer_margin_frac_x * w)
        my = int(cfg.outer_margin_frac_y * h)
        bearoff_w = int(cfg.bearoff_frac * w)
        bar_w = int(cfg.bar_frac * w)

        playable_left = mx + bearoff_w
        playable_right = w - mx - bearoff_w
        playable_width = playable_right - playable_left
        half_playable = (playable_width - bar_w) // 2

        left_half_x0 = playable_left
        left_half_x1 = playable_left + half_playable
        bar_x0 = left_half_x1
        bar_x1 = bar_x0 + bar_w
        right_half_x0 = bar_x1
        right_half_x1 = playable_right

        top_y0 = my
        top_y1 = h // 2 - my // 2
        bot_y0 = h // 2 + my // 2
        bot_y1 = h - my

        masks: Dict[str, np.ndarray] = {}
        overlay = np.zeros((h, w, 3), dtype=np.uint8)

        def tint(mask: np.ndarray, colour: Tuple[int, int, int], alpha: float = 0.35) -> None:
            colour_img = np.full_like(overlay, colour)
            blended = cv2.addWeighted(overlay, 1.0 - alpha, colour_img, alpha, 0)
            overlay[mask > 0] = blended[mask > 0]

        def make_half_points(x0: int, x1: int, is_top: bool, point_numbers: List[int]) -> None:
            span = x1 - x0
            point_w = span / 6.0

            for i in range(6):
                px0 = int(round(x0 + i * point_w))
                px1 = int(round(x0 + (i + 1) * point_w))
                point_name = f"point_{point_numbers[i]:02d}"

                if cfg.use_triangular_point_masks:
                    # Legacy visual/geometry mode: mask the printed triangular point.
                    if is_top:
                        poly = np.array(
                            [[px0, top_y0], [px1, top_y0], [(px0 + px1) // 2, top_y1]],
                            dtype=np.int32,
                        )
                    else:
                        poly = np.array(
                            [[px0, bot_y1], [px1, bot_y1], [(px0 + px1) // 2, bot_y0]],
                            dtype=np.int32,
                        )
                    mask = polygon_mask((h, w), poly)
                else:
                    # Default detection mode: grid/rectangular point columns.
                    # This removes the dependency on the printed triangle contours.
                    mask = np.zeros((h, w), dtype=np.uint8)
                    if is_top:
                        mask[top_y0:top_y1, px0:px1] = 1
                    else:
                        mask[bot_y0:bot_y1, px0:px1] = 1

                masks[point_name] = mask
                tint(mask, (0, 160, 255) if is_top else (255, 180, 0), alpha=0.18)

        # Standard numbering in a canonical rectified view:
        # top-left half 13..18, top-right half 19..24,
        # bottom-left half 12..7, bottom-right half 6..1.
        make_half_points(left_half_x0, left_half_x1, True, [13, 14, 15, 16, 17, 18])
        make_half_points(right_half_x0, right_half_x1, True, [19, 20, 21, 22, 23, 24])
        make_half_points(left_half_x0, left_half_x1, False, [12, 11, 10, 9, 8, 7])
        make_half_points(right_half_x0, right_half_x1, False, [6, 5, 4, 3, 2, 1])

        bar_mask = np.zeros((h, w), dtype=np.uint8)
        bar_mask[:, bar_x0:bar_x1] = 1
        masks["bar"] = bar_mask

        masks["bearoff_left"] = np.zeros((h, w), dtype=np.uint8)
        masks["bearoff_left"][:, mx:playable_left] = 1
        masks["bearoff_right"] = np.zeros((h, w), dtype=np.uint8)
        masks["bearoff_right"][:, playable_right:w - mx] = 1

        # Manual named ROIs.
        masks["checker_detection_area_rect"] = rect_mask((h, w), cfg.checker_detection_area_frac)
        masks["middle_strip_exclusion"] = rect_mask((h, w), cfg.middle_strip_exclusion_frac)
        masks["dice_area"] = rect_mask((h, w), cfg.dice_area_frac)
        masks["cube_area"] = rect_mask((h, w), cfg.cube_area_frac)
        masks["depth_analysis_area"] = rect_mask((h, w), cfg.depth_analysis_area_frac)

        point_names = [f"point_{i:02d}" for i in range(1, 25)]
        auxiliary_names = ["bar", "bearoff_left", "bearoff_right"]

        checker_sources = [masks[name] for name in point_names]
        if cfg.include_bar_in_checker_area:
            checker_sources.append(masks["bar"])
        if cfg.include_bearoff_in_checker_area:
            checker_sources.extend([masks["bearoff_left"], masks["bearoff_right"]])

        checker_area = union_masks((h, w), checker_sources)

        # Restrict to manual overall checker rectangle/depth area.
        rect_area = masks["checker_detection_area_rect"]
        if np.count_nonzero(rect_area) > 0:
            checker_area = cv2.bitwise_and(checker_area, rect_area)
        depth_area = masks["depth_analysis_area"]
        if np.count_nonzero(depth_area) > 0:
            checker_area = cv2.bitwise_and(checker_area, depth_area)

        # Remove non-checker areas before piece detection can see them.
        checker_area = subtract_masks(
            checker_area,
            [
                masks["dice_area"],
                masks["cube_area"],
                masks["middle_strip_exclusion"],
            ],
        )
        masks["checker_detection_area"] = checker_area

        # Visual overlays.
        tint(masks["bar"], (60, 60, 60), alpha=0.40)
        tint(masks["bearoff_left"], (40, 0, 160), alpha=0.35)
        tint(masks["bearoff_right"], (40, 160, 0), alpha=0.35)
        tint(masks["dice_area"], (255, 255, 0), alpha=0.45)
        tint(masks["cube_area"], (255, 0, 255), alpha=0.45)
        tint(masks["middle_strip_exclusion"], (0, 0, 255), alpha=0.20)

        # Draw named ROI rectangles and checker detection contour.
        self._label_rect(overlay, rect_to_pixels((h, w), cfg.checker_detection_area_frac), "checker_rect", (255, 255, 0))
        self._label_rect(overlay, rect_to_pixels((h, w), cfg.dice_area_frac), "dice_area", (0, 255, 255))
        self._label_rect(overlay, rect_to_pixels((h, w), cfg.cube_area_frac), "cube_area", (255, 0, 255))
        self._label_rect(overlay, rect_to_pixels((h, w), cfg.middle_strip_exclusion_frac), "middle_exclusion", (0, 0, 255))

        if cfg.draw_checker_detection_contour:
            contours, _ = cv2.findContours((checker_area * 255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay, contours, -1, (255, 255, 255), 2)

        return RegionMasks(
            masks=masks,
            overlay_bgr=overlay,
            point_names=point_names,
            auxiliary_names=auxiliary_names,
        )


__all__ = [
    "SegmentationConfig",
    "PointTraySegmenter",
    "polygon_mask",
    "rect_to_pixels",
    "rect_mask",
    "union_masks",
    "subtract_masks",
]


## ROI preview utility

Live ROI overlay tool. It supports automatic board lock, manual corner selection, full-frame fallback, and save/load of `manual_board_corners.json`.


In [ ]:
%%writefile "Geometric + Visual Preprocessing/roi_preview.py"
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple
import json
import sys

import cv2
import numpy as np

_MODULE_DIR = Path(__file__).resolve().parent
_PROJECT_ROOT = _MODULE_DIR.parent if _MODULE_DIR.name in {"Acquisition", "Geometric + Visual Preprocessing", "Interpretation"} else _MODULE_DIR

for _rel in ("", "Acquisition", "Geometric + Visual Preprocessing", "Interpretation"):
    _path = _PROJECT_ROOT / _rel if _rel else _PROJECT_ROOT
    if _path.exists():
        _path_str = str(_path)
        if _path_str not in sys.path:
            sys.path.insert(0, _path_str)

from streams_module import OakSRStreams
from backgammon_types import BoardLock
from board_registration import BoardRegistrar
from perspective_rectification import PerspectiveRectifier
from image_normalisation import BoardNormaliser
from point_tray_segmentation import PointTraySegmenter


WINDOW_NAME = "ROI preview"


@dataclass
class PreviewState:
    mode: str = "auto"  # auto, manual, full-frame
    manual_points: Optional[np.ndarray] = None
    pending_clicks: List[Tuple[int, int]] = None
    last_raw_frame: Optional[np.ndarray] = None

    def __post_init__(self) -> None:
        if self.pending_clicks is None:
            self.pending_clicks = []


def colour_mask(mask: np.ndarray, colour_bgr: Tuple[int, int, int]) -> np.ndarray:
    out = np.zeros((*mask.shape[:2], 3), dtype=np.uint8)
    out[mask > 0] = colour_bgr
    return out


def add_text_panel(frame_bgr: np.ndarray, lines: List[str]) -> np.ndarray:
    out = frame_bgr.copy()
    y = 24
    for line in lines:
        cv2.putText(
            out,
            line,
            (10, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )
        cv2.putText(
            out,
            line,
            (10, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 0, 0),
            1,
            cv2.LINE_AA,
        )
        y += 24
    return out


def make_manual_lock(corners_xy: np.ndarray, frame_shape_hw: Tuple[int, int]) -> BoardLock:
    return BoardLock(
        valid=True,
        corners_xy=corners_xy.astype(np.float32),
        confidence=1.0,
        method="manual",
        frame_size_hw=frame_shape_hw,
        debug={"source": "roi_preview_manual"},
    )


def make_full_frame_lock(frame_shape_hw: Tuple[int, int]) -> BoardLock:
    h, w = frame_shape_hw
    corners = np.array(
        [
            [0.0, 0.0],
            [float(w - 1), 0.0],
            [float(w - 1), float(h - 1)],
            [0.0, float(h - 1)],
        ],
        dtype=np.float32,
    )
    return BoardLock(
        valid=True,
        corners_xy=corners,
        confidence=1.0,
        method="full-frame",
        frame_size_hw=frame_shape_hw,
        debug={"source": "roi_preview_full_frame"},
    )


def draw_pending_points(frame_bgr: np.ndarray, state: PreviewState) -> np.ndarray:
    out = frame_bgr.copy()

    if state.manual_points is not None:
        pts = state.manual_points.astype(int)
        cv2.polylines(out, [pts.reshape(-1, 1, 2)], isClosed=True, color=(0, 255, 0), thickness=2)
        for i, (x, y) in enumerate(pts):
            cv2.circle(out, (int(x), int(y)), 5, (0, 255, 0), -1)
            cv2.putText(out, str(i + 1), (int(x) + 6, int(y) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    for i, (x, y) in enumerate(state.pending_clicks):
        cv2.circle(out, (x, y), 5, (0, 255, 255), -1)
        cv2.putText(out, str(i + 1), (x + 6, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    return out


def mouse_callback(event, x, y, flags, param) -> None:
    state: PreviewState = param

    if event != cv2.EVENT_LBUTTONDOWN:
        return

    if state.mode != "manual":
        return

    state.pending_clicks.append((int(x), int(y)))

    if len(state.pending_clicks) == 4:
        state.manual_points = np.array(state.pending_clicks, dtype=np.float32)
        state.pending_clicks.clear()
        print("Manual board corners set. Order assumed: top-left, top-right, bottom-right, bottom-left.")


def save_manual_corners(path: Path, corners_xy: np.ndarray) -> None:
    payload = {
        "corner_order": "top-left, top-right, bottom-right, bottom-left",
        "corners_xy": corners_xy.astype(float).tolist(),
    }
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Saved manual corners to: {path}")


def load_manual_corners(path: Path) -> Optional[np.ndarray]:
    if not path.exists():
        print(f"No saved manual corners found at: {path}")
        return None

    payload = json.loads(path.read_text(encoding="utf-8"))
    corners = np.array(payload["corners_xy"], dtype=np.float32)

    if corners.shape != (4, 2):
        print(f"Invalid saved corners shape: {corners.shape}")
        return None

    print(f"Loaded manual corners from: {path}")
    return corners


def build_roi_overlay(base_bgr: np.ndarray, regions) -> np.ndarray:
    overlay = base_bgr.copy()

    named_colours = {
        "checker_detection_area": (0, 255, 0),          # green
        "dice_area": (0, 255, 255),                     # yellow/cyan
        "cube_area": (255, 0, 255),                     # magenta
        "middle_strip_exclusion": (0, 0, 255),          # red
        "checker_detection_area_rect": (255, 255, 0),   # cyan
        "depth_analysis_area": (255, 120, 0),           # orange/blueish
    }

    # First blend filled masks.
    fill = np.zeros_like(base_bgr)
    for name, colour in named_colours.items():
        mask = regions.masks.get(name)
        if mask is None:
            continue
        fill[mask.astype(bool)] = colour

    overlay = cv2.addWeighted(overlay, 0.72, fill, 0.28, 0)

    # Then draw outlines and labels.
    for name, colour in named_colours.items():
        mask = regions.masks.get(name)
        if mask is None:
            continue

        mask_u8 = (mask.astype(np.uint8) * 255)
        ys, xs = np.where(mask > 0)
        if len(xs) > 0:
            x1, x2 = int(xs.min()), int(xs.max())
            y1, y2 = int(ys.min()), int(ys.max())

            # The checker_detection_area can contain many grid cells; drawing all
            # contours makes the preview noisy. Show it as a filled green overlay
            # with a simple bounding rectangle instead. Other named ROIs still get
            # their exact contour/rectangle outlines.
            if name == "checker_detection_area":
                cv2.rectangle(overlay, (x1, y1), (x2, y2), colour, 2)
            else:
                contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(overlay, contours, -1, colour, 2)

            cv2.putText(
                overlay,
                name,
                (x1 + 5, max(22, y1 + 22)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                colour,
                2,
                cv2.LINE_AA,
            )

    return overlay


def main() -> None:
    state = PreviewState()
    corners_path = _PROJECT_ROOT / "manual_board_corners.json"

    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
    cv2.setMouseCallback(WINDOW_NAME, mouse_callback, state)

    cam = OakSRStreams(enable_left=True, enable_depth=True, fps=15).start()

    registrar = BoardRegistrar()
    rectifier = PerspectiveRectifier()
    normaliser = BoardNormaliser()
    segmenter = PointTraySegmenter()

    print("ROI preview controls:")
    print("  a = auto board lock")
    print("  m = manual corner mode; click 4 corners: TL, TR, BR, BL")
    print("  f = full-frame fallback lock")
    print("  c = clear manual points")
    print("  s = save manual corners")
    print("  l = load manual corners")
    print("  q/Esc = quit")

    try:
        while True:
            rgb = cam.get_left_frame()
            depth = cam.get_depth_frame()

            if rgb is None:
                key = cv2.waitKey(1) & 0xFF
                if key in (ord("q"), 27):
                    break
                continue

            state.last_raw_frame = rgb.copy()
            h, w = rgb.shape[:2]

            lock: Optional[BoardLock] = None
            status_lines: List[str] = []

            if state.mode == "full-frame":
                lock = make_full_frame_lock((h, w))
                status_lines.append("Mode: FULL-FRAME fallback lock")
                status_lines.append("Useful for rough ROI tuning only.")
            elif state.mode == "manual":
                if state.manual_points is not None:
                    lock = make_manual_lock(state.manual_points, (h, w))
                    status_lines.append("Mode: MANUAL board lock")
                else:
                    view = draw_pending_points(rgb, state)
                    status_lines = [
                        "Mode: MANUAL corner selection",
                        "Click board corners in order: TL, TR, BR, BL",
                        f"Clicked: {len(state.pending_clicks)}/4",
                        "a=auto  f=full-frame  c=clear  l=load  q=quit",
                    ]
                    cv2.imshow(WINDOW_NAME, add_text_panel(view, status_lines))
                    key = cv2.waitKey(1) & 0xFF
                    if key in (ord("q"), 27):
                        break
                    elif key == ord("a"):
                        state.mode = "auto"
                    elif key == ord("f"):
                        state.mode = "full-frame"
                    elif key == ord("c"):
                        state.pending_clicks.clear()
                        state.manual_points = None
                    elif key == ord("l"):
                        loaded = load_manual_corners(corners_path)
                        if loaded is not None:
                            state.manual_points = loaded
                            state.mode = "manual"
                    continue
            else:
                lock = registrar.update(rgb)
                status_lines.append("Mode: AUTO BoardRegistrar")

            if lock is None or not lock.valid:
                view = draw_pending_points(rgb, state)
                status_lines.extend(
                    [
                        "Board lock invalid.",
                        "Press m to click manual board corners.",
                        "Press f to use full-frame fallback for rough ROI tuning.",
                        "Press l to load saved manual corners.",
                        "q/Esc=quit",
                    ]
                )
                cv2.imshow(WINDOW_NAME, add_text_panel(view, status_lines))
            else:
                if depth is None:
                    # Use a zero depth frame so the rectifier can still render RGB ROI overlay in fallback cases.
                    depth = np.zeros((h, w), dtype=np.uint16)

                # If a cropped-depth stream comes back different from RGB, the streams module should
                # normally expand it to full frame. This resize is just a defensive fallback.
                if depth.shape[:2] != rgb.shape[:2]:
                    depth = cv2.resize(depth, (w, h), interpolation=cv2.INTER_NEAREST)

                try:
                    rectified = rectifier.rectify(rgb, depth, lock)
                    normalised = normaliser.normalize(rectified)
                    regions = segmenter.segment(normalised.rgb_bgr.shape[:2])
                    preview = build_roi_overlay(normalised.rgb_bgr, regions)

                    status_lines.extend(
                        [
                            f"Lock: {lock.method} conf={lock.confidence:.2f}",
                            "green=checker_detection_area  yellow=dice_area",
                            "magenta=cube_area  red=middle_strip_exclusion",
                            "a=auto  m=manual  f=full-frame  s=save  l=load  q=quit",
                        ]
                    )
                    preview = add_text_panel(preview, status_lines)
                    cv2.imshow(WINDOW_NAME, preview)
                except Exception as exc:
                    view = draw_pending_points(rgb, state)
                    status_lines.extend(
                        [
                            f"ROI preview error: {type(exc).__name__}: {exc}",
                            "Try f for full-frame, or m to set manual corners.",
                        ]
                    )
                    cv2.imshow(WINDOW_NAME, add_text_panel(view, status_lines))

            key = cv2.waitKey(1) & 0xFF

            if key in (ord("q"), 27):
                break
            elif key == ord("a"):
                state.mode = "auto"
                print("Mode set to auto.")
            elif key == ord("m"):
                state.mode = "manual"
                state.pending_clicks.clear()
                print("Manual mode: click TL, TR, BR, BL board corners.")
            elif key == ord("f"):
                state.mode = "full-frame"
                print("Mode set to full-frame fallback.")
            elif key == ord("c"):
                state.pending_clicks.clear()
                state.manual_points = None
                print("Cleared manual corners.")
            elif key == ord("s"):
                if state.manual_points is not None:
                    save_manual_corners(corners_path, state.manual_points)
                else:
                    print("No manual corners to save.")
            elif key == ord("l"):
                loaded = load_manual_corners(corners_path)
                if loaded is not None:
                    state.manual_points = loaded
                    state.mode = "manual"

    finally:
        cam.stop()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


# 4. Interpretation

Checker/stack detection, dice/cube reading, temporal fusion, validation, and the end-to-end board-state pipeline.


## Depth stack classifier

Shared stack-detection logic used by the diagnostic viewer: temporal depth support, RGB-restored chip masks and 1/2-high hysteresis.


In [ ]:
%%writefile Interpretation/depth_stack_classifier.py
from __future__ import annotations

from dataclasses import dataclass
from collections import deque
from typing import Deque, Dict, List, Literal, Optional, Tuple

import cv2
import numpy as np

HeightStat = Literal["median", "p75", "p90", "max", "top35", "top25"]
RoiFrac = Tuple[float, float, float, float]
RectFracs = Tuple[RoiFrac, ...]


@dataclass
class ChipCandidate:
    x: float
    y: float
    radius: float
    area_px: float
    circularity: float
    source: str = "unknown"
    support_ratio: float = 0.0
    support_pixels: int = 0
    valid_depth_ratio: float = 0.0
    height_median_mm: float = 0.0
    height_p75_mm: float = 0.0
    height_p90_mm: float = 0.0
    height_max_mm: float = 0.0
    height_top35_mm: float = 0.0
    height_top25_mm: float = 0.0
    height_used_mm: float = 0.0
    raw_stack_count: int = 0
    stack_count: int = 0
    track_id: int = -1
    confidence: float = 0.0


@dataclass
class StackClassificationResult:
    candidates: List[ChipCandidate]
    height_mm: np.ndarray
    valid_overlap_mask: np.ndarray
    stable_depth_support_mask: np.ndarray
    rgb_restored_circle_mask: np.ndarray
    rgb_candidate_mask: np.ndarray
    piece_presence_mask: np.ndarray
    stack_class_bgr: np.ndarray
    overlay_bgr: np.ndarray
    diagnostics: Dict[str, float]


class EmptyBoardBaseline:
    """Median empty-board depth reference plus per-pixel depth-noise estimate."""

    def __init__(
        self,
        samples_required: int = 60,
        min_valid_ratio: float = 0.15,
        noise_percentile: float = 90.0,
        min_noise_floor_mm: float = 0.5,
    ) -> None:
        self.samples_required = int(samples_required)
        self.min_valid_ratio = float(min_valid_ratio)
        self.noise_percentile = float(noise_percentile)
        self.min_noise_floor_mm = float(min_noise_floor_mm)
        self.samples: List[np.ndarray] = []
        self.reference_depth_mm: Optional[np.ndarray] = None
        self.noise_floor_mm: Optional[np.ndarray] = None

    @property
    def ready(self) -> bool:
        return self.reference_depth_mm is not None

    def clear(self) -> None:
        self.samples.clear()
        self.reference_depth_mm = None
        self.noise_floor_mm = None

    def add_sample(self, depth_mm: np.ndarray) -> bool:
        valid_ratio = float(np.count_nonzero(depth_mm)) / float(depth_mm.size)
        if valid_ratio < self.min_valid_ratio:
            return False
        self.samples.append(depth_mm.copy())
        if len(self.samples) < self.samples_required:
            return False

        stack = np.stack([f.astype(np.float32) for f in self.samples], axis=0)
        stack[stack <= 0] = np.nan
        baseline_nan = np.nanmedian(stack, axis=0)
        dev = np.abs(stack - baseline_nan[None, :, :])
        noise = np.nanpercentile(dev, self.noise_percentile, axis=0)
        noise[~np.isfinite(noise)] = 0
        noise = np.maximum(noise, self.min_noise_floor_mm)

        baseline = baseline_nan.copy()
        baseline[~np.isfinite(baseline)] = 0
        self.reference_depth_mm = baseline.astype(np.float32)
        self.noise_floor_mm = noise.astype(np.float32)
        self.samples.clear()
        return True

    def height_above_board(self, depth_mm: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if self.reference_depth_mm is None:
            raise RuntimeError("No empty-board baseline has been captured yet.")
        if depth_mm.shape != self.reference_depth_mm.shape:
            depth_mm = cv2.resize(
                depth_mm.astype(np.float32),
                (self.reference_depth_mm.shape[1], self.reference_depth_mm.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )
        current = depth_mm.astype(np.float32)
        baseline = self.reference_depth_mm
        valid = (current > 0) & (baseline > 0)
        height = baseline - current
        height[~valid] = 0
        height[height < 0] = 0
        return height.astype(np.float32), valid.astype(np.uint8)


def depth_to_gray_fixed(depth_mm: np.ndarray, dmin: float = 360.0, dmax: float = 435.0) -> np.ndarray:
    d = depth_mm.astype(np.float32)
    out = np.clip((d - dmin) * 255.0 / max(dmax - dmin, 1.0), 0, 255).astype(np.uint8)
    out[depth_mm <= 0] = 0
    return out


def height_to_gray(height_mm: np.ndarray, height_max_mm: float = 30.0) -> np.ndarray:
    h = np.clip(height_mm.astype(np.float32), 0, max(height_max_mm, 1.0))
    return (h * 255.0 / max(height_max_mm, 1.0)).astype(np.uint8)


def _rect_mask(shape_hw: Tuple[int, int], frac: Optional[Tuple[float, float, float, float]]) -> np.ndarray:
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    if frac is None:
        return mask
    x1f, y1f, x2f, y2f = frac
    x1 = int(np.clip(x1f, 0.0, 1.0) * w)
    y1 = int(np.clip(y1f, 0.0, 1.0) * h)
    x2 = int(np.clip(x2f, 0.0, 1.0) * w)
    y2 = int(np.clip(y2f, 0.0, 1.0) * h)
    if x2 > x1 and y2 > y1:
        mask[y1:y2, x1:x2] = 255
    return mask


def _roi_mask(shape_hw: Tuple[int, int], roi_frac: Optional[RoiFrac]) -> np.ndarray:
    if roi_frac is None:
        return np.ones(shape_hw, dtype=np.uint8) * 255
    return _rect_mask(shape_hw, roi_frac)


def _allowed_mask(shape_hw: Tuple[int, int], roi_frac: Optional[RoiFrac], exclusion_fracs: RectFracs = ()) -> np.ndarray:
    mask = _roi_mask(shape_hw, roi_frac)
    for frac in exclusion_fracs:
        ex = _rect_mask(shape_hw, frac)
        mask[ex > 0] = 0
    return mask


def clean_mask(mask: np.ndarray, open_px: int = 3, close_px: int = 5, min_area_px: int = 40) -> np.ndarray:
    mask = (mask > 0).astype(np.uint8) * 255
    if open_px > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_px | 1, open_px | 1))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    if close_px > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_px | 1, close_px | 1))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = np.zeros_like(mask)
    for lab in range(1, n):
        if int(stats[lab, cv2.CC_STAT_AREA]) >= min_area_px:
            out[labels == lab] = 255
    return out


class TemporalMaskFilter:
    """Pixel-level temporal persistence filter for unstable depth support masks."""

    def __init__(
        self,
        window: int = 3,
        required: int = 2,
        *,
        open_px: int = 3,
        close_px: int = 5,
        min_area_px: int = 30,
    ) -> None:
        self.window = max(1, int(window))
        self.required = max(1, min(int(required), self.window))
        self.open_px = int(open_px)
        self.close_px = int(close_px)
        self.min_area_px = int(min_area_px)
        self.history: Deque[np.ndarray] = deque(maxlen=self.window)

    @property
    def history_size(self) -> int:
        return len(self.history)

    def reset(self) -> None:
        self.history.clear()

    def update(self, mask: np.ndarray) -> np.ndarray:
        current = (mask > 0).astype(np.uint8)
        self.history.append(current)
        stack = np.stack(list(self.history), axis=0).astype(np.uint8)
        count = np.sum(stack, axis=0)
        req = min(self.required, len(self.history))
        persistent = (count >= req).astype(np.uint8) * 255
        return clean_mask(persistent, open_px=self.open_px, close_px=self.close_px, min_area_px=self.min_area_px)



class StackCountSmoother:
    """
    Temporally smooths the 1-high vs 2-high stack label for RGB-restored chip candidates.

    This is intentionally separate from the detection gate. A candidate can be
    detected with permissive depth support, while the stack label itself changes
    only after repeated height evidence. That prevents true two-high stacks from
    flickering between 1 and 2 when passive stereo under-reads a few frames.
    """

    def __init__(
        self,
        *,
        window: int = 5,
        promote_votes: int = 2,
        demote_votes: int = 4,
        hold_misses: int = 2,
        promote_height_mm: float = 13.0,
        demote_height_mm: float = 11.0,
        match_distance_px: float = 28.0,
    ) -> None:
        self.window = max(1, int(window))
        self.promote_votes = max(1, int(promote_votes))
        self.demote_votes = max(1, int(demote_votes))
        self.hold_misses = max(0, int(hold_misses))
        self.promote_height_mm = float(promote_height_mm)
        self.demote_height_mm = float(demote_height_mm)
        self.match_distance_px = float(match_distance_px)
        self._tracks: Dict[int, Dict[str, object]] = {}
        self._next_id = 1

    def reset(self) -> None:
        self._tracks.clear()
        self._next_id = 1

    def _match_track(self, cand: ChipCandidate, used: set[int]) -> int:
        best_id = -1
        best_d = float("inf")
        max_d = max(self.match_distance_px, cand.radius * 1.8)
        for tid, tr in self._tracks.items():
            if tid in used:
                continue
            dx = float(tr["x"]) - cand.x
            dy = float(tr["y"]) - cand.y
            d = float((dx * dx + dy * dy) ** 0.5)
            if d < best_d and d <= max_d:
                best_d = d
                best_id = tid
        if best_id >= 0:
            return best_id
        tid = self._next_id
        self._next_id += 1
        self._tracks[tid] = {
            "x": cand.x,
            "y": cand.y,
            "stable": max(1, int(cand.stack_count)),
            "raw_hist": deque(maxlen=self.window),
            "height_hist": deque(maxlen=self.window),
            "misses": 0,
        }
        return tid

    def update(self, candidates: List[ChipCandidate]) -> List[ChipCandidate]:
        used: set[int] = set()
        active_ids: set[int] = set()

        for cand in candidates:
            raw = int(np.clip(cand.stack_count, 1, 2))
            cand.raw_stack_count = raw
            tid = self._match_track(cand, used)
            used.add(tid)
            active_ids.add(tid)
            tr = self._tracks[tid]

            raw_hist: Deque[int] = tr["raw_hist"]  # type: ignore[assignment]
            height_hist: Deque[float] = tr["height_hist"]  # type: ignore[assignment]
            raw_hist.append(raw)
            height_hist.append(float(cand.height_used_mm))

            prev = int(tr.get("stable", raw))
            high_votes = sum(1 for r, h in zip(raw_hist, height_hist) if int(r) >= 2 or float(h) >= self.promote_height_mm)
            low_votes = sum(1 for h in height_hist if float(h) <= self.demote_height_mm)
            one_votes = sum(1 for r in raw_hist if int(r) <= 1)

            if prev >= 2:
                # Once a chip has become a two-stack, require repeated low-height
                # evidence before demoting. This fixes 2 -> 1 flicker on edges.
                if low_votes >= self.demote_votes and one_votes >= self.demote_votes:
                    stable = 1
                else:
                    stable = 2
            else:
                if high_votes >= self.promote_votes:
                    stable = 2
                else:
                    stable = 1

            tr["stable"] = int(stable)
            # Slowly update the track centre so it follows real candidate movement
            # but does not jump wildly on a noisy frame.
            tr["x"] = 0.65 * float(tr["x"]) + 0.35 * cand.x
            tr["y"] = 0.65 * float(tr["y"]) + 0.35 * cand.y
            tr["misses"] = 0

            cand.track_id = tid
            cand.stack_count = int(stable)

        # Age unmatched tracks and remove old ones. Tracks are only used to smooth
        # labels for candidates that are actually detected; held tracks are not
        # hallucinated into the output.
        for tid in list(self._tracks.keys()):
            if tid not in active_ids:
                self._tracks[tid]["misses"] = int(self._tracks[tid].get("misses", 0)) + 1
                if int(self._tracks[tid]["misses"]) > self.hold_misses:
                    del self._tracks[tid]

        return candidates

def build_height_support_mask(
    height_mm: np.ndarray,
    valid_mask: np.ndarray,
    *,
    min_piece_height_mm: float,
    max_piece_height_mm: float,
    roi_frac: Optional[RoiFrac],
    exclusion_fracs: RectFracs = (),
    noise_floor_mm: Optional[np.ndarray] = None,
    noise_margin_mm: float = 1.5,
    min_area_px: int = 24,
) -> np.ndarray:
    """Depth-only temporal support. This is only an anchor, not the final piece shape."""
    height = height_mm.astype(np.float32).copy()
    height[valid_mask == 0] = 0
    try:
        height_smooth = cv2.medianBlur(height, 3)
    except cv2.error:
        height_smooth = height

    if noise_floor_mm is not None:
        noise = noise_floor_mm.astype(np.float32)
        if noise.shape != height_smooth.shape:
            noise = cv2.resize(noise, (height_smooth.shape[1], height_smooth.shape[0]), interpolation=cv2.INTER_NEAREST)
        min_height_map = np.maximum(float(min_piece_height_mm), noise + float(noise_margin_mm))
    else:
        min_height_map = np.full_like(height_smooth, float(min_piece_height_mm), dtype=np.float32)

    raw = ((height_smooth >= min_height_map) & (height_smooth <= float(max_piece_height_mm)) & (valid_mask > 0)).astype(np.uint8) * 255
    raw = cv2.bitwise_and(raw, _allowed_mask(raw.shape[:2], roi_frac, exclusion_fracs))
    raw = clean_mask(raw, open_px=1, close_px=3, min_area_px=max(4, int(min_area_px)))
    return raw


def _top_band_stat(values: np.ndarray, keep_fraction: float) -> float:
    if values.size == 0:
        return 0.0
    keep_fraction = float(np.clip(keep_fraction, 0.05, 1.0))
    cutoff = np.percentile(values, 100.0 * (1.0 - keep_fraction))
    top = values[values >= cutoff]
    if top.size == 0:
        top = values
    return float(np.median(top))


def classify_stack_count(height_mm: float, chip_thickness_mm: float, min_piece_height_mm: float) -> int:
    if height_mm < max(1.0, min_piece_height_mm):
        return 0
    return max(1, int(round(height_mm / max(chip_thickness_mm, 1.0))))


def _circle_mask(shape_hw: Tuple[int, int], x: float, y: float, r: float, scale: float = 1.0) -> np.ndarray:
    mask = np.zeros(shape_hw, dtype=np.uint8)
    cv2.circle(mask, (int(round(x)), int(round(y))), max(1, int(round(r * scale))), 255, -1)
    return mask


def _support_stats_for_circle(support: np.ndarray, valid: np.ndarray, cand: ChipCandidate, inner_scale: float = 0.92) -> Tuple[float, int, float]:
    cmask = _circle_mask(support.shape[:2], cand.x, cand.y, cand.radius, inner_scale)
    area = max(1, int(np.count_nonzero(cmask)))
    support_px = int(np.count_nonzero((support > 0) & (cmask > 0)))
    valid_px = int(np.count_nonzero((valid > 0) & (cmask > 0)))
    return support_px / float(area), support_px, valid_px / float(area)


def _candidate_key(c: ChipCandidate) -> Tuple[float, float, float, float]:
    source_bonus = 2.0 if c.source.startswith("rgb") else 0.0
    return (source_bonus + c.support_ratio, float(c.support_pixels), c.circularity, -abs(c.radius))


def _nms_candidates(candidates: List[ChipCandidate], min_center_dist_factor: float = 1.18) -> List[ChipCandidate]:
    """Keep one centre per physical chip, but allow touching chips about 2 radii apart."""
    if not candidates:
        return []
    ordered = sorted(candidates, key=_candidate_key, reverse=True)
    kept: List[ChipCandidate] = []
    for cand in ordered:
        duplicate = False
        for old in kept:
            dist = float(np.hypot(cand.x - old.x, cand.y - old.y))
            # Same physical stack fragments often land within about one chip radius;
            # touching chips should be closer to two radii apart and survive this.
            min_dist = min(cand.radius, old.radius) * min_center_dist_factor
            if dist < min_dist:
                duplicate = True
                break
        if not duplicate:
            kept.append(cand)
    return kept


def _build_rgb_chip_mask(rgb_bgr: np.ndarray, roi_frac: Optional[RoiFrac], exclusion_fracs: RectFracs = ()) -> np.ndarray:
    """Bright low-saturation mask for the current pale/white diagnostic chips."""
    hsv = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    lab = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    # White/off-white chips: bright, relatively unsaturated, roughly neutral in Lab.
    mask = ((v >= 125) & (s <= 95) & (l >= 120) & (np.abs(a.astype(np.int16) - 128) <= 28) & (np.abs(b.astype(np.int16) - 128) <= 38)).astype(np.uint8) * 255
    mask = cv2.bitwise_and(mask, _allowed_mask(mask.shape[:2], roi_frac, exclusion_fracs))
    mask = clean_mask(mask, open_px=3, close_px=5, min_area_px=30)
    return mask


def _component_circularity(contour: np.ndarray) -> float:
    area = float(cv2.contourArea(contour))
    peri = float(cv2.arcLength(contour, True))
    if peri <= 0:
        return 0.0
    return float(4.0 * np.pi * area / (peri * peri))


def _split_component_by_distance(
    comp_mask: np.ndarray,
    *,
    expected_radius_px: float,
    min_radius_px: float,
    max_radius_px: float,
) -> List[ChipCandidate]:
    dist = cv2.distanceTransform((comp_mask > 0).astype(np.uint8) * 255, cv2.DIST_L2, 5)
    if dist.max() <= 0:
        return []
    max_k = max(5, int(round(expected_radius_px * 1.05)) | 1)
    dil = cv2.dilate(dist, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (max_k, max_k)))
    peaks = (dist == dil) & (dist > max(2.0, expected_radius_px * 0.42)) & (comp_mask > 0)
    n, labels, stats, cents = cv2.connectedComponentsWithStats(peaks.astype(np.uint8), connectivity=8)
    out: List[ChipCandidate] = []
    area = float(np.count_nonzero(comp_mask))
    for lab in range(1, n):
        x, y = cents[lab]
        r = float(np.clip(expected_radius_px, min_radius_px, max_radius_px))
        out.append(ChipCandidate(float(x), float(y), r, area, 1.0, source="rgb-split"))
    return out


def _rgb_circle_candidates(
    rgb_bgr: np.ndarray,
    stable_support: np.ndarray,
    *,
    roi_frac: Optional[RoiFrac],
    exclusion_fracs: RectFracs = (),
    expected_radius_px: float,
    min_radius_px: float,
    max_radius_px: float,
    split_touching: bool,
) -> Tuple[List[ChipCandidate], np.ndarray]:
    rgb_mask = _build_rgb_chip_mask(rgb_bgr, roi_frac, exclusion_fracs)
    n, labels, stats, cents = cv2.connectedComponentsWithStats(rgb_mask, connectivity=8)
    out: List[ChipCandidate] = []
    expected_area = float(np.pi * expected_radius_px * expected_radius_px)
    min_area = max(20.0, np.pi * min_radius_px * min_radius_px * 0.40)
    max_area_single = np.pi * max_radius_px * max_radius_px * 1.35

    for lab in range(1, n):
        area = float(stats[lab, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        comp = (labels == lab).astype(np.uint8) * 255
        contours, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        circ = _component_circularity(contour)
        x, y = cents[lab]
        (_, _), rr = cv2.minEnclosingCircle(contour)
        bbox_w = float(stats[lab, cv2.CC_STAT_WIDTH])
        bbox_h = float(stats[lab, cv2.CC_STAT_HEIGHT])
        elong = max(bbox_w, bbox_h) / max(1.0, min(bbox_w, bbox_h))

        should_split = split_touching and (area > expected_area * 1.35 or elong > 1.45 or area > max_area_single)
        if should_split:
            split = _split_component_by_distance(
                comp,
                expected_radius_px=expected_radius_px,
                min_radius_px=min_radius_px,
                max_radius_px=max_radius_px,
            )
            if len(split) >= 2:
                out.extend(split)
                continue

        # Use the expected physical radius when available so small depth/RGB holes do
        # not shrink the diagnostic footprint.
        r_est = expected_radius_px if expected_radius_px > 0 else np.sqrt(area / np.pi)
        r = float(np.clip(max(min_radius_px, min(max_radius_px, r_est, rr * 1.05)), min_radius_px, max_radius_px))
        out.append(ChipCandidate(float(x), float(y), r, area, circ, source="rgb-component"))
    return out, rgb_mask


def _support_seed_candidates(
    support: np.ndarray,
    *,
    expected_radius_px: float,
    min_radius_px: float,
    max_radius_px: float,
    min_component_area_px: int,
) -> List[ChipCandidate]:
    n, labels, stats, cents = cv2.connectedComponentsWithStats((support > 0).astype(np.uint8) * 255, connectivity=8)
    out: List[ChipCandidate] = []
    for lab in range(1, n):
        area = int(stats[lab, cv2.CC_STAT_AREA])
        if area < min_component_area_px:
            continue
        x, y = cents[lab]
        r = float(np.clip(expected_radius_px, min_radius_px, max_radius_px))
        out.append(ChipCandidate(float(x), float(y), r, float(area), 0.5, source="depth-seed"))
    return out


def _hough_candidates_constrained_by_rgb_and_support(
    rgb_bgr: np.ndarray,
    rgb_mask: np.ndarray,
    stable_support: np.ndarray,
    *,
    min_radius_px: float,
    max_radius_px: float,
    min_support_pixels: int,
) -> List[ChipCandidate]:
    search = cv2.bitwise_and(rgb_mask, cv2.dilate((stable_support > 0).astype(np.uint8) * 255, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))))
    if np.count_nonzero(search) == 0:
        return []
    gray = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    masked = cv2.bitwise_and(gray, gray, mask=cv2.dilate(search, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))))
    circles = cv2.HoughCircles(masked, cv2.HOUGH_GRADIENT, dp=1.2, minDist=max(8.0, min_radius_px * 1.8), param1=80, param2=18, minRadius=int(min_radius_px), maxRadius=int(max_radius_px))
    if circles is None:
        return []
    out: List[ChipCandidate] = []
    for x, y, r in np.round(circles[0]).astype(np.float32):
        cand = ChipCandidate(float(x), float(y), float(r), float(np.pi * r * r), 1.0, source="rgb-hough")
        _, support_px, _ = _support_stats_for_circle(stable_support, np.ones_like(stable_support, dtype=np.uint8), cand, inner_scale=0.92)
        if support_px >= min_support_pixels:
            out.append(cand)
    return out


def generate_restored_circle_candidates(
    rgb_bgr: np.ndarray,
    stable_support: np.ndarray,
    valid_mask: np.ndarray,
    *,
    min_radius_px: float,
    max_radius_px: float,
    expected_radius_px: Optional[float],
    min_candidate_support_ratio: float,
    min_support_pixels: int,
    use_hough: bool,
    split_touching: bool,
    roi_frac: Optional[RoiFrac],
    exclusion_fracs: RectFracs = (),
) -> Tuple[List[ChipCandidate], np.ndarray, np.ndarray, int]:
    if expected_radius_px is None or expected_radius_px <= 0:
        expected_radius_px = (float(min_radius_px) + float(max_radius_px)) * 0.5

    proposals: List[ChipCandidate] = []
    rgb_props, rgb_mask = _rgb_circle_candidates(
        rgb_bgr,
        stable_support,
        roi_frac=roi_frac,
        exclusion_fracs=exclusion_fracs,
        expected_radius_px=float(expected_radius_px),
        min_radius_px=min_radius_px,
        max_radius_px=max_radius_px,
        split_touching=split_touching,
    )
    proposals.extend(rgb_props)

    if use_hough:
        proposals.extend(_hough_candidates_constrained_by_rgb_and_support(rgb_bgr, rgb_mask, stable_support, min_radius_px=min_radius_px, max_radius_px=max_radius_px, min_support_pixels=min_support_pixels))

    # Fallback: if RGB misses a chip due blur/reflection, still allow a depth seed,
    # but it is lower priority and gets suppressed by RGB candidates in NMS.
    proposals.extend(_support_seed_candidates(stable_support, expected_radius_px=float(expected_radius_px), min_radius_px=min_radius_px, max_radius_px=max_radius_px, min_component_area_px=max(4, min_support_pixels)))
    proposed_count = len(proposals)

    accepted: List[ChipCandidate] = []
    for cand in proposals:
        support_ratio, support_px, valid_ratio = _support_stats_for_circle(stable_support, valid_mask, cand, inner_scale=0.92)
        cand.support_ratio = float(support_ratio)
        cand.support_pixels = int(support_px)
        cand.valid_depth_ratio = float(valid_ratio)

        # RGB candidates are already shape-valid, so a small amount of temporally
        # stable depth support is enough. Depth-only seeds remain stricter.
        if cand.source.startswith("rgb"):
            if support_px >= min_support_pixels and valid_ratio >= 0.01:
                accepted.append(cand)
        else:
            if support_px >= max(min_support_pixels, 8) and support_ratio >= min_candidate_support_ratio:
                accepted.append(cand)

    accepted = _nms_candidates(accepted, min_center_dist_factor=1.18)

    candidate_mask = np.zeros_like(stable_support)
    for cand in accepted:
        cv2.circle(candidate_mask, (int(round(cand.x)), int(round(cand.y))), max(2, int(round(cand.radius))), 255, -1)
    return accepted, candidate_mask, rgb_mask, proposed_count


def _height_values_in_candidate(
    height_mm: np.ndarray,
    valid_mask: np.ndarray,
    cand: ChipCandidate,
    *,
    min_depth_pixels: int,
    min_depth_ratio: float,
    sample_scale: float = 0.72,
) -> Tuple[np.ndarray, float]:
    mask = _circle_mask(height_mm.shape[:2], cand.x, cand.y, cand.radius, sample_scale)
    total = max(1, int(np.count_nonzero(mask)))
    good = (mask > 0) & (valid_mask > 0) & (height_mm > 0)
    ratio = float(np.count_nonzero(good)) / float(total)
    vals = height_mm[good].astype(np.float32)
    if vals.size < min_depth_pixels or ratio < min_depth_ratio:
        mask = _circle_mask(height_mm.shape[:2], cand.x, cand.y, cand.radius, 0.92)
        total = max(1, int(np.count_nonzero(mask)))
        good = (mask > 0) & (valid_mask > 0) & (height_mm > 0)
        ratio = float(np.count_nonzero(good)) / float(total)
        vals = height_mm[good].astype(np.float32)
    return vals, ratio


def _estimate_stack_from_distribution(vals: np.ndarray, chip_thickness_mm: float, min_piece_height_mm: float, max_piece_height_mm: float, requested_height: float) -> Tuple[int, float]:
    """Classify stack height from the distribution, not from a few high spikes."""
    t = max(float(chip_thickness_mm), 1.0)
    vals = vals[np.isfinite(vals)]
    vals = vals[(vals >= max(1.0, min_piece_height_mm * 0.65)) & (vals <= max_piece_height_mm)]
    if vals.size == 0:
        return 0, 0.0

    max_stack = max(1, min(8, int(np.ceil(max_piece_height_mm / t))))
    band_counts = []
    band_medians = []
    for k in range(1, max_stack + 1):
        centre = k * t
        lo = max(max(1.0, min_piece_height_mm * 0.65), centre - 0.45 * t)
        hi = centre + 0.55 * t
        band = vals[(vals >= lo) & (vals <= hi)]
        band_counts.append(int(band.size))
        band_medians.append(float(np.median(band)) if band.size else centre)

    best_idx = int(np.argmax(band_counts))
    best_count = band_counts[best_idx]
    best_stack = best_idx + 1
    n = int(vals.size)

    if best_count >= max(3, int(0.18 * n)):
        return best_stack, band_medians[best_idx]

    # Fallback: use the requested robust statistic, but round conservatively. This
    # helps very sparse edge stacks while avoiding single-chip overestimation.
    est = max(0, classify_stack_count(requested_height, t, min_piece_height_mm))
    if est > 1:
        # Require at least some evidence in the estimated band.
        idx = min(est, len(band_counts)) - 1
        if band_counts[idx] < max(2, int(0.08 * n)) and band_counts[0] >= max(2, band_counts[idx]):
            est = 1
            requested_height = band_medians[0] if band_counts[0] else float(np.median(vals))
    return est, float(requested_height)



def _estimate_stack_1_or_2(
    vals: np.ndarray,
    *,
    min_piece_height_mm: float,
    max_piece_height_mm: float,
    promote_height_mm: float,
    demote_height_mm: float,
    requested_height: float,
) -> Tuple[int, float]:
    """Return only 0, 1, or 2 for this project.

    The measured depth of a two-chip stack is often lower than the ideal 20 mm,
    especially near board edges. Instead of rounding by chip thickness, this uses
    a robust upper-band statistic and a lower promote threshold.
    """
    vals = vals[np.isfinite(vals)].astype(np.float32)
    vals = vals[(vals >= max(1.0, min_piece_height_mm * 0.6)) & (vals <= max_piece_height_mm)]
    if vals.size == 0:
        return 0, 0.0

    p50 = float(np.percentile(vals, 50))
    p75 = float(np.percentile(vals, 75))
    p90 = float(np.percentile(vals, 90))
    top35 = _top_band_stat(vals, 0.35)
    top25 = _top_band_stat(vals, 0.25)
    used = max(float(requested_height), top35)

    high_pixels = int(np.count_nonzero(vals >= promote_height_mm))
    high_ratio = high_pixels / max(float(vals.size), 1.0)

    # Promote on a robust top-face estimate, not a single spike. The p75/top35
    # tests handle dense centre-board readings; p90 plus a few high pixels helps
    # sparse edge/corner readings.
    is_two = (
        top35 >= promote_height_mm
        or p75 >= promote_height_mm
        or (p90 >= promote_height_mm and high_pixels >= max(2, int(0.05 * vals.size)))
        or high_ratio >= 0.12
    )

    # A genuinely low top band is one-high. Borderline values are handled by the
    # temporal smoother; without a smoother they remain conservative.
    if not is_two and top25 <= demote_height_mm:
        return 1, used
    return (2 if is_two else 1), used

def classify_rgb_candidates_by_depth(
    rgb_bgr: np.ndarray,
    depth_mm: np.ndarray,
    baseline: EmptyBoardBaseline,
    *,
    chip_thickness_mm: float = 10.0,
    min_piece_height_mm: float = 3.0,
    max_piece_height_mm: float = 45.0,
    height_stat: HeightStat = "top35",
    min_depth_pixels: int = 8,
    min_depth_ratio: float = 0.025,
    height_max_visual_mm: float = 30.0,
    min_radius_px: float = 7.0,
    max_radius_px: float = 30.0,
    expected_radius_px: Optional[float] = None,
    split_touching: bool = True,
    use_hough: bool = False,
    min_candidate_support_ratio: float = 0.025,
    min_support_pixels: int = 6,
    draw_rejected_candidates: bool = False,
    roi_frac: Optional[RoiFrac] = (0.18, 0.00, 0.88, 1.00),
    middle_exclusion_frac: Optional[Tuple[float, float, float, float]] = (0.00, 0.40, 1.00, 0.60),
    temporal_filter: Optional[TemporalMaskFilter] = None,
    stack_smoother: Optional[StackCountSmoother] = None,
    stack_promote_height_mm: float = 16.5,
    stack_demote_height_mm: float = 14.5,
    noise_margin_mm: float = 1.5,
) -> StackClassificationResult:
    height_mm, valid_overlap = baseline.height_above_board(depth_mm)

    if height_mm.shape[:2] != rgb_bgr.shape[:2]:
        height_for_rgb = cv2.resize(height_mm, (rgb_bgr.shape[1], rgb_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
        valid_for_rgb = cv2.resize(valid_overlap, (rgb_bgr.shape[1], rgb_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
        noise_floor = cv2.resize(baseline.noise_floor_mm, (rgb_bgr.shape[1], rgb_bgr.shape[0]), interpolation=cv2.INTER_NEAREST) if baseline.noise_floor_mm is not None else None
    else:
        height_for_rgb = height_mm
        valid_for_rgb = valid_overlap
        noise_floor = baseline.noise_floor_mm

    exclusion_fracs: RectFracs = tuple(frac for frac in (middle_exclusion_frac,) if frac is not None)

    raw_support = build_height_support_mask(
        height_for_rgb,
        valid_for_rgb,
        min_piece_height_mm=min_piece_height_mm,
        max_piece_height_mm=max_piece_height_mm,
        roi_frac=roi_frac,
        exclusion_fracs=exclusion_fracs,
        noise_floor_mm=noise_floor,
        noise_margin_mm=noise_margin_mm,
        min_area_px=max(3, min_support_pixels // 2),
    )
    stable_support = temporal_filter.update(raw_support) if temporal_filter is not None else raw_support

    candidates, restored_circle_mask, rgb_mask, proposed_count = generate_restored_circle_candidates(
        rgb_bgr,
        stable_support,
        valid_for_rgb.astype(np.uint8),
        min_radius_px=min_radius_px,
        max_radius_px=max_radius_px,
        expected_radius_px=expected_radius_px,
        min_candidate_support_ratio=min_candidate_support_ratio,
        min_support_pixels=min_support_pixels,
        use_hough=use_hough,
        split_touching=split_touching,
        roi_frac=roi_frac,
        exclusion_fracs=exclusion_fracs,
    )

    stat_getters = {
        "median": lambda v: float(np.median(v)),
        "p75": lambda v: float(np.percentile(v, 75)),
        "p90": lambda v: float(np.percentile(v, 90)),
        "max": lambda v: float(np.max(v)),
        "top35": lambda v: _top_band_stat(v, 0.35),
        "top25": lambda v: _top_band_stat(v, 0.25),
    }

    class_bgr = np.zeros((*rgb_bgr.shape[:2], 3), dtype=np.uint8)
    overlay = rgb_bgr.copy()
    classified: List[ChipCandidate] = []
    rejected = 0
    vals_min = max(1.0, min_piece_height_mm * 0.60)

    # First classify each candidate for this frame without drawing. Then apply
    # stack-count smoothing across frames. Drawing after smoothing ensures the
    # mask and overlay show the stable class, while labels still expose raw
    # values for debugging.
    for cand in candidates:
        vals, depth_ratio = _height_values_in_candidate(
            height_for_rgb,
            valid_for_rgb,
            cand,
            min_depth_pixels=min_depth_pixels,
            min_depth_ratio=min_depth_ratio,
        )
        cand.valid_depth_ratio = max(cand.valid_depth_ratio, depth_ratio)
        vals = vals[(vals >= vals_min) & (vals <= max_piece_height_mm)]

        if vals.size >= min_depth_pixels and depth_ratio >= min_depth_ratio:
            cand.height_median_mm = float(np.median(vals))
            cand.height_p75_mm = float(np.percentile(vals, 75))
            cand.height_p90_mm = float(np.percentile(vals, 90))
            cand.height_max_mm = float(np.max(vals))
            cand.height_top35_mm = _top_band_stat(vals, 0.35)
            cand.height_top25_mm = _top_band_stat(vals, 0.25)
            requested_height = stat_getters[height_stat](vals)
            cand.stack_count, cand.height_used_mm = _estimate_stack_1_or_2(
                vals,
                min_piece_height_mm=min_piece_height_mm,
                max_piece_height_mm=max_piece_height_mm,
                promote_height_mm=stack_promote_height_mm,
                demote_height_mm=stack_demote_height_mm,
                requested_height=requested_height,
            )
            cand.raw_stack_count = cand.stack_count
            cand.confidence = min(
                1.0,
                0.45 * min(1.0, depth_ratio / 0.12)
                + 0.35 * min(1.0, cand.support_pixels / max(float(min_support_pixels * 4), 1.0))
                + 0.20 * min(1.0, cand.support_ratio / max(min_candidate_support_ratio, 1e-6)),
            )
        else:
            cand.raw_stack_count = 0
            cand.stack_count = 0
            cand.confidence = 0.0

        if cand.stack_count <= 0:
            rejected += 1
        else:
            classified.append(cand)

    if stack_smoother is not None:
        classified = stack_smoother.update(classified)

    for cand in classified:
        cx, cy, r = int(round(cand.x)), int(round(cand.y)), max(2, int(round(cand.radius)))
        if cand.stack_count == 1:
            colour = (0, 220, 0)
            label = "1"
        else:
            colour = (0, 220, 255)
            label = "2"

        cv2.circle(class_bgr, (cx, cy), r, colour, -1)
        cv2.circle(overlay, (cx, cy), r, colour, 2)
        cv2.putText(overlay, label, (cx - 8, cy + 6), cv2.FONT_HERSHEY_SIMPLEX, 0.60, colour, 2, cv2.LINE_AA)
        raw_note = f"r{cand.raw_stack_count}" if cand.raw_stack_count and cand.raw_stack_count != cand.stack_count else ""
        track_note = f" T{cand.track_id}" if cand.track_id >= 0 else ""
        cv2.putText(
            overlay,
            f"{cand.height_used_mm:.1f}mm {raw_note}{track_note}",
            (cx - 34, cy + r + 13),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.34,
            colour,
            1,
            cv2.LINE_AA,
        )

    if draw_rejected_candidates:
        for cand in candidates:
            if cand.stack_count > 0:
                continue
            cx, cy, r = int(round(cand.x)), int(round(cand.y)), max(2, int(round(cand.radius)))
            cv2.circle(overlay, (cx, cy), r, (80, 80, 80), 1)
            cv2.putText(overlay, "?", (cx - 5, cy + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (120, 120, 120), 1, cv2.LINE_AA)

    filtered_circle_mask = np.zeros_like(restored_circle_mask)
    for cand in classified:
        cv2.circle(filtered_circle_mask, (int(round(cand.x)), int(round(cand.y))), max(2, int(round(cand.radius))), 255, -1)

    diagnostics = {
        "candidate_count": float(len(candidates)),
        "proposed_count": float(proposed_count),
        "classified_count": float(len(classified)),
        "rejected_count": float(rejected),
        "valid_overlap_ratio": float(np.count_nonzero(valid_for_rgb)) / float(valid_for_rgb.size),
        "raw_support_ratio": float(np.count_nonzero(raw_support)) / float(raw_support.size),
        "stable_support_ratio": float(np.count_nonzero(stable_support)) / float(stable_support.size),
        "rgb_restored_ratio": float(np.count_nonzero(filtered_circle_mask)) / float(filtered_circle_mask.size),
        "rgb_mask_ratio": float(np.count_nonzero(rgb_mask)) / float(rgb_mask.size),
        "hough_enabled": float(bool(use_hough)),
        "split_touching_enabled": float(bool(split_touching)),
        "min_candidate_support_ratio": float(min_candidate_support_ratio),
        "temporal_history": float(temporal_filter.history_size if temporal_filter is not None else 0),
        "stack_promote_height_mm": float(stack_promote_height_mm),
        "stack_demote_height_mm": float(stack_demote_height_mm),
    }

    return StackClassificationResult(
        candidates=classified,
        height_mm=height_for_rgb,
        valid_overlap_mask=valid_for_rgb.astype(np.uint8),
        stable_depth_support_mask=stable_support,
        rgb_restored_circle_mask=filtered_circle_mask,
        rgb_candidate_mask=rgb_mask,
        piece_presence_mask=stable_support,
        stack_class_bgr=class_bgr,
        overlay_bgr=overlay,
        diagnostics=diagnostics,
    )


## RGB-D piece detection / segmentation

Geometry-constrained grid/slot detector using `checker_detection_area` while excluding dice, cube and middle-strip regions.


In [ ]:
%%writefile Interpretation/piece_detection.py
from __future__ import annotations

from collections import Counter, deque
from dataclasses import dataclass
from typing import Deque, Dict, Iterable, List, Optional, Tuple

import cv2
import numpy as np

from backgammon_types import (
    NormalisedBoard,
    PieceDetectionResult,
    PieceInstance,
    RegionMasks,
    make_empty_region_counts,
)


@dataclass
class PieceDetectionConfig:
    # Height model. Chips are only expected to be either 1-high or 2-high.
    min_piece_height_mm: float = 3.0
    max_piece_height_mm: float = 35.0
    # Stack classification is deliberately binary: 1-high or 2-high only.
    # Promote/demote hysteresis avoids flickering around the boundary.
    stack_split_height_mm: float = 16.5  # backwards-compatible alias / default promote value
    stack_promote_height_mm: float = 16.5
    stack_demote_height_mm: float = 14.5
    chip_thickness_mm: float = 10.0
    max_stack_count: int = 2

    # Slot geometry. If expected_piece_radius_px is None, radius is estimated
    # from each point's mask width in the rectified board view.
    expected_piece_radius_px: Optional[float] = None
    slot_radius_from_point_width: float = 0.43
    slot_spacing_radius_mult: float = 1.75
    max_slots_per_point: int = 5

    # Evidence thresholds. Keep min_support_pixels permissive; temporal smoothing
    # should stabilise the result rather than filtering real edge/corner chips out.
    min_support_pixels: int = 6
    min_height_pixels: int = 6
    min_valid_depth_ratio: float = 0.01

    # Temporal support mask for depth flicker removal.
    temporal_support_window: int = 3
    temporal_support_required: int = 2

    # Slot-level hysteresis. This stops 2-high stacks flickering to 1-high when
    # passive stereo briefly under-reads the stack top.
    slot_history_window: int = 5
    slot_promote_votes: int = 2
    slot_demote_votes: int = 4
    slot_hold_misses: int = 1

    # Region policy.
    checker_detection_mask_name: str = "checker_detection_area"
    include_bar: bool = True
    include_bearoff: bool = False

    # Mask cleanup for the height support mask.
    morph_open_ksize: int = 3
    morph_close_ksize: int = 5


class RGBDPieceDetector:
    """
    Geometry-constrained RGB-D checker detector.

    The detector no longer searches for circles globally. It only samples legal
    checker slots inside the board's point masks (plus the bar if enabled), uses
    a temporally stable height-above-board support mask, and classifies each
    occupied slot as either a 1-high chip or a 2-high stack.

    This remains a classical baseline. The slot decision could be swapped for
    a learned classifier later if the dataset supports it.
    """

    def __init__(self, config: Optional[PieceDetectionConfig] = None) -> None:
        self.config = config or PieceDetectionConfig()
        self._support_history: Deque[np.ndarray] = deque(maxlen=self.config.temporal_support_window)
        self._slot_histories: Dict[Tuple[str, int], Deque[int]] = {}
        self._slot_stable: Dict[Tuple[str, int], int] = {}
        self._slot_misses: Dict[Tuple[str, int], int] = {}

    @staticmethod
    def _mean_lab(rgb_bgr: np.ndarray, mask: np.ndarray) -> np.ndarray:
        lab = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2LAB)
        pixels = lab[mask > 0]
        if len(pixels) == 0:
            return np.array([0.0, 0.0, 0.0], dtype=np.float32)
        return pixels.mean(axis=0).astype(np.float32)

    @staticmethod
    def _circle_mask(shape_hw: Tuple[int, int], x: float, y: float, r: float) -> np.ndarray:
        mask = np.zeros(shape_hw, dtype=np.uint8)
        cv2.circle(mask, (int(round(x)), int(round(y))), int(round(max(1.0, r))), 1, -1)
        return mask

    @staticmethod
    def _circle_contour(x: float, y: float, r: float) -> np.ndarray:
        pts = []
        for a in np.linspace(0, 2 * np.pi, 48, endpoint=False):
            pts.append([int(round(x + np.cos(a) * r)), int(round(y + np.sin(a) * r))])
        return np.array(pts, dtype=np.int32).reshape((-1, 1, 2))

    def _region_names_to_process(self, regions: RegionMasks) -> List[str]:
        names = list(regions.point_names)
        if self.config.include_bar and "bar" in regions.masks:
            names.append("bar")
        if self.config.include_bearoff:
            for name in ("bearoff_left", "bearoff_right"):
                if name in regions.masks:
                    names.append(name)
        return names

    def _checker_area_mask(self, shape_hw: Tuple[int, int], regions: RegionMasks) -> np.ndarray:
        if self.config.checker_detection_mask_name in regions.masks:
            return regions.masks[self.config.checker_detection_mask_name].astype(np.uint8)
        out = np.zeros(shape_hw, dtype=np.uint8)
        for name in self._region_names_to_process(regions):
            out = cv2.bitwise_or(out, regions.masks[name].astype(np.uint8))
        return out

    def _raw_support_mask(self, board: NormalisedBoard, checker_area: np.ndarray) -> np.ndarray:
        cfg = self.config
        raw = (
            (board.height_map_mm >= cfg.min_piece_height_mm)
            & (board.height_map_mm <= cfg.max_piece_height_mm)
            & (board.valid_mask > 0)
            & (checker_area > 0)
        ).astype(np.uint8)
        if cfg.morph_open_ksize > 1:
            k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (cfg.morph_open_ksize, cfg.morph_open_ksize))
            raw = cv2.morphologyEx(raw, cv2.MORPH_OPEN, k)
        if cfg.morph_close_ksize > 1:
            k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (cfg.morph_close_ksize, cfg.morph_close_ksize))
            raw = cv2.morphologyEx(raw, cv2.MORPH_CLOSE, k)
        return raw.astype(np.uint8)

    def _stable_support_mask(self, raw: np.ndarray) -> np.ndarray:
        self._support_history.append(raw.copy())
        stack = np.stack(list(self._support_history), axis=0)
        required = min(self.config.temporal_support_required, len(self._support_history))
        return (stack.sum(axis=0) >= required).astype(np.uint8)

    def _slot_radius(self, region_mask: np.ndarray, image_width: int) -> float:
        cfg = self.config
        if cfg.expected_piece_radius_px is not None:
            return float(cfg.expected_piece_radius_px)
        ys, xs = np.where(region_mask > 0)
        if xs.size == 0:
            return max(5.0, 0.025 * image_width)
        width = float(xs.max() - xs.min() + 1)
        return float(np.clip(width * cfg.slot_radius_from_point_width, 5.0, 0.055 * image_width))

    def _slot_centres_for_point(self, region_name: str, region_mask: np.ndarray, shape_hw: Tuple[int, int]) -> List[Tuple[float, float, float]]:
        h, w = shape_hw
        ys, xs = np.where(region_mask > 0)
        if xs.size == 0:
            return []
        cx = float(np.mean(xs))
        radius = self._slot_radius(region_mask, w)
        spacing = radius * self.config.slot_spacing_radius_mult
        is_top = float(np.mean(ys)) < h / 2.0
        base_y = float(ys.min()) if is_top else float(ys.max())

        centres: List[Tuple[float, float, float]] = []
        for slot_idx in range(self.config.max_slots_per_point):
            if is_top:
                cy = base_y + radius + slot_idx * spacing
            else:
                cy = base_y - radius - slot_idx * spacing
            if cy < radius * 0.25 or cy > h - radius * 0.25:
                break
            centres.append((cx, cy, radius))
        return centres

    def _slot_centres_for_rect_region(self, region_mask: np.ndarray, shape_hw: Tuple[int, int]) -> List[Tuple[float, float, float]]:
        # Bar fallback. This is intentionally simpler than points because bar use
        # is secondary in the current pipeline.
        h, w = shape_hw
        ys, xs = np.where(region_mask > 0)
        if xs.size == 0:
            return []
        radius = self._slot_radius(region_mask, w)
        cx = float(np.mean(xs))
        y0, y1 = float(ys.min()), float(ys.max())
        centres = []
        y = y0 + radius
        idx = 0
        while y <= y1 - radius and idx < self.config.max_slots_per_point * 2:
            centres.append((cx, y, radius))
            y += radius * self.config.slot_spacing_radius_mult
            idx += 1
        return centres

    def _height_for_slot(self, board: NormalisedBoard, slot_mask: np.ndarray) -> Tuple[float, int, int]:
        vals = board.height_map_mm[(slot_mask > 0) & (board.valid_mask > 0)]
        vals = vals[(vals >= self.config.min_piece_height_mm * 0.5) & (vals <= self.config.max_piece_height_mm)]
        if vals.size == 0:
            return 0.0, 0, 0
        # Use an upper-band statistic to estimate the top face of a chip/stack,
        # but avoid using a single noisy maximum.
        threshold = np.percentile(vals, 65)
        top_vals = vals[vals >= threshold]
        if top_vals.size < 3:
            top_vals = vals
        height = float(np.median(top_vals))
        high_pixels = int(np.count_nonzero(vals >= self.config.stack_promote_height_mm))
        return height, int(vals.size), high_pixels

    def _smooth_slot_count(self, key: Tuple[str, int], raw_count: int, height_mm: float) -> int:
        cfg = self.config
        hist = self._slot_histories.setdefault(key, deque(maxlen=cfg.slot_history_window))
        # Store the raw count, but encode absence as 0. Height evidence is used
        # directly below so a one-frame under-read does not demote a true stack.
        hist.append(int(raw_count))
        counts = Counter(hist)
        prev = self._slot_stable.get(key, 0)

        # Track recent height evidence in a companion deque stored under a
        # sentinel key. This keeps the public structures simple while allowing
        # promote/demote hysteresis by actual mm values.
        height_key = (key[0], key[1] + 10_000)
        height_hist = self._slot_histories.setdefault(height_key, deque(maxlen=cfg.slot_history_window))
        if raw_count > 0:
            height_hist.append(2 if height_mm >= cfg.stack_promote_height_mm else 1 if height_mm <= cfg.stack_demote_height_mm else raw_count)
        else:
            height_hist.append(0)
        hcounts = Counter(height_hist)

        if prev == 0:
            if hcounts[2] >= cfg.slot_promote_votes or counts[2] >= cfg.slot_promote_votes:
                new = 2
            elif counts[1] >= cfg.slot_promote_votes:
                new = 1
            else:
                new = 0
        elif prev == 1:
            if hcounts[2] >= cfg.slot_promote_votes or counts[2] >= cfg.slot_promote_votes:
                new = 2
            elif counts[0] >= cfg.slot_demote_votes:
                new = 0
            else:
                new = 1
        else:  # prev == 2
            # A two-stack only drops to one-high after repeated low-height
            # evidence. This is the key fix for green/yellow flicker.
            if counts[0] >= cfg.slot_demote_votes:
                new = 0
            elif hcounts[1] >= cfg.slot_demote_votes and counts[1] >= max(1, cfg.slot_demote_votes - 1):
                new = 1
            else:
                new = 2

        self._slot_stable[key] = int(new)
        return int(new)

    def _raw_slot_count(self, board: NormalisedBoard, slot_mask: np.ndarray, support_pixels: int) -> Tuple[int, float, int, int]:
        height_mm, height_pixels, high_pixels = self._height_for_slot(board, slot_mask)
        if support_pixels < self.config.min_support_pixels and height_pixels < self.config.min_height_pixels:
            return 0, height_mm, height_pixels, high_pixels
        if height_mm <= 0:
            return 0, height_mm, height_pixels, high_pixels
        # Binary 1-vs-2 classification. Use a lower promote threshold than the
        # ideal 20 mm because passive stereo often under-reads stack height,
        # especially near edges. Temporal hysteresis handles borderline values.
        strong_high = high_pixels >= max(2, self.config.min_support_pixels // 2)
        raw_count = 2 if (height_mm >= self.config.stack_promote_height_mm or strong_high) else 1
        raw_count = int(np.clip(raw_count, 1, self.config.max_stack_count))
        return raw_count, height_mm, height_pixels, high_pixels

    def detect(self, board: NormalisedBoard, regions: RegionMasks) -> PieceDetectionResult:
        h, w = board.rgb_bgr.shape[:2]
        overlay = board.rgb_bgr.copy()
        region_order = self._region_names_to_process(regions)
        region_counts = make_empty_region_counts(regions.point_names + regions.auxiliary_names)
        pieces: List[PieceInstance] = []
        raw_records = []

        checker_area = self._checker_area_mask((h, w), regions)
        raw_support = self._raw_support_mask(board, checker_area)
        stable_support = self._stable_support_mask(raw_support)
        slot_candidate_mask = np.zeros((h, w), dtype=np.uint8)
        stack_class_bgr = np.zeros((h, w, 3), dtype=np.uint8)

        for region_name in region_order:
            if region_name not in regions.masks:
                continue
            region_mask = cv2.bitwise_and(regions.masks[region_name].astype(np.uint8), checker_area)
            if np.count_nonzero(region_mask) == 0:
                continue
            if region_name.startswith("point_"):
                centres = self._slot_centres_for_point(region_name, region_mask, (h, w))
            else:
                centres = self._slot_centres_for_rect_region(region_mask, (h, w))

            for slot_idx, (cx, cy, radius) in enumerate(centres):
                full_slot_mask = self._circle_mask((h, w), cx, cy, radius)
                slot_mask = cv2.bitwise_and(full_slot_mask, checker_area)
                support_pixels = int(np.count_nonzero((stable_support > 0) & (slot_mask > 0)))
                raw_count, height_mm, height_pixels, high_pixels = self._raw_slot_count(board, slot_mask, support_pixels)
                stable_count = self._smooth_slot_count((region_name, slot_idx), raw_count, height_mm)
                if stable_count <= 0:
                    continue

                slot_candidate_mask[slot_mask > 0] = 255
                colour = (0, 255, 0) if stable_count == 1 else (0, 220, 255)
                stack_class_bgr[slot_mask > 0] = colour
                mean_lab = self._mean_lab(board.rgb_bgr, slot_mask)
                raw_records.append(
                    {
                        "region_name": region_name,
                        "slot_idx": slot_idx,
                        "centroid_xy": (cx, cy),
                        "radius_px": radius,
                        "area_px": float(np.count_nonzero(slot_mask)),
                        "height_mm": height_mm,
                        "stack_count": stable_count,
                        "raw_count": raw_count,
                        "support_pixels": support_pixels,
                        "height_pixels": height_pixels,
                        "high_pixels": high_pixels,
                        "mean_lab": mean_lab,
                        "mask": slot_mask,
                        "contour": self._circle_contour(cx, cy, radius),
                    }
                )

        # Assign light/dark using LAB lightness over detected slots. This works
        # best once non-checker areas have already been removed by RegionMasks.
        if raw_records:
            lightness = np.float32([rec["mean_lab"][0] for rec in raw_records]).reshape(-1, 1)
            if len(raw_records) >= 2:
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1)
                _, labels, centers = cv2.kmeans(lightness, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                centers = centers.ravel()
                light_cluster = int(np.argmax(centers))
                colour_labels = ["light" if int(label[0]) == light_cluster else "dark" for label in labels]
            else:
                colour_labels = ["light" if float(lightness[0, 0]) >= 128.0 else "dark"]

            for rec, colour_name in zip(raw_records, colour_labels):
                confidence = float(np.clip(0.45 + 0.05 * rec["stack_count"] + 0.02 * min(rec["support_pixels"], 10), 0.0, 1.0))
                piece = PieceInstance(
                    region_name=rec["region_name"],
                    colour_name=colour_name,
                    centroid_xy=rec["centroid_xy"],
                    area_px=rec["area_px"],
                    radius_px=rec["radius_px"],
                    height_mm=rec["height_mm"],
                    stack_count=rec["stack_count"],
                    confidence=confidence,
                    contour=rec["contour"],
                )
                pieces.append(piece)
                if piece.region_name in region_counts:
                    region_counts[piece.region_name][piece.colour_name] += piece.stack_count

                draw_colour = (40, 255, 40) if colour_name == "light" else (40, 40, 255)
                cv2.drawContours(overlay, [piece.contour], -1, draw_colour, 2)
                label = f"{piece.region_name}:{colour_name[0]}x{piece.stack_count}"
                cv2.putText(
                    overlay,
                    label,
                    (int(piece.centroid_xy[0] - piece.radius_px), int(piece.centroid_xy[1])),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.36,
                    draw_colour,
                    1,
                    cv2.LINE_AA,
                )

        confidence = float(np.mean([p.confidence for p in pieces])) if pieces else 0.0
        return PieceDetectionResult(
            pieces=pieces,
            region_counts=region_counts,
            overlay_bgr=overlay,
            confidence=confidence,
            stable_depth_support_mask=(stable_support * 255).astype(np.uint8),
            slot_candidate_mask=slot_candidate_mask,
            checker_detection_area_mask=(checker_area * 255).astype(np.uint8),
            stack_class_bgr=stack_class_bgr,
            debug={
                "raw_support_ratio": float(np.mean(raw_support)),
                "stable_support_ratio": float(np.mean(stable_support)),
                "piece_count": len(pieces),
                "config": self.config.__dict__.copy(),
            },
        )


__all__ = ["PieceDetectionConfig", "RGBDPieceDetector"]


## Dice and cube detection / reading

Reads dice and cube candidates inside `dice_area` and `cube_area` when region masks are available.


In [ ]:
%%writefile Interpretation/dice_cube_reader.py
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Optional, Tuple

import cv2
import numpy as np

from backgammon_types import CubeObservation, DiceCubeObservation, DiceObservation, NormalisedBoard, RegionMasks


try:
    import pytesseract  # type: ignore
except Exception:  # pragma: no cover - optional dependency
    pytesseract = None


@dataclass
class DiceCubeConfig:
    min_square_area_px: int = 150
    max_square_area_px: int = 20000
    pip_min_area_px: int = 5
    pip_max_area_px: int = 500
    cube_min_area_px: int = 250
    cube_max_area_px: int = 30000


class DiceCubeReader:
    """
    Classical dice / doubling-cube observation step.

    Dice and cube searches can be restricted to RegionMasks['dice_area'] and
    RegionMasks['cube_area'] when those masks are available. This keeps checker detection and dice/cube reading from using the same pixels.
    """

    def __init__(self, config: Optional[DiceCubeConfig] = None) -> None:
        self.config = config or DiceCubeConfig()

    @staticmethod
    def _aspect_ratio_ok(w: int, h: int, tol: float = 0.35) -> bool:
        ratio = w / max(h, 1)
        return abs(ratio - 1.0) <= tol

    @staticmethod
    def _masked_gray(gray: np.ndarray, mask: Optional[np.ndarray]) -> np.ndarray:
        if mask is None or np.count_nonzero(mask) == 0:
            return gray
        return cv2.bitwise_and(gray, gray, mask=(mask.astype(np.uint8) * 255))

    @staticmethod
    def _box_center_in_mask(box: Tuple[int, int, int, int], mask: Optional[np.ndarray]) -> bool:
        if mask is None or np.count_nonzero(mask) == 0:
            return True
        x1, y1, x2, y2 = box
        cx = int(round((x1 + x2) / 2))
        cy = int(round((y1 + y2) / 2))
        h, w = mask.shape[:2]
        if cx < 0 or cy < 0 or cx >= w or cy >= h:
            return False
        return bool(mask[cy, cx] > 0)

    def _find_square_candidates(self, gray: np.ndarray) -> List[Tuple[int, int, int, int]]:
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 150)
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boxes = []
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < self.config.min_square_area_px or area > self.config.max_square_area_px:
                continue
            x, y, w, h = cv2.boundingRect(contour)
            if not self._aspect_ratio_ok(w, h):
                continue
            boxes.append((x, y, x + w, y + h))
        return boxes

    def _count_pips(self, patch_bgr: np.ndarray) -> Tuple[Optional[int], float]:
        gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (3, 3), 0)
        _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        pip_count = 0
        for contour in contours:
            area = cv2.contourArea(contour)
            if self.config.pip_min_area_px <= area <= self.config.pip_max_area_px:
                pip_count += 1
        if 1 <= pip_count <= 6:
            return pip_count, min(1.0, 0.3 + 0.1 * pip_count)
        return None, 0.0

    def _read_cube_value(self, patch_bgr: np.ndarray) -> Tuple[Optional[int], float, Optional[str]]:
        gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, None, fx=3.0, fy=3.0, interpolation=cv2.INTER_CUBIC)
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        if pytesseract is None:
            return None, 0.0, None
        text = pytesseract.image_to_string(thresh, config="--psm 8 -c tessedit_char_whitelist=012468")
        text = "".join(ch for ch in text if ch.isdigit())
        if not text:
            return None, 0.0, None
        try:
            value = int(text)
        except ValueError:
            return None, 0.0, text
        if value in {2, 4, 8, 16, 32, 64}:
            return value, 0.8, text
        return None, 0.2, text

    def read(self, board: NormalisedBoard, regions: Optional[RegionMasks] = None) -> DiceCubeObservation:
        overlay = board.rgb_bgr.copy()
        dice_mask = regions.masks.get("dice_area") if regions is not None else None
        cube_mask = regions.masks.get("cube_area") if regions is not None else None

        dice_search = self._masked_gray(board.rgb_gray, dice_mask)
        cube_search = self._masked_gray(board.rgb_gray, cube_mask)
        dice_boxes = [b for b in self._find_square_candidates(dice_search) if self._box_center_in_mask(b, dice_mask)]
        cube_boxes = [b for b in self._find_square_candidates(cube_search) if self._box_center_in_mask(b, cube_mask)]

        dice: List[DiceObservation] = []
        cube: Optional[CubeObservation] = None

        for x1, y1, x2, y2 in dice_boxes:
            patch = board.rgb_bgr[y1:y2, x1:x2]
            if patch.size == 0:
                continue
            dice_value, dice_conf = self._count_pips(patch)
            if dice_value is not None:
                obs = DiceObservation((x1, y1, x2, y2), dice_value, dice_conf)
                dice.append(obs)
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (40, 255, 40), 2)
                cv2.putText(overlay, f"d{dice_value}", (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (40, 255, 40), 1, cv2.LINE_AA)

        dice = sorted(dice, key=lambda d: d.confidence, reverse=True)[:2]

        for x1, y1, x2, y2 in cube_boxes:
            patch = board.rgb_bgr[y1:y2, x1:x2]
            if patch.size == 0:
                continue
            area = (x2 - x1) * (y2 - y1)
            if not (self.config.cube_min_area_px <= area <= self.config.cube_max_area_px):
                continue
            cube_value, cube_conf, cube_text = self._read_cube_value(patch)
            cube = CubeObservation((x1, y1, x2, y2), cube_value, cube_conf, cube_text)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (255, 100, 0), 2)
            cv2.putText(overlay, f"cube:{cube_value if cube_value is not None else '?'}", (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 100, 0), 1, cv2.LINE_AA)
            break

        confidence_parts = [d.confidence for d in dice]
        if cube is not None:
            confidence_parts.append(cube.confidence)
        return DiceCubeObservation(
            dice=dice,
            cube=cube,
            overlay_bgr=overlay,
            confidence=float(np.mean(confidence_parts)) if confidence_parts else 0.0,
        )


__all__ = ["DiceCubeConfig", "DiceCubeReader"]


## Temporal state estimator

Fuses board-state predictions over a sliding window.


In [ ]:
%%writefile Interpretation/temporal_state_estimator.py
from __future__ import annotations

from collections import Counter, deque
from dataclasses import dataclass
from typing import Deque, Dict, Iterable, List, Optional, Tuple

from backgammon_types import BoardState, TemporalEstimate


def _mode_or_default(values: Iterable[Optional[int]], default: Optional[int] = None) -> Optional[int]:
    filtered = [v for v in values if v is not None]
    if not filtered:
        return default
    return Counter(filtered).most_common(1)[0][0]


def _state_signature(state: BoardState) -> Tuple:
    region_items = tuple(
        sorted((region, counts.get("light", 0), counts.get("dark", 0)) for region, counts in state.region_counts.items())
    )
    return region_items, state.dice, state.cube_value


class TemporalStateEstimator:
    """
    Fuses candidate board states over a short rolling window.

    The fusion strategy is intentionally simple: mode/median-like consensus over
    recent keyframes. This makes it easy to reason about the output during the
    data-collection phase and replace later with a learned temporal model.
    """

    def __init__(self, window_size: int = 5) -> None:
        self.window_size = max(1, window_size)
        self.history: Deque[BoardState] = deque(maxlen=self.window_size)
        self.last_committed_state: Optional[BoardState] = None

    def _fuse_region_counts(self) -> Dict[str, Dict[str, int]]:
        region_names = sorted({name for state in self.history for name in state.region_counts.keys()})
        fused: Dict[str, Dict[str, int]] = {}

        for region_name in region_names:
            light_values = [state.region_counts.get(region_name, {}).get("light", 0) for state in self.history]
            dark_values = [state.region_counts.get(region_name, {}).get("dark", 0) for state in self.history]

            light_mode = Counter(light_values).most_common(1)[0][0] if light_values else 0
            dark_mode = Counter(dark_values).most_common(1)[0][0] if dark_values else 0
            fused[region_name] = {"light": int(light_mode), "dark": int(dark_mode)}

        return fused

    def update(self, candidate_state: BoardState) -> TemporalEstimate:
        self.history.append(candidate_state)

        fused = BoardState(
            region_counts=self._fuse_region_counts(),
            dice=(
                _mode_or_default([s.dice[0] for s in self.history]),
                _mode_or_default([s.dice[1] for s in self.history]),
            ),
            cube_value=_mode_or_default([s.cube_value for s in self.history]),
            confidence=sum(s.confidence for s in self.history) / max(len(self.history), 1),
            timestamp=candidate_state.timestamp,
            metadata={"history_size": len(self.history)},
        )

        changed = True
        if self.last_committed_state is not None:
            changed = _state_signature(fused) != _state_signature(self.last_committed_state)

        return TemporalEstimate(
            candidate_state=candidate_state,
            fused_state=fused,
            changed_vs_last_commit=changed,
            history_size=len(self.history),
        )

    def commit(self, state: BoardState) -> None:
        self.last_committed_state = state


## Rule-based validation

Checks plausibility of fused board states.


In [ ]:
%%writefile Interpretation/rule_validation.py
from __future__ import annotations

from typing import Optional

from backgammon_types import BoardState, ValidationReport


class BoardStateValidator:
    """
    Rule-based validation pass for the fused board state.
    """

    def __init__(self, expected_checkers_per_colour: int = 15, require_exact_totals: bool = False) -> None:
        self.expected_checkers_per_colour = expected_checkers_per_colour
        self.require_exact_totals = require_exact_totals

    @staticmethod
    def _is_power_of_two(value: int) -> bool:
        return value > 0 and (value & (value - 1)) == 0

    def validate(self, state: BoardState) -> ValidationReport:
        errors = []
        warnings = []

        total_light = 0
        total_dark = 0

        for region_name, counts in state.region_counts.items():
            light = int(counts.get("light", 0))
            dark = int(counts.get("dark", 0))

            if light < 0 or dark < 0:
                errors.append(f"Negative checker count in {region_name}.")

            if light > 0 and dark > 0:
                errors.append(f"Both colours occupy {region_name}, which is not a legal settled point state.")

            total_light += light
            total_dark += dark

        if self.require_exact_totals:
            if total_light != self.expected_checkers_per_colour:
                errors.append(f"Light total is {total_light}, expected {self.expected_checkers_per_colour}.")
            if total_dark != self.expected_checkers_per_colour:
                errors.append(f"Dark total is {total_dark}, expected {self.expected_checkers_per_colour}.")
        else:
            if total_light > self.expected_checkers_per_colour:
                errors.append(f"Light total exceeds {self.expected_checkers_per_colour}.")
            if total_dark > self.expected_checkers_per_colour:
                errors.append(f"Dark total exceeds {self.expected_checkers_per_colour}.")
            if total_light < self.expected_checkers_per_colour:
                warnings.append(f"Light total below {self.expected_checkers_per_colour}; detection may be incomplete.")
            if total_dark < self.expected_checkers_per_colour:
                warnings.append(f"Dark total below {self.expected_checkers_per_colour}; detection may be incomplete.")

        for die_value in state.dice:
            if die_value is not None and not (1 <= die_value <= 6):
                errors.append(f"Invalid die value {die_value}.")

        if state.cube_value is not None and not self._is_power_of_two(state.cube_value):
            errors.append(f"Cube value {state.cube_value} is not a power of two.")

        confidence = max(0.0, 1.0 - 0.15 * len(errors) - 0.05 * len(warnings))
        return ValidationReport(
            valid=len(errors) == 0,
            errors=errors,
            warnings=warnings,
            confidence=confidence,
        )


## Lightweight event inference


In [ ]:
%%writefile Interpretation/event_inference.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import time

from backgammon_types import BoardState, StateEvent


@dataclass
class EventInferenceConfig:
    emit_initial_state: bool = True
    emit_region_count_changes: bool = True
    emit_dice_changes: bool = True
    emit_cube_changes: bool = True


class BoardStateEventInferer:
    """
    Lightweight, non-CNN event inference layer.

    This does not attempt full legal move reconstruction yet. It turns changes
    between committed BoardState objects into coarse events that can be logged
    and inspected during experiments:
      - initial_state
      - region_count_changed
      - dice_changed
      - cube_changed

    This can be extended later with a legal
    move/cube-action inference engine.
    """

    def __init__(self, config: Optional[EventInferenceConfig] = None) -> None:
        self.config = config or EventInferenceConfig()
        self.previous_state: Optional[BoardState] = None

    @staticmethod
    def _normalise_counts(counts: Dict[str, Dict[str, int]]) -> Dict[str, Dict[str, int]]:
        out: Dict[str, Dict[str, int]] = {}
        for region, colours in counts.items():
            out[region] = {
                "light": int(colours.get("light", 0)),
                "dark": int(colours.get("dark", 0)),
            }
        return out

    @staticmethod
    def _region_count_deltas(previous: BoardState, current: BoardState) -> Dict[str, Dict[str, int]]:
        prev = BoardStateEventInferer._normalise_counts(previous.region_counts)
        cur = BoardStateEventInferer._normalise_counts(current.region_counts)
        regions = sorted(set(prev) | set(cur))
        deltas: Dict[str, Dict[str, int]] = {}
        for region in regions:
            light_delta = cur.get(region, {}).get("light", 0) - prev.get(region, {}).get("light", 0)
            dark_delta = cur.get(region, {}).get("dark", 0) - prev.get(region, {}).get("dark", 0)
            if light_delta or dark_delta:
                deltas[region] = {"light": light_delta, "dark": dark_delta}
        return deltas

    def infer(self, previous: Optional[BoardState], current: BoardState) -> List[StateEvent]:
        now = current.timestamp or time.time()
        events: List[StateEvent] = []

        if previous is None:
            if self.config.emit_initial_state:
                events.append(
                    StateEvent(
                        event_type="initial_state",
                        description="Initial committed board state observed.",
                        timestamp=now,
                        confidence=current.confidence,
                        payload={
                            "region_counts": current.region_counts,
                            "dice": list(current.dice),
                            "cube_value": current.cube_value,
                        },
                    )
                )
            return events

        if self.config.emit_region_count_changes:
            deltas = self._region_count_deltas(previous, current)
            if deltas:
                events.append(
                    StateEvent(
                        event_type="region_count_changed",
                        description="One or more checker region counts changed.",
                        timestamp=now,
                        confidence=current.confidence,
                        payload={"deltas": deltas},
                    )
                )

        if self.config.emit_dice_changes and tuple(previous.dice) != tuple(current.dice):
            events.append(
                StateEvent(
                    event_type="dice_changed",
                    description=f"Dice changed from {previous.dice} to {current.dice}.",
                    timestamp=now,
                    confidence=current.confidence,
                    payload={"previous": list(previous.dice), "current": list(current.dice)},
                )
            )

        if self.config.emit_cube_changes and previous.cube_value != current.cube_value:
            events.append(
                StateEvent(
                    event_type="cube_changed",
                    description=f"Cube value changed from {previous.cube_value} to {current.cube_value}.",
                    timestamp=now,
                    confidence=current.confidence,
                    payload={"previous": previous.cube_value, "current": current.cube_value},
                )
            )

        return events

    def update(self, current: BoardState) -> List[StateEvent]:
        events = self.infer(self.previous_state, current)
        self.previous_state = current
        return events

    def reset(self) -> None:
        self.previous_state = None


__all__ = ["EventInferenceConfig", "BoardStateEventInferer"]


## End-to-end board-state pipeline

Runs registration, rectification, normalisation, segmentation, detection, temporal fusion and validation, then returns a `PipelineResult`.


In [ ]:
%%writefile Interpretation/board_state_pipeline.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional
import time

from backgammon_types import BoardState, PipelineResult
from board_registration import BoardRegistrar
from perspective_rectification import PerspectiveRectifier
from image_normalisation import BoardNormaliser
from point_tray_segmentation import PointTraySegmenter
from piece_detection import RGBDPieceDetector
from dice_cube_reader import DiceCubeReader
from temporal_state_estimator import TemporalStateEstimator
from rule_validation import BoardStateValidator
from event_inference import BoardStateEventInferer


@dataclass
class PipelineConfig:
    commit_threshold: float = 0.55


class BoardStatePipeline:
    """
    Runs the board-state pipeline from frame/keyframe input through to a pipeline result.

    The caller should pass either a raw RGB-D frame or an accepted KeyframePacket.
    This class returns a PipelineResult containing both intermediate artifacts and
    the final temporal/validation decision for that frame.
    """

    def __init__(self, config: Optional[PipelineConfig] = None) -> None:
        self.config = config or PipelineConfig()
        self.registrar = BoardRegistrar()
        self.rectifier = PerspectiveRectifier()
        self.normaliser = BoardNormaliser()
        self.segmenter = PointTraySegmenter()
        self.piece_detector = RGBDPieceDetector()
        self.dice_cube_reader = DiceCubeReader()
        self.temporal = TemporalStateEstimator(window_size=5)
        self.validator = BoardStateValidator(require_exact_totals=False)
        self.event_inferer = BoardStateEventInferer()

    @staticmethod
    def _candidate_state_from_results(piece_result, dice_cube_result) -> BoardState:
        dice_values = [obs.value for obs in dice_cube_result.dice][:2]
        while len(dice_values) < 2:
            dice_values.append(None)
        cube_value = dice_cube_result.cube.value if dice_cube_result.cube is not None else None
        confidence_parts = [piece_result.confidence, dice_cube_result.confidence]
        confidence = sum(confidence_parts) / max(len(confidence_parts), 1)
        return BoardState(
            region_counts=piece_result.region_counts,
            dice=(dice_values[0], dice_values[1]),
            cube_value=cube_value,
            confidence=confidence,
            timestamp=time.time(),
        )

    def process_frame(self, rgb_bgr, depth_mm) -> PipelineResult:
        board_lock = self.registrar.update(rgb_bgr)
        if not board_lock.valid:
            return PipelineResult(
                board_lock=board_lock,
                rectified=None,
                normalised=None,
                regions=None,
                pieces=None,
                dice_cube=None,
                temporal=None,
                validation=None,
                committed=False,
                state_changed=False,
                debug={"stage": "board_registration"},
            )

        rectified = self.rectifier.rectify(rgb_bgr, depth_mm, board_lock)
        normalised = self.normaliser.normalize(rectified)
        regions = self.segmenter.segment(normalised.rgb_bgr.shape[:2])

        # Geometry-constrained interpretation. Piece detection uses RegionMasks
        # to avoid dice/cube/bear-off areas. Dice/cube reading also uses manual
        # dice_area/cube_area masks when they are present.
        pieces = self.piece_detector.detect(normalised, regions)
        dice_cube = self.dice_cube_reader.read(normalised, regions)

        candidate_state = self._candidate_state_from_results(pieces, dice_cube)
        temporal_estimate = self.temporal.update(candidate_state)
        validation = self.validator.validate(temporal_estimate.fused_state)

        committed = bool(validation.valid and temporal_estimate.fused_state.confidence >= self.config.commit_threshold)
        events = []
        if committed:
            events = self.event_inferer.update(temporal_estimate.fused_state)
            self.temporal.commit(temporal_estimate.fused_state)

        return PipelineResult(
            board_lock=board_lock,
            rectified=rectified,
            normalised=normalised,
            regions=regions,
            pieces=pieces,
            dice_cube=dice_cube,
            temporal=temporal_estimate,
            validation=validation,
            committed=committed,
            state_changed=temporal_estimate.changed_vs_last_commit,
            debug={"stage": "complete"},
            events=events,
        )

    def process_keyframe_packet(self, packet) -> PipelineResult:
        return self.process_frame(packet.left_raw, packet.depth_raw)


__all__ = ["PipelineConfig", "BoardStatePipeline"]


# 5. Evaluation and logging utilities


## Pipeline result/event logger


In [ ]:
%%writefile Evaluation/pipeline_result_logger.py
from __future__ import annotations

from dataclasses import asdict, is_dataclass, dataclass
from pathlib import Path
from typing import Any, Dict, Optional
import json
import time

import numpy as np

from backgammon_types import PipelineResult, StateEvent


@dataclass
class PipelineLogConfig:
    root_dir: str = "pipeline_logs"
    session_name: Optional[str] = None
    write_intermediates_summary: bool = True


class PipelineResultLogger:
    """Writes PipelineResult summaries and StateEvent records to JSONL files."""

    def __init__(self, config: Optional[PipelineLogConfig] = None) -> None:
        self.config = config or PipelineLogConfig()
        session = self.config.session_name or time.strftime("pipeline_%Y%m%d_%H%M%S")
        self.session_dir = Path(self.config.root_dir) / session
        self.session_dir.mkdir(parents=True, exist_ok=True)
        self.results_path = self.session_dir / "pipeline_results.jsonl"
        self.events_path = self.session_dir / "events.jsonl"
        self.session_path = self.session_dir / "session.json"
        self._results_file = self.results_path.open("a", encoding="utf-8")
        self._events_file = self.events_path.open("a", encoding="utf-8")
        self.session_path.write_text(json.dumps({"session": session, "created": time.time()}, indent=2), encoding="utf-8")

    def close(self) -> None:
        self._results_file.close()
        self._events_file.close()

    def __enter__(self) -> "PipelineResultLogger":
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        self.close()

    @staticmethod
    def _safe(value: Any) -> Any:
        if isinstance(value, np.ndarray):
            return {"type": "ndarray", "shape": list(value.shape), "dtype": str(value.dtype)}
        if is_dataclass(value):
            return PipelineResultLogger._safe(asdict(value))
        if isinstance(value, dict):
            return {str(k): PipelineResultLogger._safe(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [PipelineResultLogger._safe(v) for v in value]
        if isinstance(value, (str, int, float, bool)) or value is None:
            return value
        return str(value)

    @staticmethod
    def result_summary(result: PipelineResult) -> Dict[str, Any]:
        state = result.temporal.fused_state if result.temporal is not None else None
        return {
            "timestamp": time.time(),
            "committed": result.committed,
            "state_changed": result.state_changed,
            "board_lock": {
                "valid": bool(result.board_lock.valid),
                "method": result.board_lock.method,
                "confidence": result.board_lock.confidence,
            },
            "validation": None if result.validation is None else {
                "valid": result.validation.valid,
                "errors": result.validation.errors,
                "warnings": result.validation.warnings,
                "confidence": result.validation.confidence,
            },
            "state": None if state is None else {
                "region_counts": state.region_counts,
                "dice": list(state.dice),
                "cube_value": state.cube_value,
                "confidence": state.confidence,
                "timestamp": state.timestamp,
            },
            "piece_confidence": None if result.pieces is None else result.pieces.confidence,
            "dice_cube_confidence": None if result.dice_cube is None else result.dice_cube.confidence,
            "debug": result.debug,
        }

    def record(self, result: PipelineResult) -> None:
        summary = self._safe(self.result_summary(result))
        self._results_file.write(json.dumps(summary) + "\n")
        self._results_file.flush()
        for event in result.events:
            self._events_file.write(json.dumps(self._safe(event)) + "\n")
        self._events_file.flush()


__all__ = ["PipelineLogConfig", "PipelineResultLogger"]


## Known-state evaluator


In [ ]:
%%writefile Evaluation/board_state_evaluator.py
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Optional, Tuple
import json

from backgammon_types import BoardState


@dataclass
class BoardStateEvaluation:
    total_regions: int
    exact_region_matches: int
    total_colour_slots: int
    correct_colour_counts: int
    absolute_checker_error: int
    dice_match: Optional[bool]
    cube_match: Optional[bool]
    details: Dict[str, Any] = field(default_factory=dict)

    @property
    def region_accuracy(self) -> float:
        return self.exact_region_matches / max(self.total_regions, 1)

    @property
    def colour_count_accuracy(self) -> float:
        return self.correct_colour_counts / max(self.total_colour_slots, 1)


class BoardStateEvaluator:
    """
    Simple evaluation helper for known-position tests.

    Expected JSON format:
    {
      "region_counts": {"point_01": {"light": 2, "dark": 0}, ...},
      "dice": [1, 2],
      "cube_value": 2
    }
    """

    @staticmethod
    def load_expected(path: str | Path) -> Dict[str, Any]:
        return json.loads(Path(path).read_text(encoding="utf-8"))

    @staticmethod
    def _counts(obj: Dict[str, Any], region: str, colour: str) -> int:
        return int(obj.get(region, {}).get(colour, 0))

    def evaluate(self, expected: Dict[str, Any], observed: BoardState) -> BoardStateEvaluation:
        expected_counts = expected.get("region_counts", {})
        observed_counts = observed.region_counts
        regions = sorted(set(expected_counts) | set(observed_counts))
        exact_region_matches = 0
        correct_colour_counts = 0
        total_colour_slots = 0
        absolute_checker_error = 0
        region_details = {}

        for region in regions:
            exp_l = self._counts(expected_counts, region, "light")
            exp_d = self._counts(expected_counts, region, "dark")
            obs_l = self._counts(observed_counts, region, "light")
            obs_d = self._counts(observed_counts, region, "dark")
            exact = (exp_l == obs_l and exp_d == obs_d)
            exact_region_matches += int(exact)
            correct_colour_counts += int(exp_l == obs_l) + int(exp_d == obs_d)
            total_colour_slots += 2
            absolute_checker_error += abs(exp_l - obs_l) + abs(exp_d - obs_d)
            if not exact:
                region_details[region] = {
                    "expected": {"light": exp_l, "dark": exp_d},
                    "observed": {"light": obs_l, "dark": obs_d},
                }

        expected_dice = expected.get("dice")
        dice_match = None if expected_dice is None else tuple(expected_dice) == tuple(observed.dice)
        expected_cube = expected.get("cube_value")
        cube_match = None if expected_cube is None else expected_cube == observed.cube_value

        return BoardStateEvaluation(
            total_regions=len(regions),
            exact_region_matches=exact_region_matches,
            total_colour_slots=total_colour_slots,
            correct_colour_counts=correct_colour_counts,
            absolute_checker_error=absolute_checker_error,
            dice_match=dice_match,
            cube_match=cube_match,
            details={"region_mismatches": region_details},
        )


__all__ = ["BoardStateEvaluation", "BoardStateEvaluator"]


## Full live pipeline test runner


In [ ]:
%%writefile Evaluation/pipeline_live_test.py
from __future__ import annotations

from pathlib import Path
from typing import Optional
import argparse
import sys

_MODULE_DIR = Path(__file__).resolve().parent
_PROJECT_ROOT = _MODULE_DIR.parent if _MODULE_DIR.name in {"Acquisition", "Geometric + Visual Preprocessing", "Interpretation", "Evaluation"} else _MODULE_DIR
for _rel in ("", "Acquisition", "Geometric + Visual Preprocessing", "Interpretation", "Evaluation"):
    _p = _PROJECT_ROOT / _rel if _rel else _PROJECT_ROOT
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from keyframe_gate import OakSRKeyframeGate
from board_state_pipeline import BoardStatePipeline
from pipeline_result_logger import PipelineLogConfig, PipelineResultLogger


def build_arg_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Run the full board-state pipeline on live keyframes and optionally log results.")
    p.add_argument("--frames", type=int, default=300, help="Maximum camera frames to inspect.")
    p.add_argument("--process-all", action="store_true", help="Process every packet instead of accepted keyframes only.")
    p.add_argument("--log", action="store_true", help="Write pipeline_results.jsonl and events.jsonl.")
    p.add_argument("--log-dir", default="pipeline_logs", help="Output directory for logs.")
    p.add_argument("--print-every", type=int, default=1, help="Print every N processed results.")
    return p


def main(argv: Optional[list[str]] = None) -> None:
    args = build_arg_parser().parse_args(argv)
    gate = OakSRKeyframeGate().start()
    pipeline = BoardStatePipeline()
    logger = PipelineResultLogger(PipelineLogConfig(root_dir=args.log_dir)) if args.log else None
    processed = 0

    try:
        for _ in range(args.frames):
            packet = gate.get_packet(block=True)
            if packet is None:
                continue
            if not args.process_all and not packet.keyframe:
                continue

            result = pipeline.process_keyframe_packet(packet)
            processed += 1
            if logger is not None:
                logger.record(result)

            if processed % max(args.print_every, 1) == 0:
                print("-" * 72)
                print(f"frame={packet.frame_index} committed={result.committed} changed={result.state_changed} events={len(result.events)}")
                print(f"lock={result.board_lock.valid} method={result.board_lock.method} conf={result.board_lock.confidence:.2f}")
                if result.validation is not None:
                    print(f"validation={result.validation.valid} warnings={result.validation.warnings} errors={result.validation.errors}")
                if result.temporal is not None:
                    state = result.temporal.fused_state
                    print(f"state_conf={state.confidence:.2f} dice={state.dice} cube={state.cube_value}")
                    # Print only occupied regions to keep output readable.
                    occupied = {
                        r: c for r, c in state.region_counts.items()
                        if c.get("light", 0) or c.get("dark", 0)
                    }
                    print("occupied:", occupied)
                for event in result.events:
                    print(f"event: {event.event_type} | {event.description}")

    except KeyboardInterrupt:
        pass
    finally:
        gate.stop()
        if logger is not None:
            logger.close()
            print(f"Logs written under: {logger.session_dir}")


if __name__ == "__main__":
    main()


# 6. Validation and testing workflow

Use this section after exporting the module files.


## 6.1 Clear cached modules

Jupyter can keep older module versions in memory, so I run this after editing exported files.


In [ ]:
import sys

MODULES_TO_CLEAR = {
    "streams_module",
    "keyframe_gate",
    "rgbd_recorder",
    "live_stream_viewer",
    "depth_stack_live_viewer",
    "board_registration",
    "perspective_rectification",
    "image_normalisation",
    "point_tray_segmentation",
    "roi_preview",
    "depth_stack_classifier",
    "piece_detection",
    "dice_cube_reader",
    "temporal_state_estimator",
    "rule_validation",
    "board_state_pipeline",
}

for name in list(sys.modules):
    if name in MODULES_TO_CLEAR:
        del sys.modules[name]

print("Cleared cached project modules.")


## 6.2 Syntax check


In [ ]:
import compileall
from pathlib import Path

paths_to_check = [
    "backgammon_types.py",
    "Acquisition",
    "Geometric + Visual Preprocessing",
    "Interpretation",
]

ok = True
for path in paths_to_check:
    p = Path(path)
    if not p.exists():
        print(f"Missing: {path}")
        ok = False
        continue

    if p.is_dir():
        ok = compileall.compile_dir(str(p), force=True, quiet=1) and ok
    else:
        ok = compileall.compile_file(str(p), force=True, quiet=1) and ok

print("Syntax check:", "PASSED" if ok else "FAILED")


## 6.3 Confirm current ROI values

This checks that Python is importing the current `point_tray_segmentation.py` rather than an older cached version.


In [ ]:
from pathlib import Path
import sys

# Ensure module paths are present.
PROJECT_ROOT = Path.cwd()
for rel in ["", "Acquisition", "Geometric + Visual Preprocessing", "Interpretation", "Evaluation"]:
    p = PROJECT_ROOT / rel if rel else PROJECT_ROOT
    if str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

if "point_tray_segmentation" in sys.modules:
    del sys.modules["point_tray_segmentation"]

import point_tray_segmentation

print("Imported from:", point_tray_segmentation.__file__)

cfg = point_tray_segmentation.SegmentationConfig()
print("checker_detection_area_frac:", cfg.checker_detection_area_frac)
print("middle_strip_exclusion_frac:", cfg.middle_strip_exclusion_frac)
print("dice_area_frac:", cfg.dice_area_frac)
print("cube_area_frac:", cfg.cube_area_frac)
print("depth_analysis_area_frac:", cfg.depth_analysis_area_frac)
print("use_triangular_point_masks:", cfg.use_triangular_point_masks)
print("draw_checker_detection_contour:", cfg.draw_checker_detection_contour)


## 6.4 ROI preview

Run this to view the named ROIs over the rectified board.

Controls in the preview window:

```text
a = automatic board lock
m = manual corner mode; click TL, TR, BR, BL
f = full-frame fallback
c = clear manual corners
s = save manual corners
l = load manual_board_corners.json
q / Esc = quit
```

flow:

1. Run the cell.
2. Press `l` to load the saved manual corners.
3. Check that the green checker area, yellow dice area, magenta cube area, and red middle-strip exclusion align correctly.


In [ ]:
# ROI preview with manual-corner load support.
%run "Geometric + Visual Preprocessing/roi_preview.py"


## 6.5 Depth stack diagnostic viewer

These are the current working settings, updated to match the latest ROI values.

Use this first without recording:


In [ ]:
%run Acquisition/depth_stack_live_viewer.py --fps 15 --baseline-frames 60 --chip-thickness-mm 10 --height-stat top35 --temporal-window 3 --temporal-require 2 --noise-margin-mm 1.5 --expected-chip-radius-px 16 --min-support-pixels 6 --roi-frac 0.10,0.00,1.00,1.00 --middle-exclusion-frac 0.00,0.38,1.00,0.60 --stack-promote-height-mm 16.5 --stack-demote-height-mm 14.5 --stack-temporal-window 5 --stack-promote-votes 2 --stack-demote-votes 4 --stack-hold-misses 2


For troubleshooting recordings:


In [ ]:
%run Acquisition/depth_stack_live_viewer.py --fps 15 --baseline-frames 60 --chip-thickness-mm 10 --height-stat top35 --temporal-window 3 --temporal-require 2 --noise-margin-mm 1.5 --expected-chip-radius-px 16 --min-support-pixels 6 --roi-frac 0.10,0.00,1.00,1.00 --middle-exclusion-frac 0.00,0.38,1.00,0.60 --stack-promote-height-mm 16.5 --stack-demote-height-mm 14.5 --stack-temporal-window 5 --stack-promote-votes 2 --stack-demote-votes 4 --stack-hold-misses 2 --record --record-every 3


## 6.6 Live stream / pre-CNN diagnostic viewer

Use this to inspect the main acquisition and preprocessing pipeline.


In [ ]:
# Raw streams only:
# %run Acquisition/live_stream_viewer.py --mode streams --fps 15

# Keyframe-gate diagnostics:
# %run Acquisition/live_stream_viewer.py --mode gate --fps 15

# Full pre-CNN diagnostic viewer:
%run Acquisition/live_stream_viewer.py --mode all --fps 15


## 6.7 Import check

Run after writing all files and clearing caches.


In [ ]:
from board_state_pipeline import BoardStatePipeline
from rgbd_recorder import RGBDRecorder, RecorderConfig
from board_registration import BoardRegistrar
from perspective_rectification import PerspectiveRectifier
from image_normalisation import BoardNormaliser
from point_tray_segmentation import PointTraySegmenter
from piece_detection import RGBDPieceDetector
from dice_cube_reader import DiceCubeReader
from live_stream_viewer import LiveStreamViewer, LiveViewConfig

print("Project modules imported successfully.")


## 6.8 Example full pipeline loop

This processes accepted keyframes through `BoardStatePipeline`.


In [ ]:
import cv2

from keyframe_gate import OakSRKeyframeGate
from board_state_pipeline import BoardStatePipeline

gate = OakSRKeyframeGate().start()
pipeline = BoardStatePipeline()

try:
    for _ in range(300):
        packet = gate.get_packet(block=True)
        if packet is None or not packet.keyframe:
            continue

        result = pipeline.process_keyframe_packet(packet)

        print(
            "committed=", result.committed,
            "state_changed=", result.state_changed,
            "validation=", result.validation.valid if result.validation else None,
        )

        if result.normalised is not None:
            cv2.imshow("Normalised RGB", result.normalised.rgb_bgr)
            cv2.imshow("Depth BW", result.normalised.depth_gray)

        if result.pieces is not None:
            cv2.imshow("Pieces", result.pieces.overlay_bgr)

        if result.dice_cube is not None:
            cv2.imshow("Dice / Cube", result.dice_cube.overlay_bgr)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    gate.stop()
    cv2.destroyAllWindows()


## 6.9 Current test commands

Run ROI preview and press `l` to load `manual_board_corners.json`:

```python
%run "Geometric + Visual Preprocessing/roi_preview.py"
```

Run the tuned depth-stack diagnostic viewer:

```python
%run Acquisition/depth_stack_live_viewer.py --fps 15 --baseline-frames 60 --chip-thickness-mm 10 --height-stat top35 --temporal-window 3 --temporal-require 2 --noise-margin-mm 1.5 --expected-chip-radius-px 16 --min-support-pixels 6 --roi-frac 0.10,0.00,1.00,1.00 --middle-exclusion-frac 0.00,0.38,1.00,0.60 --stack-promote-height-mm 16.5 --stack-demote-height-mm 14.5 --stack-temporal-window 5 --stack-promote-votes 2 --stack-demote-votes 4 --stack-hold-misses 2
```

Run the full pipeline on accepted keyframes and log state/events:

```python
%run Evaluation/pipeline_live_test.py --frames 300 --log --print-every 1
```

Run the full pipeline on every packet if keyframe gating is too strict:

```python
%run Evaluation/pipeline_live_test.py --frames 150 --process-all --log --print-every 1
```


In [ ]:
%run "Geometric + Visual Preprocessing/roi_preview.py"